# BioHub Cell Tracking Architecture — Kaggle Full Dataset Pipeline

**Independent Architecture Execution Pipeline**  
Processes full 3D+t Zarr microscopy volumes using regional inference, 3D U-Net detection, canonical global node reconciliation, STGNN spatio-temporal association, and Global ILP optimization.


## 1. Project Setup & Package Unpacking
Extracts core Python architecture modules (`src/`), execution pipeline (`kaggle_pipeline.py`), and pre-trained STGNN checkpoint.

In [ ]:
import os
import sys
import base64
from pathlib import Path

# Ensure working directory is setup
working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()
if str(working_dir) not in sys.path:
    sys.path.insert(0, str(working_dir))

print('==================================================================')
print(' UNPACKING BIOHUB CELL TRACKING ARCHITECTURE CODE & CHECKPOINT')
print('==================================================================')

# Dictionary of source files
src_files = {}
src_files['src/__init__.py'] = '"""\nBioHub Cell Tracking - Foundation Package\n"""\n\n__version__ = "0.1.0"\n'
src_files['src/association/__init__.py'] = '"""\nTemporal Candidate Edge STGNN Association Package.\n"""\n\nfrom .labels import CandidateEdgeLabeler, LabelStats, TemporalSplitter, TemporalSplit\nfrom .base import GeometricBaseline\nfrom .stgnn import STGNNModel, STGNNMessagePassingLayer\nfrom .dataset import AssociationDataset\nfrom .evaluation import AssociationEvaluator, EvaluationMetrics\nfrom .trainer import STGNNTrainer, STGNNTrainerConfig, OverfitTestResult\n\n__all__ = [\n    "CandidateEdgeLabeler",\n    "LabelStats",\n    "TemporalSplitter",\n    "TemporalSplit",\n    "GeometricBaseline",\n    "STGNNModel",\n    "STGNNMessagePassingLayer",\n    "AssociationDataset",\n    "AssociationEvaluator",\n    "EvaluationMetrics",\n    "STGNNTrainer",\n    "STGNNTrainerConfig",\n    "OverfitTestResult",\n]\n'
src_files['src/association/base.py'] = '"""\nDeterministic Geometric Baseline Association Model.\n"""\n\nfrom typing import List, Dict, Any, Union\nimport numpy as np\n\nfrom src.graph.candidate_graph import TemporalCandidateGraph, CandidateEdge\n\n\nclass GeometricBaseline:\n    """\n    Deterministic geometric baseline association model.\n\n    Computes association evidence strictly from physical distance using exponential decay:\n        Score(e) = exp(- d_phys(e) / sigma_scale_um)\n\n    Guarantees:\n    - Score is strictly in [0.0, 1.0].\n    - Score decreases monotonically with physical distance (micrometers).\n    - Fully deterministic and calibration-ready.\n    """\n\n    def __init__(self, sigma_scale_um: float = 2.0):\n        self.sigma_scale_um = max(1e-6, sigma_scale_um)\n\n    def predict_edge_scores(\n        self,\n        edges: Union[List[CandidateEdge], TemporalCandidateGraph]\n    ) -> np.ndarray:\n        """\n        Computes geometric baseline scores for candidate edges.\n\n        :param edges: List of CandidateEdge objects or TemporalCandidateGraph\n        :return: NumPy array shape (E,) containing baseline scores in [0, 1]\n        """\n        edge_list = edges.edges if isinstance(edges, TemporalCandidateGraph) else edges\n        if not edge_list:\n            return np.empty(0, dtype=np.float32)\n\n        distances = np.array([e.euclidean_distance_phys for e in edge_list], dtype=np.float32)\n        scores = np.exp(-distances / self.sigma_scale_um)\n        return np.clip(scores, 0.0, 1.0).astype(np.float32)\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Returns baseline configuration."""\n        return {\n            "model_type": "GeometricBaseline",\n            "sigma_scale_um": self.sigma_scale_um,\n            "formula": f"exp(-d_phys / {self.sigma_scale_um})"\n        }\n'
src_files['src/association/dataset.py'] = '"""\nGraph Association Dataset and Feature/Tensor Preparation.\n"""\n\nfrom typing import Dict, Any, Optional, Tuple, List\nimport numpy as np\nimport torch\n\nfrom src.graph.candidate_graph import TemporalCandidateGraph, CandidateEdge\nfrom src.graph.features import GraphFeatureExtractor, NodeFeatures, EdgeFeatures\nfrom src.graph.diagnostics import NodeMatchingResult\nfrom src.association.labels import CandidateEdgeLabeler, LabelStats, TemporalSplitter, TemporalSplit\n\n\nclass AssociationDataset:\n    """\n    Constructs PyTorch tensor representations from TemporalCandidateGraph for model training and evaluation.\n    """\n\n    def __init__(\n        self,\n        graph: TemporalCandidateGraph,\n        feature_extractor: Optional[GraphFeatureExtractor] = None,\n        labeler: Optional[CandidateEdgeLabeler] = None,\n        geff_edges: Optional[List[Tuple[int, int]]] = None,\n        matching_result: Optional[NodeMatchingResult] = None\n    ):\n        self.graph = graph\n        self.feature_extractor = feature_extractor or GraphFeatureExtractor()\n        self.labeler = labeler or CandidateEdgeLabeler()\n\n        # Extract graph node & edge features\n        feats = self.feature_extractor.extract(graph)\n        self.node_features: NodeFeatures = feats["node_features"]\n        self.edge_features: EdgeFeatures = feats["edge_features"]\n\n        # Generate ground-truth labels if GEFF data provided\n        if geff_edges is not None and matching_result is not None:\n            self.labels, self.label_stats = self.labeler.create_labels(graph, geff_edges, matching_result)\n        else:\n            self.labels = np.zeros(graph.candidate_count(), dtype=np.float32)\n            self.label_stats = LabelStats(\n                total_candidate_edges=graph.candidate_count(),\n                positive_edges=0,\n                negative_edges=graph.candidate_count(),\n                positive_rate_pct=0.0,\n                negatives_per_positive=0.0,\n                pos_edge_distances_um=[],\n                neg_edge_distances_um=[]\n            )\n\n        # Convert to PyTorch Tensors\n        self.node_feats_tensor = torch.from_numpy(self.node_features.features).float()\n        self.edge_index_tensor = torch.from_numpy(self.edge_features.edge_index).long()\n        self.edge_feats_tensor = torch.from_numpy(self.edge_features.features).float()\n        self.labels_tensor = torch.from_numpy(self.labels).float()\n\n    def get_split_tensors(\n        self,\n        split_mask: np.ndarray,\n        device: str = "cpu"\n    ) -> Dict[str, torch.Tensor]:\n        """\n        Extracts subgraph tensor dict corresponding to candidate edge split mask.\n\n        :param split_mask: Boolean numpy array of shape (E,) selecting edges\n        :param device: PyTorch device (\'cpu\' or \'cuda\')\n        :return: Dict containing PyTorch tensors\n        """\n        mask_idx = np.where(split_mask)[0]\n        dev = torch.device(device)\n\n        if len(mask_idx) == 0:\n            return {\n                "node_feats": self.node_feats_tensor.to(dev),\n                "edge_index": torch.empty((2, 0), dtype=torch.long, device=dev),\n                "edge_feats": torch.empty((0, self.edge_features.features.shape[1]), dtype=torch.float32, device=dev),\n                "labels": torch.empty(0, dtype=torch.float32, device=dev),\n                "edge_mask": torch.empty(0, dtype=torch.bool, device=dev),\n            }\n\n        edge_index_sub = self.edge_index_tensor[:, mask_idx].to(dev)\n        edge_feats_sub = self.edge_feats_tensor[mask_idx].to(dev)\n        labels_sub = self.labels_tensor[mask_idx].to(dev)\n\n        return {\n            "node_feats": self.node_feats_tensor.to(dev),\n            "edge_index": edge_index_sub,\n            "edge_feats": edge_feats_sub,\n            "labels": labels_sub,\n            "edge_mask": torch.from_numpy(split_mask).to(dev),\n        }\n'
src_files['src/association/evaluation.py'] = '"""\nDiagnostic Evaluation Metrics, Geometric Baseline Comparison & Displacement-Binned Breakdown.\n"""\n\nfrom dataclasses import dataclass, field\nfrom typing import List, Dict, Any, Optional, Tuple\nimport numpy as np\n\nfrom src.association.base import GeometricBaseline\n\n\n@dataclass\nclass EvaluationMetrics:\n    """Container for candidate edge association classification metrics."""\n    roc_auc: float\n    pr_auc: float\n    precision: float\n    recall: float\n    f1: float\n    positive_edge_recall: float\n    negative_edge_specificity: float\n    mean_prob_positive: float\n    mean_prob_negative: float\n    total_edges: int\n    pos_count: int\n    neg_count: int\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Convert metrics to dictionary representation."""\n        return {\n            "roc_auc": self.roc_auc,\n            "pr_auc": self.pr_auc,\n            "precision": self.precision,\n            "recall": self.recall,\n            "f1": self.f1,\n            "positive_edge_recall": self.positive_edge_recall,\n            "negative_edge_specificity": self.negative_edge_specificity,\n            "mean_prob_positive": self.mean_prob_positive,\n            "mean_prob_negative": self.mean_prob_negative,\n            "total_edges": self.total_edges,\n            "pos_count": self.pos_count,\n            "neg_count": self.neg_count,\n        }\n\n\nclass AssociationEvaluator:\n    """\n    Evaluates candidate edge probability predictions against ground-truth binary labels.\n    """\n\n    @staticmethod\n    def compute_metrics(\n        y_true: np.ndarray,\n        y_prob: np.ndarray,\n        threshold: float = 0.5,\n        strict_single_class_check: bool = False\n    ) -> EvaluationMetrics:\n        """\n        Computes ROC-AUC, PR-AUC, Precision, Recall, F1, Specificity, and Probability stats.\n\n        :param y_true: Binary ground-truth labels (E,) float or int (0 or 1)\n        :param y_prob: Predicted continuous probabilities (E,) in [0, 1]\n        :param threshold: Binary classification threshold (default 0.5)\n        :param strict_single_class_check: If True, raises ValueError if y_true lacks positive or negative labels.\n        :return: EvaluationMetrics dataclass\n        """\n        y_true = np.asarray(y_true, dtype=np.float32)\n        y_prob = np.asarray(y_prob, dtype=np.float32)\n\n        total_edges = len(y_true)\n        pos_mask = (y_true == 1.0)\n        neg_mask = (y_true == 0.0)\n\n        pos_count = int(np.sum(pos_mask))\n        neg_count = int(np.sum(neg_mask))\n\n        if strict_single_class_check:\n            if total_edges > 0 and (pos_count == 0 or neg_count == 0):\n                raise ValueError(\n                    f"Single-class evaluation population is invalid for ROC-AUC evaluation. "\n                    f"Found pos_count={pos_count}, neg_count={neg_count} among {total_edges} candidate edges."\n                )\n\n        if total_edges == 0:\n            return EvaluationMetrics(\n                roc_auc=0.0, pr_auc=0.0, precision=0.0, recall=0.0, f1=0.0,\n                positive_edge_recall=0.0, negative_edge_specificity=0.0,\n                mean_prob_positive=0.0, mean_prob_negative=0.0,\n                total_edges=0, pos_count=0, neg_count=0\n            )\n\n        mean_prob_pos = float(np.mean(y_prob[pos_mask])) if pos_count > 0 else 0.0\n        mean_prob_neg = float(np.mean(y_prob[neg_mask])) if neg_count > 0 else 0.0\n\n        # Threshold-based metrics\n        y_pred = (y_prob >= threshold).astype(np.float32)\n\n        tp = int(np.sum((y_pred == 1.0) & (y_true == 1.0)))\n        fp = int(np.sum((y_pred == 1.0) & (y_true == 0.0)))\n        tn = int(np.sum((y_pred == 0.0) & (y_true == 0.0)))\n        fn = int(np.sum((y_pred == 0.0) & (y_true == 1.0)))\n\n        prec = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0\n        rec = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0\n        f1 = float(2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0\n\n        spec = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0\n\n        # ROC-AUC calculation\n        roc_auc = AssociationEvaluator._compute_roc_auc(y_true, y_prob)\n\n        # PR-AUC calculation\n        pr_auc = AssociationEvaluator._compute_pr_auc(y_true, y_prob)\n\n        return EvaluationMetrics(\n            roc_auc=roc_auc,\n            pr_auc=pr_auc,\n            precision=prec,\n            recall=rec,\n            f1=f1,\n            positive_edge_recall=rec,\n            negative_edge_specificity=spec,\n            mean_prob_positive=mean_prob_pos,\n            mean_prob_negative=mean_prob_neg,\n            total_edges=total_edges,\n            pos_count=pos_count,\n            neg_count=neg_count,\n        )\n\n    @staticmethod\n    def _trapezoid(y: np.ndarray, x: np.ndarray) -> float:\n        """Trapezoidal integration compatible across NumPy 1.x and 2.x."""\n        if hasattr(np, "trapezoid"):\n            return float(np.trapezoid(y, x))\n        elif hasattr(np, "trapz"):\n            return float(np.trapz(y, x))\n        else:\n            return float(np.sum(0.5 * (y[:-1] + y[1:]) * np.diff(x)))\n\n    @staticmethod\n    def _compute_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:\n        """Calculates area under ROC curve using trapezoidal numerical integration."""\n        pos_mask = (y_true == 1.0)\n        neg_mask = (y_true == 0.0)\n        n_pos = np.sum(pos_mask)\n        n_neg = np.sum(neg_mask)\n\n        if n_pos == 0 or n_neg == 0:\n            return 0.5\n\n        # Sort by decreasing score\n        sort_indices = np.argsort(-y_prob)\n        sorted_true = y_true[sort_indices]\n\n        tps = np.cumsum(sorted_true == 1.0)\n        fps = np.cumsum(sorted_true == 0.0)\n\n        tpr = tps / n_pos\n        fpr = fps / n_neg\n\n        # Add origin (0, 0)\n        tpr = np.concatenate([[0.0], tpr])\n        fpr = np.concatenate([[0.0], fpr])\n\n        return AssociationEvaluator._trapezoid(tpr, fpr)\n\n    @staticmethod\n    def _compute_pr_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:\n        """Calculates area under Precision-Recall curve using trapezoidal numerical integration."""\n        n_pos = np.sum(y_true == 1.0)\n        if n_pos == 0:\n            return 0.0\n\n        sort_indices = np.argsort(-y_prob)\n        sorted_true = y_true[sort_indices]\n\n        tps = np.cumsum(sorted_true == 1.0)\n        fps = np.cumsum(sorted_true == 0.0)\n\n        precision = tps / (tps + fps)\n        recall = tps / n_pos\n\n        precision = np.concatenate([[1.0], precision])\n        recall = np.concatenate([[0.0], recall])\n\n        return AssociationEvaluator._trapezoid(precision, recall)\n\n    @staticmethod\n    def evaluate_displacement_bins(\n        y_true: np.ndarray,\n        y_prob: np.ndarray,\n        distances_um: np.ndarray\n    ) -> Dict[str, EvaluationMetrics]:\n        """\n        Evaluates metrics broken down by physical motion displacement bins:\n        - Low: d < 1.5 um\n        - Medium: 1.5 um <= d <= 3.5 um\n        - High: d > 3.5 um\n        """\n        low_mask = (distances_um < 1.5)\n        med_mask = (distances_um >= 1.5) & (distances_um <= 3.5)\n        high_mask = (distances_um > 3.5)\n\n        return {\n            "Low (<1.5 um)": AssociationEvaluator.compute_metrics(y_true[low_mask], y_prob[low_mask]),\n            "Medium (1.5-3.5 um)": AssociationEvaluator.compute_metrics(y_true[med_mask], y_prob[med_mask]),\n            "High (>3.5 um)": AssociationEvaluator.compute_metrics(y_true[high_mask], y_prob[high_mask]),\n        }\n'
src_files['src/association/labels.py'] = '"""\nDeterministic Ground-Truth Candidate Edge Label Generation and Temporal Data Splitting.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Dict, Any, Optional, Set\nimport numpy as np\n\nfrom src.graph.nodes import CanonicalNode, NodeSet\nfrom src.graph.candidate_graph import TemporalCandidateGraph, CandidateEdge\nfrom src.graph.diagnostics import NodeMatchingResult\n\n\n@dataclass\nclass LabelStats:\n    """Statistics describing generated candidate edge ground-truth labels."""\n    total_candidate_edges: int\n    positive_edges: int\n    negative_edges: int\n    positive_rate_pct: float\n    negatives_per_positive: float\n    pos_edge_distances_um: List[float]\n    neg_edge_distances_um: List[float]\n\n\nclass CandidateEdgeLabeler:\n    """\n    Generates binary ground-truth labels for candidate edges AFTER candidate graph construction.\n\n    Label = 1 if the candidate edge corresponds to a true GEFF temporal edge.\n    Label = 0 if the candidate edge is not a true GEFF temporal edge.\n    """\n\n    def create_labels(\n        self,\n        graph: TemporalCandidateGraph,\n        geff_edges: List[Tuple[int, int]],\n        matching_result: NodeMatchingResult\n    ) -> Tuple[np.ndarray, LabelStats]:\n        """\n        Creates ground-truth labels array for all candidate edges in the graph.\n\n        :param graph: TemporalCandidateGraph instance\n        :param geff_edges: List of directed (source_geff_id, dest_geff_id) ground-truth edges\n        :param matching_result: NodeMatchingResult mapping GEFF IDs -> Canonical Node IDs\n        :return: Tuple of (labels_array shape (E,) float32, LabelStats)\n        """\n        geff_to_can = matching_result.geff_to_canonical_map\n\n        # Map GEFF GT edge pairs to canonical node ID pairs\n        true_canonical_edge_pairs: Set[Tuple[int, int]] = set()\n        for u_gt, v_gt in geff_edges:\n            u_can = geff_to_can.get(int(u_gt))\n            v_can = geff_to_can.get(int(v_gt))\n            if u_can is not None and v_can is not None:\n                true_canonical_edge_pairs.add((u_can, v_can))\n\n        labels = []\n        pos_distances = []\n        neg_distances = []\n\n        for edge in graph.edges:\n            pair = (edge.source_node_id, edge.destination_node_id)\n            if pair in true_canonical_edge_pairs:\n                labels.append(1.0)\n                pos_distances.append(edge.euclidean_distance_phys)\n            else:\n                labels.append(0.0)\n                neg_distances.append(edge.euclidean_distance_phys)\n\n        labels_arr = np.array(labels, dtype=np.float32)\n        n_total = len(labels)\n        n_pos = int(np.sum(labels_arr == 1.0))\n        n_neg = int(np.sum(labels_arr == 0.0))\n\n        pos_rate = float(100.0 * n_pos / n_total) if n_total > 0 else 0.0\n        neg_per_pos = float(n_neg / n_pos) if n_pos > 0 else 0.0\n\n        stats = LabelStats(\n            total_candidate_edges=n_total,\n            positive_edges=n_pos,\n            negative_edges=n_neg,\n            positive_rate_pct=pos_rate,\n            negatives_per_positive=neg_per_pos,\n            pos_edge_distances_um=pos_distances,\n            neg_edge_distances_um=neg_distances,\n        )\n\n        return labels_arr, stats\n\n\n@dataclass\nclass TemporalSplit:\n    """Container holding mask indices for Train, Validation, and Test candidate edges."""\n    train_mask: np.ndarray  # boolean array (E,)\n    val_mask: np.ndarray  # boolean array (E,)\n    test_mask: np.ndarray  # boolean array (E,)\n    train_frames: Tuple[int, int]\n    val_frames: Tuple[int, int]\n    test_frames: Tuple[int, int]\n\n\nclass TemporalSplitter:\n    """\n    Splits candidate edges into Train, Validation, and Test sets based strictly on source frame t.\n\n    Guarantees:\n    - Zero data leakage between temporal splits (edges belong to split of source frame t).\n    """\n\n    def __init__(\n        self,\n        train_range: Tuple[int, int] = (0, 69),\n        val_range: Tuple[int, int] = (70, 84),\n        test_range: Tuple[int, int] = (85, 99)\n    ):\n        self.train_range = train_range\n        self.val_range = val_range\n        self.test_range = test_range\n\n    def split_graph_edges(self, graph: TemporalCandidateGraph) -> TemporalSplit:\n        """\n        Creates boolean masks partitioning graph.edges into train, val, and test.\n\n        :param graph: TemporalCandidateGraph instance\n        :return: TemporalSplit dataclass\n        """\n        n_edges = graph.candidate_count()\n        train_mask = np.zeros(n_edges, dtype=bool)\n        val_mask = np.zeros(n_edges, dtype=bool)\n        test_mask = np.zeros(n_edges, dtype=bool)\n\n        for idx, edge in enumerate(graph.edges):\n            t = edge.source_frame\n            if self.train_range[0] <= t <= self.train_range[1]:\n                train_mask[idx] = True\n            elif self.val_range[0] <= t <= self.val_range[1]:\n                val_mask[idx] = True\n            elif self.test_range[0] <= t <= self.test_range[1]:\n                test_mask[idx] = True\n\n        return TemporalSplit(\n            train_mask=train_mask,\n            val_mask=val_mask,\n            test_mask=test_mask,\n            train_frames=self.train_range,\n            val_frames=self.val_range,\n            test_frames=self.test_range,\n        )\n'
src_files['src/association/stgnn.py'] = '"""\nSpatio-Temporal Graph Neural Network (STGNN) Candidate Edge Association Model.\n"""\n\nfrom typing import Dict, Any, Optional, Tuple\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass STGNNMessagePassingLayer(nn.Module):\n    """\n    Edge-aware graph message-passing layer.\n\n    Computes directional messages between connected nodes, aggregates incoming/outgoing\n    neighbor context, and updates node & edge representations simultaneously.\n    """\n\n    def __init__(self, hidden_dim: int, dropout: float = 0.1):\n        super().__init__()\n        self.hidden_dim = hidden_dim\n\n        # Message MLP: [h_src, h_dst, e_attr] -> message\n        self.message_mlp = nn.Sequential(\n            nn.Linear(hidden_dim * 3, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Dropout(dropout)\n        )\n\n        # Node update MLP: [h_node, msg_in, msg_out] -> h_node_updated\n        self.node_update_mlp = nn.Sequential(\n            nn.Linear(hidden_dim * 3, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Dropout(dropout)\n        )\n\n        # Edge update MLP: [e_attr, h_src, h_dst, message] -> e_attr_updated\n        self.edge_update_mlp = nn.Sequential(\n            nn.Linear(hidden_dim * 4, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Dropout(dropout)\n        )\n\n    def forward(\n        self,\n        node_feats: torch.Tensor,  # (N, D_hidden)\n        edge_index: torch.Tensor,  # (2, E)\n        edge_feats: torch.Tensor   # (E, D_hidden)\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        """\n        Forward pass for message passing layer.\n        """\n        N = node_feats.size(0)\n        E = edge_index.size(1)\n\n        if E == 0:\n            return node_feats, edge_feats\n\n        src_idx = edge_index[0]  # (E,)\n        dst_idx = edge_index[1]  # (E,)\n\n        h_src = node_feats[src_idx]  # (E, D_hidden)\n        h_dst = node_feats[dst_idx]  # (E, D_hidden)\n\n        # 1. Compute messages for each edge\n        msg_in_concat = torch.cat([h_src, h_dst, edge_feats], dim=-1)  # (E, 3*D_hidden)\n        messages = self.message_mlp(msg_in_concat)  # (E, D_hidden)\n\n        # 2. Aggregate messages at target nodes (incoming) and source nodes (outgoing)\n        msg_in_agg = torch.zeros(N, self.hidden_dim, device=node_feats.device)\n        msg_out_agg = torch.zeros(N, self.hidden_dim, device=node_feats.device)\n        count_in = torch.zeros(N, 1, device=node_feats.device)\n        count_out = torch.zeros(N, 1, device=node_feats.device)\n\n        msg_in_agg.index_add_(0, dst_idx, messages)\n        count_in.index_add_(0, dst_idx, torch.ones(E, 1, device=node_feats.device))\n\n        msg_out_agg.index_add_(0, src_idx, messages)\n        count_out.index_add_(0, src_idx, torch.ones(E, 1, device=node_feats.device))\n\n        msg_in_mean = msg_in_agg / torch.clamp(count_in, min=1.0)\n        msg_out_mean = msg_out_agg / torch.clamp(count_out, min=1.0)\n\n        # 3. Update node representations\n        node_update_concat = torch.cat([node_feats, msg_in_mean, msg_out_mean], dim=-1)\n        node_feats_updated = node_feats + self.node_update_mlp(node_update_concat)\n\n        # 4. Update edge representations using updated node states\n        h_src_up = node_feats_updated[src_idx]\n        h_dst_up = node_feats_updated[dst_idx]\n        edge_update_concat = torch.cat([edge_feats, h_src_up, h_dst_up, messages], dim=-1)\n        edge_feats_updated = edge_feats + self.edge_update_mlp(edge_update_concat)\n\n        return node_feats_updated, edge_feats_updated\n\n\nclass STGNNModel(nn.Module):\n    """\n    Spatio-Temporal Graph Neural Network for scoring temporal candidate edge associations.\n    """\n\n    def __init__(\n        self,\n        node_in_dim: int = 9,\n        edge_in_dim: int = 16,\n        hidden_dim: int = 64,\n        num_layers: int = 2,\n        dropout: float = 0.1,\n        geometry_only: bool = False\n    ):\n        super().__init__()\n        self.node_in_dim = node_in_dim\n        self.edge_in_dim = edge_in_dim\n        self.hidden_dim = hidden_dim\n        self.num_layers = num_layers\n        self.geometry_only = geometry_only\n\n        # Node & Edge Encoders\n        self.node_encoder = nn.Sequential(\n            nn.Linear(node_in_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU()\n        )\n\n        self.edge_encoder = nn.Sequential(\n            nn.Linear(edge_in_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU()\n        )\n\n        # Message Passing Layers\n        self.gnn_layers = nn.ModuleList([\n            STGNNMessagePassingLayer(hidden_dim=hidden_dim, dropout=dropout)\n            for _ in range(num_layers)\n        ])\n\n        # Final Edge Classifier Head\n        self.classifier = nn.Sequential(\n            nn.Linear(hidden_dim * 3, hidden_dim),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_dim, 1)\n        )\n\n    def forward(\n        self,\n        node_feats: torch.Tensor,  # (N, D_node)\n        edge_index: torch.Tensor,  # (2, E)\n        edge_feats: torch.Tensor   # (E, D_edge)\n    ) -> torch.Tensor:\n        """\n        Forward pass producing raw uncalibrated logits for each candidate edge.\n\n        :return: Tensor of shape (E, 1) raw edge logits\n        """\n        E = edge_index.size(1)\n        if E == 0:\n            return torch.empty((0, 1), device=node_feats.device)\n\n        # 1. Encode raw inputs\n        h_node = self.node_encoder(node_feats)  # (N, D_hidden)\n        h_edge = self.edge_encoder(edge_feats)  # (E, D_hidden)\n\n        # 2. Apply GNN message passing unless geometry_only ablation is active\n        if not self.geometry_only:\n            for gnn_layer in self.gnn_layers:\n                h_node, h_edge = gnn_layer(h_node, edge_index, h_edge)\n\n        # 3. Predict candidate edge logits\n        src_idx = edge_index[0]\n        dst_idx = edge_index[1]\n\n        h_src = h_node[src_idx]\n        h_dst = h_node[dst_idx]\n\n        combined_edge_repr = torch.cat([h_src, h_dst, h_edge], dim=-1)  # (E, 3*D_hidden)\n        logits = self.classifier(combined_edge_repr)  # (E, 1)\n\n        return logits\n\n    def predict_probabilities(\n        self,\n        node_feats: torch.Tensor,\n        edge_index: torch.Tensor,\n        edge_feats: torch.Tensor\n    ) -> torch.Tensor:\n        """\n        Produces calibrated sigmoid probabilities P(edge represents true temporal continuation) in [0, 1].\n        """\n        with torch.no_grad():\n            logits = self.forward(node_feats, edge_index, edge_feats)\n            return torch.sigmoid(logits).squeeze(-1)\n'
src_files['src/association/trainer.py'] = '"""\nSTGNN Association Model Trainer, Pipeline Integrity Auditor & Controlled Overfit Tester.\n"""\n\nfrom dataclasses import dataclass, field\nfrom typing import Dict, Any, Optional, List, Tuple\nimport copy\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.optim import AdamW\n\nfrom src.association.stgnn import STGNNModel\nfrom src.association.dataset import AssociationDataset\nfrom src.association.evaluation import AssociationEvaluator, EvaluationMetrics\n\n\n@dataclass\nclass STGNNTrainerConfig:\n    """\n    Configuration parameters for STGNN model training.\n    """\n    epochs: int = 40\n    learning_rate: float = 1e-3\n    weight_decay: float = 1e-4\n    hidden_dim: int = 64\n    num_layers: int = 2\n    dropout: float = 0.1\n    pos_weight: Optional[float] = None  # Class weight for BCE loss\n    seed: int = 42\n\n\n@dataclass\nclass OverfitTestResult:\n    """Diagnostic outcome of Part 13 Controlled Overfit Test."""\n    initial_loss: float\n    final_loss: float\n    initial_mean_pos_prob: float\n    final_mean_pos_prob: float\n    initial_mean_neg_prob: float\n    final_mean_neg_prob: float\n    initial_roc_auc: float\n    final_roc_auc: float\n    overfit_success: bool\n\n\nclass STGNNTrainer:\n    """\n    Trainer for STGNN association model with pipeline integrity checks and controlled overfit testing.\n    """\n\n    def __init__(\n        self,\n        model: STGNNModel,\n        config: Optional[STGNNTrainerConfig] = None,\n        device: str = "cpu"\n    ):\n        self.model = model\n        self.config = config or STGNNTrainerConfig()\n        self.device = torch.device(device)\n        self.model.to(self.device)\n\n        # Set deterministic seeds\n        torch.manual_seed(self.config.seed)\n        np.random.seed(self.config.seed)\n\n    def run_pipeline_integrity_checks(\n        self,\n        dataset: AssociationDataset,\n        train_mask: np.ndarray,\n        val_mask: np.ndarray,\n        test_mask: np.ndarray\n    ) -> Dict[str, bool]:\n        """\n        Part 12 — Verifies all 12 mandatory pipeline integrity conditions.\n        """\n        checks = {}\n        tensors = dataset.get_split_tensors(train_mask, device=str(self.device))\n\n        node_feats = tensors["node_feats"]\n        edge_index = tensors["edge_index"]\n        edge_feats = tensors["edge_feats"]\n        labels = tensors["labels"]\n\n        E = edge_index.size(1)\n        N = node_feats.size(0)\n\n        # 1. Non-empty graph\n        checks["1_non_empty_graph"] = bool(N > 0 and E > 0)\n        # 2. Non-zero edge count\n        checks["2_non_zero_edge_count"] = bool(E > 0)\n\n        # 3 & 4. Positive and Negative labels present across ALL splits (Train, Val, Test)\n        val_tensors = dataset.get_split_tensors(val_mask, device=str(self.device))\n        test_tensors = dataset.get_split_tensors(test_mask, device=str(self.device))\n\n        tr_pos = int(torch.sum(labels == 1.0).item())\n        tr_neg = int(torch.sum(labels == 0.0).item())\n\n        val_pos = int(torch.sum(val_tensors["labels"] == 1.0).item())\n        val_neg = int(torch.sum(val_tensors["labels"] == 0.0).item())\n\n        test_pos = int(torch.sum(test_tensors["labels"] == 1.0).item())\n        test_neg = int(torch.sum(test_tensors["labels"] == 0.0).item())\n\n        checks["3_positive_labels_present_all_splits"] = bool(tr_pos > 0 and val_pos > 0 and test_pos > 0)\n        checks["4_negative_labels_present_all_splits"] = bool(tr_neg > 0 and val_neg > 0 and test_neg > 0)\n\n        # 5. Finite node features\n        checks["5_finite_node_features"] = bool(torch.all(torch.isfinite(node_feats)).item())\n\n        # 6. Finite edge features\n        checks["6_finite_edge_features"] = bool(torch.all(torch.isfinite(edge_feats)).item())\n\n        # 7. Correct source/destination indices\n        if E > 0:\n            valid_src = bool((edge_index[0] >= 0).all() and (edge_index[0] < N).all())\n            valid_dst = bool((edge_index[1] >= 0).all() and (edge_index[1] < N).all())\n            checks["7_correct_node_indices"] = bool(valid_src and valid_dst)\n        else:\n            checks["7_correct_node_indices"] = True\n\n        # 8. Correct label alignment\n        checks["8_correct_label_alignment"] = bool(labels.size(0) == E)\n\n        # 9. Correct train/val/test split disjointness\n        no_overlap = bool(\n            not np.any(train_mask & val_mask) and\n            not np.any(train_mask & test_mask) and\n            not np.any(val_mask & test_mask)\n        )\n        checks["9_correct_split_disjointness"] = no_overlap\n\n        # 10. Model output shape matching candidate edge count\n        self.model.eval()\n        with torch.no_grad():\n            logits = self.model(node_feats, edge_index, edge_feats)\n        checks["10_output_shape_matches_edges"] = bool(logits.shape == (E, 1))\n\n        # 11. Gradients reaching model\n        self.model.train()\n        logits_tr = self.model(node_feats, edge_index, edge_feats)\n        loss = F.binary_cross_entropy_with_logits(logits_tr.squeeze(-1), labels)\n        self.model.zero_grad()\n        loss.backward()\n\n        grads_present = True\n        for param in self.model.parameters():\n            if param.requires_grad:\n                if param.grad is None or float(param.grad.abs().sum()) == 0.0:\n                    grads_present = False\n                    break\n        checks["11_gradients_reaching_model"] = grads_present\n\n        # 12. Parameters actually changing during optimizer step\n        param_before = [p.clone() for p in self.model.parameters() if p.requires_grad]\n        optimizer = AdamW(self.model.parameters(), lr=1e-2)\n        optimizer.step()\n        param_after = [p for p in self.model.parameters() if p.requires_grad]\n\n        params_changed = any(\n            not torch.equal(b, a) for b, a in zip(param_before, param_after)\n        )\n        checks["12_parameters_actually_changing"] = params_changed\n\n        return checks\n\n    def run_controlled_overfit_test(\n        self,\n        dataset: AssociationDataset,\n        train_mask: np.ndarray,\n        epochs: int = 50\n    ) -> OverfitTestResult:\n        """\n        Part 13 — Controlled Overfit Test: Trains STGNN on small fixed candidate graph\n        to verify that loss decreases and positive/negative separation increases.\n        """\n        tensors = dataset.get_split_tensors(train_mask, device=str(self.device))\n        node_feats = tensors["node_feats"]\n        edge_index = tensors["edge_index"]\n        edge_feats = tensors["edge_feats"]\n        labels = tensors["labels"]\n\n        model = STGNNModel(\n            node_in_dim=node_feats.size(1),\n            edge_in_dim=edge_feats.size(1),\n            hidden_dim=32,\n            num_layers=2,\n            dropout=0.0\n        ).to(self.device)\n\n        optimizer = AdamW(model.parameters(), lr=1e-2)\n        loss_fn = nn.BCEWithLogitsLoss()\n\n        # Initial stats\n        model.eval()\n        with torch.no_grad():\n            init_logits = model(node_feats, edge_index, edge_feats).squeeze(-1)\n            init_loss = float(loss_fn(init_logits, labels).item())\n            init_probs = torch.sigmoid(init_logits).cpu().numpy()\n            init_metrics = AssociationEvaluator.compute_metrics(labels.cpu().numpy(), init_probs)\n\n        # Train loop\n        model.train()\n        for epoch in range(epochs):\n            optimizer.zero_grad()\n            logits = model(node_feats, edge_index, edge_feats).squeeze(-1)\n            loss = loss_fn(logits, labels)\n            loss.backward()\n            optimizer.step()\n\n        # Final stats\n        model.eval()\n        with torch.no_grad():\n            final_logits = model(node_feats, edge_index, edge_feats).squeeze(-1)\n            final_loss = float(loss_fn(final_logits, labels).item())\n            final_probs = torch.sigmoid(final_logits).cpu().numpy()\n            final_metrics = AssociationEvaluator.compute_metrics(labels.cpu().numpy(), final_probs)\n\n        success = bool(final_loss < init_loss and final_metrics.roc_auc >= init_metrics.roc_auc)\n\n        return OverfitTestResult(\n            initial_loss=init_loss,\n            final_loss=final_loss,\n            initial_mean_pos_prob=init_metrics.mean_prob_positive,\n            final_mean_pos_prob=final_metrics.mean_prob_positive,\n            initial_mean_neg_prob=init_metrics.mean_prob_negative,\n            final_mean_neg_prob=final_metrics.mean_prob_negative,\n            initial_roc_auc=init_metrics.roc_auc,\n            final_roc_auc=final_metrics.roc_auc,\n            overfit_success=success\n        )\n\n    def train(\n        self,\n        dataset: AssociationDataset,\n        train_mask: np.ndarray,\n        val_mask: np.ndarray,\n        epochs: Optional[int] = None\n    ) -> Dict[str, Any]:\n        """\n        Trains STGNN model using train split and evaluates validation split per epoch.\n        """\n        n_epochs = epochs if epochs is not None else self.config.epochs\n        train_tensors = dataset.get_split_tensors(train_mask, device=str(self.device))\n        val_tensors = dataset.get_split_tensors(val_mask, device=str(self.device))\n\n        optimizer = AdamW(\n            self.model.parameters(),\n            lr=self.config.learning_rate,\n            weight_decay=self.config.weight_decay\n        )\n\n        pos_weight_tensor = None\n        if self.config.pos_weight is not None:\n            pos_weight_tensor = torch.tensor([self.config.pos_weight], device=self.device)\n\n        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)\n\n        history = {\n            "train_loss": [],\n            "val_loss": [],\n            "val_roc_auc": [],\n            "val_f1": []\n        }\n\n        best_val_loss = float("inf")\n        best_model_state = None\n\n        for epoch in range(1, n_epochs + 1):\n            # Training step\n            self.model.train()\n            optimizer.zero_grad()\n\n            tr_logits = self.model(\n                train_tensors["node_feats"],\n                train_tensors["edge_index"],\n                train_tensors["edge_feats"]\n            ).squeeze(-1)\n\n            tr_loss = loss_fn(tr_logits, train_tensors["labels"])\n            tr_loss.backward()\n            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)\n            optimizer.step()\n\n            # Validation step\n            self.model.eval()\n            with torch.no_grad():\n                val_logits = self.model(\n                    val_tensors["node_feats"],\n                    val_tensors["edge_index"],\n                    val_tensors["edge_feats"]\n                ).squeeze(-1)\n\n                val_loss = float(loss_fn(val_logits, val_tensors["labels"]).item())\n                val_probs = torch.sigmoid(val_logits).cpu().numpy()\n                val_metrics = AssociationEvaluator.compute_metrics(val_tensors["labels"].cpu().numpy(), val_probs)\n\n            history["train_loss"].append(float(tr_loss.item()))\n            history["val_loss"].append(val_loss)\n            history["val_roc_auc"].append(val_metrics.roc_auc)\n            history["val_f1"].append(val_metrics.f1)\n\n            if val_loss < best_val_loss:\n                best_val_loss = val_loss\n                best_model_state = copy.deepcopy(self.model.state_dict())\n\n        if best_model_state is not None:\n            self.model.load_state_dict(best_model_state)\n\n        return history\n'
src_files['src/config.py'] = '"""\nConfiguration specifications for spatial resolution, dataset paths, and coordinate systems.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import Tuple, Dict, Any, Optional\nfrom pathlib import Path\n\n\n@dataclass(frozen=True)\nclass SpatialResolution:\n    """\n    Explicit physical voxel scale configuration (in micrometers).\n\n    Default competition values:\n    - Z spacing: 1.625 um/voxel\n    - Y spacing: 0.40625 um/voxel\n    - X spacing: 0.40625 um/voxel\n    """\n    z_spacing: float = 1.625   # micrometers per voxel along Z\n    y_spacing: float = 0.40625 # micrometers per voxel along Y\n    x_spacing: float = 0.40625 # micrometers per voxel along X\n\n    @property\n    def scale_tuple_zyx(self) -> Tuple[float, float, float]:\n        """Returns physical scaling factors in (Z, Y, X) order."""\n        return (self.z_spacing, self.y_spacing, self.x_spacing)\n\n    @property\n    def scale_tuple_xyz(self) -> Tuple[float, float, float]:\n        """Returns physical scaling factors in (X, Y, Z) order."""\n        return (self.x_spacing, self.y_spacing, self.z_spacing)\n\n    @property\n    def anisotropy_ratio_z_to_xy(self) -> float:\n        """Ratio of Z resolution to X/Y resolution (anisotropy factor = 4.0)."""\n        return self.z_spacing / self.x_spacing\n\n\n@dataclass\nclass DatasetConfig:\n    """\n    Configuration for dataset paths, expected dimensions, and metadata.\n    """\n    zarr_path: Path\n    geff_path: Path\n    resolution: SpatialResolution = SpatialResolution()\n    expected_shape_tzyx: Tuple[int, int, int, int] = (100, 64, 256, 256)\n    expected_dtype: str = "uint16"\n\n    @classmethod\n    def default_local_sample(\n        cls,\n        base_dir: Optional[Path] = None,\n        sample_id: str = "6bba_05b6850b"\n    ) -> "DatasetConfig":\n        """Returns config instance pointing to the local development sample."""\n        if base_dir is None:\n            base_dir = Path("/home/naya/Documents/BioHub_Cell_Tracking_Project/train")\n        return cls(\n            zarr_path=base_dir / f"{sample_id}.zarr",\n            geff_path=base_dir / f"{sample_id}.geff",\n            resolution=SpatialResolution(),\n        )\n'
src_files['src/data/__init__.py'] = '"""\nData loading and coordinate conversion utilities.\n"""\n\nfrom .coordinates import CoordinateTransformer, voxel_to_physical, physical_to_voxel\nfrom .nodes import CellNode\nfrom .zarr_loader import ZarrLoader\nfrom .geff_loader import GeffLoader\n\n__all__ = [\n    "CoordinateTransformer",\n    "voxel_to_physical",\n    "physical_to_voxel",\n    "CellNode",\n    "ZarrLoader",\n    "GeffLoader",\n]\n'
src_files['src/data/coordinates.py'] = '"""\nCoordinate conventions, transformations, and distance metrics.\n\nCoordinate System Conventions:\n-----------------------------\n1. Voxel Coordinate System:\n   - Tuple: (z, y, x)\n   - Type: Discrete integer pixel indices on 3D grid.\n   - Bounds: 0 <= z < 64, 0 <= y < 256, 0 <= x < 256.\n\n2. Physical Coordinate System:\n   - Tuple: (z_phys, y_phys, x_phys)\n   - Type: Continuous floating-point physical coordinates in micrometers (um).\n   - Scaling:\n       z_phys = z * z_spacing (1.625 um)\n       y_phys = y * y_spacing (0.40625 um)\n       x_phys = x * x_spacing (0.40625 um)\n   - Z-Anisotropy Ratio: 1.625 / 0.40625 = 4.0\n"""\n\nfrom typing import Tuple, Union\nimport math\nimport numpy as np\nfrom src.config import SpatialResolution\n\n\nclass CoordinateTransformer:\n    """\n    Handles bidirectional coordinate conversions between 3D voxel grid indices (Z, Y, X)\n    and physical space (Z_phys, Y_phys, X_phys) in micrometers.\n    """\n\n    def __init__(self, resolution: SpatialResolution = SpatialResolution()):\n        self.resolution = resolution\n\n    def voxel_to_physical(\n        self, voxel_zyx: Tuple[Union[int, float], Union[int, float], Union[int, float]]\n    ) -> Tuple[float, float, float]:\n        """\n        Converts 3D voxel coordinates (z, y, x) to physical coordinates in micrometers (z_phys, y_phys, x_phys).\n        """\n        z, y, x = voxel_zyx\n        z_phys = float(z) * self.resolution.z_spacing\n        y_phys = float(y) * self.resolution.y_spacing\n        x_phys = float(x) * self.resolution.x_spacing\n        return (z_phys, y_phys, x_phys)\n\n    def physical_to_voxel(\n        self,\n        physical_zyx: Tuple[float, float, float],\n        mode: str = "round"\n    ) -> Tuple[int, int, int]:\n        """\n        Converts physical coordinates (z_phys, y_phys, x_phys) to discrete voxel indices (z, y, x).\n\n        :param physical_zyx: (z_phys, y_phys, x_phys) in micrometers\n        :param mode: Rounding mode - \'round\', \'floor\', or \'ceil\'\n        """\n        z_p, y_p, x_p = physical_zyx\n        z_f = z_p / self.resolution.z_spacing\n        y_f = y_p / self.resolution.y_spacing\n        x_f = x_p / self.resolution.x_spacing\n\n        if mode == "round":\n            return (int(round(z_f)), int(round(y_f)), int(round(x_f)))\n        elif mode == "floor":\n            return (int(math.floor(z_f)), int(math.floor(y_f)), int(math.floor(x_f)))\n        elif mode == "ceil":\n            return (int(math.ceil(z_f)), int(math.ceil(y_f)), int(math.ceil(x_f)))\n        else:\n            raise ValueError(f"Unknown rounding mode: {mode}. Use \'round\', \'floor\', or \'ceil\'.")\n\n    def euclidean_distance_physical(\n        self,\n        p1_zyx: Tuple[float, float, float],\n        p2_zyx: Tuple[float, float, float]\n    ) -> float:\n        """Computes true Euclidean distance in physical space (micrometers)."""\n        dz = p1_zyx[0] - p2_zyx[0]\n        dy = p1_zyx[1] - p2_zyx[1]\n        dx = p1_zyx[2] - p2_zyx[2]\n        return float(math.sqrt(dz * dz + dy * dy + dx * dx))\n\n    def anisotropic_voxel_distance(\n        self,\n        v1_zyx: Tuple[int, int, int],\n        v2_zyx: Tuple[int, int, int]\n    ) -> float:\n        """\n        Computes physical distance directly from voxel indices by applying anisotropy scaling.\n        """\n        p1 = self.voxel_to_physical(v1_zyx)\n        p2 = self.voxel_to_physical(v2_zyx)\n        return self.euclidean_distance_physical(p1, p2)\n\n\n# Convenience module-level functions using default competition resolution\n_default_transformer = CoordinateTransformer()\n\n\ndef voxel_to_physical(\n    voxel_zyx: Tuple[Union[int, float], Union[int, float], Union[int, float]]\n) -> Tuple[float, float, float]:\n    """Convenience function using default competition resolution."""\n    return _default_transformer.voxel_to_physical(voxel_zyx)\n\n\ndef physical_to_voxel(\n    physical_zyx: Tuple[float, float, float], mode: str = "round"\n) -> Tuple[int, int, int]:\n    """Convenience function using default competition resolution."""\n    return _default_transformer.physical_to_voxel(physical_zyx, mode=mode)\n'
src_files['src/data/geff_loader.py'] = '"""\nGEFF (Graph Exchange Format for Functional/Biological Networks) ground-truth tracking loader.\n"""\n\nfrom typing import List, Dict, Tuple, Optional, Set, Union\nfrom pathlib import Path\nimport numpy as np\nimport zarr\nimport networkx as nx\n\nfrom src.data.nodes import CellNode\nfrom src.data.coordinates import CoordinateTransformer\n\n\nclass GeffLoader:\n    """\n    Loads cell node centroids and directed temporal tracking trajectories from GEFF zarr stores.\n    """\n\n    def __init__(\n        self,\n        geff_path: Union[str, Path],\n        transformer: Optional[CoordinateTransformer] = None\n    ):\n        self.geff_path = Path(geff_path)\n        self.transformer = transformer or CoordinateTransformer()\n\n        if not self.geff_path.exists():\n            raise FileNotFoundError(f"GEFF directory not found at: {self.geff_path}")\n\n        self._root = zarr.open(str(self.geff_path), mode="r")\n\n        # Load raw backing Zarr arrays\n        self.node_ids = np.asarray(self._root["nodes/ids"][:])\n        self.t_values = np.asarray(self._root["nodes/props/t/values"][:])\n        self.z_values = np.asarray(self._root["nodes/props/z/values"][:])\n        self.y_values = np.asarray(self._root["nodes/props/y/values"][:])\n        self.x_values = np.asarray(self._root["nodes/props/x/values"][:])\n\n        # Load directed edge pairs (parent_id -> child_id)\n        if "edges/ids" in self._root:\n            self.directed_edges = np.asarray(self._root["edges/ids"][:])\n        else:\n            self.directed_edges = np.empty((0, 2), dtype=np.uint64)\n\n        # Build fast lookup indexes\n        self._id_to_index: Dict[int, int] = {\n            int(nid): idx for idx, nid in enumerate(self.node_ids)\n        }\n        self._frame_to_indices: Dict[int, List[int]] = {}\n        for idx, t in enumerate(self.t_values):\n            t_int = int(t)\n            if t_int not in self._frame_to_indices:\n                self._frame_to_indices[t_int] = []\n            self._frame_to_indices[t_int].append(idx)\n\n    @property\n    def num_nodes(self) -> int:\n        """Returns total number of cell nodes across all frames."""\n        return len(self.node_ids)\n\n    @property\n    def num_edges(self) -> int:\n        """Returns total number of directed temporal edges."""\n        return len(self.directed_edges)\n\n    @property\n    def temporal_range(self) -> Tuple[int, int]:\n        """Returns (min_frame, max_frame) tuple."""\n        return (int(self.t_values.min()), int(self.t_values.max()))\n\n    def get_node_by_id(self, node_id: int) -> Optional[CellNode]:\n        """Retrieves a single CellNode by its unique node ID."""\n        idx = self._id_to_index.get(int(node_id))\n        if idx is None:\n            return None\n\n        voxel_pos = (int(self.z_values[idx]), int(self.y_values[idx]), int(self.x_values[idx]))\n        return CellNode.create(\n            node_id=int(self.node_ids[idx]),\n            frame=int(self.t_values[idx]),\n            voxel_pos=voxel_pos,\n            transformer=self.transformer\n        )\n\n    def get_nodes_by_frame(self, t: int) -> List[CellNode]:\n        """Retrieves all CellNodes active at timeframe t."""\n        indices = self._frame_to_indices.get(int(t), [])\n        nodes = []\n        for idx in indices:\n            voxel_pos = (int(self.z_values[idx]), int(self.y_values[idx]), int(self.x_values[idx]))\n            nodes.append(\n                CellNode.create(\n                    node_id=int(self.node_ids[idx]),\n                    frame=int(self.t_values[idx]),\n                    voxel_pos=voxel_pos,\n                    transformer=self.transformer\n                )\n            )\n        return nodes\n\n    def get_all_nodes(self) -> List[CellNode]:\n        """Retrieves all CellNodes in dataset."""\n        nodes = []\n        for idx in range(self.num_nodes):\n            voxel_pos = (int(self.z_values[idx]), int(self.y_values[idx]), int(self.x_values[idx]))\n            nodes.append(\n                CellNode.create(\n                    node_id=int(self.node_ids[idx]),\n                    frame=int(self.t_values[idx]),\n                    voxel_pos=voxel_pos,\n                    transformer=self.transformer\n                )\n            )\n        return nodes\n\n    def to_networkx_graph(self) -> nx.DiGraph:\n        """\n        Constructs a NetworkX directed graph representation of cell trajectories.\n\n        Node attributes:\n        - frame (t)\n        - voxel_pos (z, y, x)\n        - physical_pos (z_phys, y_phys, x_phys)\n\n        Edge attributes:\n        - temporal_delta (t_child - t_parent)\n        - physical_distance (Euclidean physical distance between centroids)\n        """\n        graph = nx.DiGraph()\n\n        # Add nodes\n        for idx in range(self.num_nodes):\n            nid = int(self.node_ids[idx])\n            t = int(self.t_values[idx])\n            v_pos = (int(self.z_values[idx]), int(self.y_values[idx]), int(self.x_values[idx]))\n            p_pos = self.transformer.voxel_to_physical(v_pos)\n\n            graph.add_node(\n                nid,\n                frame=t,\n                voxel_pos=v_pos,\n                physical_pos=p_pos,\n                z_phys=p_pos[0],\n                y_phys=p_pos[1],\n                x_phys=p_pos[2],\n            )\n\n        # Add directed edges\n        for u, v in self.directed_edges:\n            u_id, v_id = int(u), int(v)\n            if u_id in graph and v_id in graph:\n                p_u = graph.nodes[u_id]["physical_pos"]\n                p_v = graph.nodes[v_id]["physical_pos"]\n                t_u = graph.nodes[u_id]["frame"]\n                t_v = graph.nodes[v_id]["frame"]\n                dist = self.transformer.euclidean_distance_physical(p_u, p_v)\n\n                graph.add_edge(\n                    u_id,\n                    v_id,\n                    temporal_delta=t_v - t_u,\n                    physical_distance=dist\n                )\n\n        return graph\n'
src_files['src/data/nodes.py'] = '"""\nInternal canonical cell node data representation.\n"""\n\nfrom dataclasses import dataclass, field\nfrom typing import Tuple, Dict, Any, Optional\nfrom src.data.coordinates import CoordinateTransformer\n\n\n@dataclass\nclass CellNode:\n    """\n    Internal canonical representation of a detected cell centroid at a specific timeframe.\n    """\n    node_id: int\n    frame: int  # Time index t (0-indexed)\n    voxel_pos: Tuple[int, int, int]  # (z, y, x) discrete voxel coordinates\n    physical_pos: Tuple[float, float, float]  # (z_phys, y_phys, x_phys) in micrometers\n    attributes: Dict[str, Any] = field(default_factory=dict)\n\n    @classmethod\n    def create(\n        cls,\n        node_id: int,\n        frame: int,\n        voxel_pos: Tuple[int, int, int],\n        transformer: Optional[CoordinateTransformer] = None,\n        attributes: Optional[Dict[str, Any]] = None\n    ) -> "CellNode":\n        """Factory method to construct a CellNode automatically calculating physical_pos."""\n        if transformer is None:\n            transformer = CoordinateTransformer()\n        phys_pos = transformer.voxel_to_physical(voxel_pos)\n        return cls(\n            node_id=node_id,\n            frame=frame,\n            voxel_pos=voxel_pos,\n            physical_pos=phys_pos,\n            attributes=attributes or {}\n        )\n\n    def distance_physical(self, other: "CellNode") -> float:\n        """Compute Euclidean distance to another node in physical space (micrometers)."""\n        dz = self.physical_pos[0] - other.physical_pos[0]\n        dy = self.physical_pos[1] - other.physical_pos[1]\n        dx = self.physical_pos[2] - other.physical_pos[2]\n        return (dz * dz + dy * dy + dx * dx) ** 0.5\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Returns node representation as dictionary."""\n        return {\n            "node_id": self.node_id,\n            "frame": self.frame,\n            "z_voxel": self.voxel_pos[0],\n            "y_voxel": self.voxel_pos[1],\n            "x_voxel": self.voxel_pos[2],\n            "z_phys": self.physical_pos[0],\n            "y_phys": self.physical_pos[1],\n            "x_phys": self.physical_pos[2],\n            "attributes": self.attributes,\n        }\n'
src_files['src/data/zarr_loader.py'] = '"""\nLazy and chunked Zarr microscopy volume data loader.\n"""\n\nfrom typing import Tuple, Optional, Dict, Any, Union\nfrom pathlib import Path\nimport numpy as np\nimport zarr\n\n\nclass ZarrLoader:\n    """\n    Provides memory-efficient lazy loading of 4D microscopy volumes stored in Zarr format.\n\n    Guarantees:\n    - Never loads the complete 4D Zarr volume into RAM.\n    - Uses chunk-level indexing for retrieving individual 3D temporal frames or spatial crops.\n    """\n\n    def __init__(self, zarr_path: Union[str, Path], level_key: str = "0"):\n        self.zarr_path = Path(zarr_path)\n        self.level_key = level_key\n\n        if not self.zarr_path.exists():\n            raise FileNotFoundError(f"Zarr directory not found at: {self.zarr_path}")\n\n        # Open root zarr group/array lazily\n        self._root = zarr.open(str(self.zarr_path), mode="r")\n\n        # Handle multiscale group or direct array\n        if hasattr(self._root, "__getitem__") and self.level_key in self._root:\n            self._array = self._root[self.level_key]\n        elif isinstance(self._root, zarr.Array):\n            self._array = self._root\n        else:\n            # Fallback: search for first array in group\n            for key in self._root.keys():\n                if isinstance(self._root[key], zarr.Array):\n                    self._array = self._root[key]\n                    break\n            else:\n                raise ValueError(f"Could not locate microscopy array inside Zarr group at {self.zarr_path}")\n\n    @property\n    def shape(self) -> Tuple[int, int, int, int]:\n        """Returns (T, Z, Y, X) shape tuple."""\n        return tuple(self._array.shape)\n\n    @property\n    def dtype(self) -> np.dtype:\n        """Returns array data type (e.g., uint16)."""\n        return self._array.dtype\n\n    @property\n    def chunks(self) -> Tuple[int, ...]:\n        """Returns chunk shape tuple."""\n        return tuple(self._array.chunks)\n\n    def get_frame(self, t: int) -> np.ndarray:\n        """\n        Retrieves the 3D volumetric stack (Z, Y, X) for a single timeframe t.\n\n        :param t: Time index (0-indexed)\n        :return: 3D numpy array of shape (Z, Y, X)\n        """\n        if t < 0 or t >= self.shape[0]:\n            raise IndexError(f"Time index t={t} out of bounds for dataset with T={self.shape[0]}")\n        # Lazy slice indexing loads only chunk t\n        return np.asarray(self._array[t, :, :, :])\n\n    def get_crop(\n        self,\n        t: int,\n        z_range: Tuple[int, int],\n        y_range: Tuple[int, int],\n        x_range: Tuple[int, int]\n    ) -> np.ndarray:\n        """\n        Retrieves a 3D subvolume crop for frame t without loading unneeded voxels.\n\n        :param t: Time index\n        :param z_range: (z_start, z_end) slice bounds\n        :param y_range: (y_start, y_end) slice bounds\n        :param x_range: (x_start, x_end) slice bounds\n        :return: 3D numpy array of shape (dz, dy, dx)\n        """\n        if t < 0 or t >= self.shape[0]:\n            raise IndexError(f"Time index t={t} out of bounds")\n\n        z_start, z_end = z_range\n        y_start, y_end = y_range\n        x_start, x_end = x_range\n\n        return np.asarray(self._array[t, z_start:z_end, y_start:y_end, x_start:x_end])\n\n    def get_intensity_at_voxel(self, t: int, z: int, y: int, x: int) -> int:\n        """Retrieves raw intensity value at specific (t, z, y, x) voxel location."""\n        return int(self._array[t, z, y, x])\n\n    def close(self):\n        """Closes any open zarr resources if needed."""\n        pass\n'
src_files['src/detection/__init__.py'] = '"""\nCell-Center Detection Module for 3D Microscopy Volumes.\n"""\n\nfrom .base import BaseDetector, DetectionCandidate, HeuristicIntensityDetector\nfrom .targets import TargetGenerator, GaussianHeatmapConfig\nfrom .dataset import Frame3DDataset\nfrom .unet3d import UNet3D\nfrom .peaks import PeakExtractor, PeakConfig\nfrom .evaluation import DetectionEvaluator, DetectionMetrics\nfrom .trainer import DetectorTrainer, DetectorTrainingConfig\n\n__all__ = [\n    "BaseDetector",\n    "DetectionCandidate",\n    "HeuristicIntensityDetector",\n    "TargetGenerator",\n    "GaussianHeatmapConfig",\n    "Frame3DDataset",\n    "UNet3D",\n    "PeakExtractor",\n    "PeakConfig",\n    "DetectionEvaluator",\n    "DetectionMetrics",\n    "DetectorTrainer",\n    "DetectorTrainingConfig",\n]\n\n'
src_files['src/detection/base.py'] = '"""\nAbstract Base Class and data structures for 3D Cell-Center Detectors.\n"""\n\nfrom abc import ABC, abstractmethod\nfrom dataclasses import dataclass, field\nfrom typing import Tuple, List, Dict, Any, Optional\nimport numpy as np\n\nfrom src.data.nodes import CellNode\nfrom src.data.coordinates import CoordinateTransformer\n\n\n@dataclass\nclass DetectionCandidate:\n    """\n    Represents a detected cell-center candidate before temporal graph association.\n    """\n    candidate_id: int\n    frame: int  # Time index t (0-indexed)\n    voxel_pos: Tuple[int, int, int]  # (z, y, x) in voxel indices\n    physical_pos: Tuple[float, float, float]  # (z_phys, y_phys, x_phys) in micrometers\n    confidence: float  # Model output response score in [0, 1]\n    attributes: Dict[str, Any] = field(default_factory=dict)\n\n    def to_cell_node(self) -> CellNode:\n        """Converts detection candidate to internal CellNode representation."""\n        attrs = dict(self.attributes)\n        attrs["confidence"] = self.confidence\n        return CellNode(\n            node_id=self.candidate_id,\n            frame=self.frame,\n            voxel_pos=self.voxel_pos,\n            physical_pos=self.physical_pos,\n            attributes=attrs\n        )\n\n\nclass BaseDetector(ABC):\n    """\n    Abstract interface for 3D cell-center detectors.\n    Decouples detector implementations (e.g. U-Net3D, Keypoint Net, classical peak finders)\n    from downstream temporal graph and tracking modules.\n    """\n\n    @abstractmethod\n    def predict_heatmap(self, volume_3d: np.ndarray) -> np.ndarray:\n        """\n        Predicts 3D continuous cell-center response heatmap from 3D microscopy image.\n\n        :param volume_3d: 3D input numpy array of shape (Z, Y, X), normalized in [0, 1]\n        :return: 3D heatmap array of shape (Z, Y, X) with values in [0, 1]\n        """\n        pass\n\n    @abstractmethod\n    def detect_candidates(\n        self,\n        volume_3d: np.ndarray,\n        frame_idx: int,\n        transformer: Optional[CoordinateTransformer] = None\n    ) -> List[DetectionCandidate]:\n        """\n        Full detection pipeline: predicts heatmap and extracts 3D peak candidates.\n\n        :param volume_3d: 3D input volume of shape (Z, Y, X)\n        :param frame_idx: Frame/time index t\n        :param transformer: CoordinateTransformer instance for voxel-to-physical conversion\n        :return: List of DetectionCandidate instances\n        """\n        pass\n\n\nclass HeuristicIntensityDetector(BaseDetector):\n    """\n    Intensity-based 3D peak detector for volumes without ground truth annotations.\n    """\n\n    def __init__(self, min_confidence: float = 0.1, min_physical_distance_um: float = 3.0):\n        from src.detection.peaks import PeakExtractor, PeakConfig\n        self.peak_config = PeakConfig(min_confidence=min_confidence, min_physical_distance_um=min_physical_distance_um)\n        self.peak_extractor = PeakExtractor(config=self.peak_config)\n\n    def predict_heatmap(self, volume_3d: np.ndarray) -> np.ndarray:\n        v_min, v_max = float(np.min(volume_3d)), float(np.max(volume_3d))\n        if v_max > v_min:\n            return (volume_3d.astype(np.float32) - v_min) / (v_max - v_min + 1e-6)\n        return np.zeros_like(volume_3d, dtype=np.float32)\n\n    def detect_candidates(\n        self,\n        volume_3d: np.ndarray,\n        frame_idx: int,\n        transformer: Optional[CoordinateTransformer] = None\n    ) -> List[DetectionCandidate]:\n        heatmap = self.predict_heatmap(volume_3d)\n        return self.peak_extractor.extract_peaks_from_heatmap(heatmap, frame_idx)\n\n'
src_files['src/detection/dataset.py'] = '"""\nPyTorch Dataset abstraction for 3D microscopy frames and target heatmaps.\n"""\n\nfrom typing import List, Tuple, Optional, Dict, Any, Union\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom src.data.zarr_loader import ZarrLoader\nfrom src.data.geff_loader import GeffLoader\nfrom src.detection.targets import TargetGenerator, GaussianHeatmapConfig\n\n\nclass Frame3DDataset(Dataset):\n    """\n    PyTorch Dataset for 3D microscopy frame-by-frame cell-center detection training and evaluation.\n\n    Features:\n    - Lazy loading: reads individual 3D frames or spatial subvolume crops from ZarrLoader.\n    - Intensity normalization: scales uint16 microscopy voxels to [0, 1].\n    - Target generation: generates anisotropic 3D Gaussian cell-center heatmaps on-the-fly.\n    - Configurable spatial subvolume cropping for fast CPU development/training.\n    """\n\n    def __init__(\n        self,\n        zarr_loader: ZarrLoader,\n        geff_loader: GeffLoader,\n        target_generator: Optional[TargetGenerator] = None,\n        frame_indices: Optional[List[int]] = None,\n        crop_bounds: Optional[Tuple[Tuple[int, int], Tuple[int, int], Tuple[int, int]]] = None,\n        norm_min: float = 0.0,\n        norm_max: float = 2000.0  # Clipping upper intensity threshold for normalization\n    ):\n        self.zarr_loader = zarr_loader\n        self.geff_loader = geff_loader\n        self.target_generator = target_generator or TargetGenerator()\n        self.crop_bounds = crop_bounds\n\n        if frame_indices is None:\n            self.frame_indices = list(range(self.zarr_loader.shape[0]))\n        else:\n            self.frame_indices = frame_indices\n\n        self.norm_min = norm_min\n        self.norm_max = norm_max\n\n    def __len__(self) -> int:\n        return len(self.frame_indices)\n\n    def _normalize_image(self, img_3d: np.ndarray) -> np.ndarray:\n        """Normalizes 3D uint16 microscopy image volume to float32 in [0, 1]."""\n        clipped = np.clip(img_3d.astype(np.float32), self.norm_min, self.norm_max)\n        normalized = (clipped - self.norm_min) / (self.norm_max - self.norm_min + 1e-6)\n        return normalized.astype(np.float32)\n\n    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, int]:\n        frame_t = self.frame_indices[idx]\n\n        # 1. Retrieve 3D volume or spatial crop lazily (Z, Y, X)\n        if self.crop_bounds is not None:\n            z_range, y_range, x_range = self.crop_bounds\n            img_3d = self.zarr_loader.get_crop(frame_t, z_range, y_range, x_range)\n            z_offset, y_offset, x_offset = z_range[0], y_range[0], x_range[0]\n        else:\n            img_3d = self.zarr_loader.get_frame(frame_t)\n            z_offset, y_offset, x_offset = 0, 0, 0\n\n        norm_img_3d = self._normalize_image(img_3d)\n\n        # 2. Retrieve ground truth nodes for frame_t\n        raw_nodes = self.geff_loader.get_nodes_by_frame(frame_t)\n\n        # Shift node coordinates if using spatial crop\n        shifted_nodes = []\n        for n in raw_nodes:\n            z_v, y_v, x_v = n.voxel_pos\n            if self.crop_bounds is not None:\n                (z_s, z_e), (y_s, y_e), (x_s, x_e) = self.crop_bounds\n                if not (z_s <= z_v < z_e and y_s <= y_v < y_e and x_s <= x_v < x_e):\n                    continue\n                z_v -= z_s\n                y_v -= y_s\n                x_v -= x_s\n\n            shifted_nodes.append(\n                n.create(\n                    node_id=n.node_id,\n                    frame=n.frame,\n                    voxel_pos=(z_v, y_v, x_v),\n                    transformer=self.geff_loader.transformer,\n                    attributes=n.attributes\n                )\n            )\n\n        # 3. Generate 3D target heatmap (Z, Y, X)\n        target_heatmap_3d = self.target_generator.generate_heatmap_for_frame(\n            nodes=shifted_nodes,\n            grid_shape=norm_img_3d.shape\n        )\n\n        # 4. Convert to PyTorch Tensors [Channels=1, Z, Y, X]\n        image_tensor = torch.from_numpy(norm_img_3d).unsqueeze(0)  # Shape: (1, Z, Y, X)\n        target_tensor = torch.from_numpy(target_heatmap_3d).unsqueeze(0)  # Shape: (1, Z, Y, X)\n\n        return image_tensor, target_tensor, frame_t\n'
src_files['src/detection/evaluation.py'] = '"""\nGround-Truth Detection Evaluation & Localization Error Analysis.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Dict, Any, Optional\nimport numpy as np\n\nfrom src.data.nodes import CellNode\nfrom src.data.coordinates import CoordinateTransformer\nfrom src.detection.base import DetectionCandidate\n\n\n@dataclass\nclass DetectionMetrics:\n    """\n    Spatial cell-center detection evaluation metrics.\n    Strictly isolated from temporal tracking/association performance.\n    """\n    num_gt_cells: int\n    num_pred_candidates: int\n    true_positives: int\n    false_positives: int\n    false_negatives: int\n    precision: float\n    recall: float\n    f1_score: float\n    mean_localization_error_um: float\n    median_localization_error_um: float\n\n    def summary_string(self) -> str:\n        return (\n            f"GT Cells: {self.num_gt_cells} | Pred Candidates: {self.num_pred_candidates}\\n"\n            f"TP: {self.true_positives} | FP: {self.false_positives} | FN: {self.false_negatives}\\n"\n            f"Precision: {self.precision:.4f} | Recall: {self.recall:.4f} | F1-Score: {self.f1_score:.4f}\\n"\n            f"Mean Localization Error: {self.mean_localization_error_um:.3f} um | "\n            f"Median Error: {self.median_localization_error_um:.3f} um"\n        )\n\n\nclass DetectionEvaluator:\n    """\n    Evaluates cell-center detection performance against ground-truth GEFF nodes using physical distance matching.\n    """\n\n    def __init__(\n        self,\n        max_matching_distance_um: float = 5.0,  # Matching distance threshold in micrometers (um)\n        transformer: Optional[CoordinateTransformer] = None\n    ):\n        self.max_matching_distance_um = max_matching_distance_um\n        self.transformer = transformer or CoordinateTransformer()\n\n    def evaluate_frame(\n        self,\n        gt_nodes: List[CellNode],\n        pred_candidates: List[DetectionCandidate]\n    ) -> DetectionMetrics:\n        """\n        Evaluates predictions for a single 3D frame.\n        """\n        n_gt = len(gt_nodes)\n        n_pred = len(pred_candidates)\n\n        if n_gt == 0 and n_pred == 0:\n            return DetectionMetrics(0, 0, 0, 0, 0, 1.0, 1.0, 1.0, 0.0, 0.0)\n        if n_gt == 0:\n            return DetectionMetrics(0, n_pred, 0, n_pred, 0, 0.0, 0.0, 0.0, 0.0, 0.0)\n        if n_pred == 0:\n            return DetectionMetrics(n_gt, 0, 0, 0, n_gt, 0.0, 0.0, 0.0, 0.0, 0.0)\n\n        # Build distance matrix (N_pred, N_gt) in physical micrometers\n        dist_matrix = np.zeros((n_pred, n_gt), dtype=np.float32)\n        for i, pred in enumerate(pred_candidates):\n            for j, gt in enumerate(gt_nodes):\n                dist = self.transformer.euclidean_distance_physical(pred.physical_pos, gt.physical_pos)\n                dist_matrix[i, j] = dist\n\n        # Greedy bipartite matching by physical distance ascending\n        pairs = []\n        for i in range(n_pred):\n            for j in range(n_gt):\n                d = dist_matrix[i, j]\n                if d <= self.max_matching_distance_um:\n                    pairs.append((d, i, j))\n\n        pairs.sort(key=lambda x: x[0])\n\n        matched_pred = set()\n        matched_gt = set()\n        matched_errors = []\n\n        for dist, i, j in pairs:\n            if i not in matched_pred and j not in matched_gt:\n                matched_pred.add(i)\n                matched_gt.add(j)\n                matched_errors.append(dist)\n\n        tp = len(matched_pred)\n        fp = n_pred - tp\n        fn = n_gt - tp\n\n        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0\n\n        mean_err = float(np.mean(matched_errors)) if matched_errors else 0.0\n        median_err = float(np.median(matched_errors)) if matched_errors else 0.0\n\n        return DetectionMetrics(\n            num_gt_cells=n_gt,\n            num_pred_candidates=n_pred,\n            true_positives=tp,\n            false_positives=fp,\n            false_negatives=fn,\n            precision=precision,\n            recall=recall,\n            f1_score=f1,\n            mean_localization_error_um=mean_err,\n            median_localization_error_um=median_err\n        )\n'
src_files['src/detection/peaks.py'] = '"""\nDeterministic 3D Local Maxima Extraction and Anisotropic Non-Maximum Suppression (NMS).\n"""\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Optional\nimport numpy as np\nfrom scipy.ndimage import maximum_filter\n\nfrom src.config import SpatialResolution\nfrom src.data.coordinates import CoordinateTransformer\nfrom src.detection.base import DetectionCandidate\n\n\n@dataclass\nclass PeakConfig:\n    """\n    Configuration for 3D peak extraction and non-maximum suppression (NMS).\n    """\n    min_confidence: float = 0.2           # Heatmap response threshold for candidate peaks\n    min_physical_distance_um: float = 3.0  # Minimum physical NMS radius in micrometers (um)\n    max_candidates_per_frame: int = 200    # Safety cap on maximum candidates per frame\n\n\nclass PeakExtractor:\n    """\n    Extracts candidate cell centers from continuous 3D predicted heatmaps using anisotropic local maxima filtering and NMS.\n    """\n\n    def __init__(\n        self,\n        config: PeakConfig = PeakConfig(),\n        resolution: SpatialResolution = SpatialResolution()\n    ):\n        self.config = config\n        self.resolution = resolution\n        self.transformer = CoordinateTransformer(resolution)\n\n    def extract_peaks_from_heatmap(\n        self,\n        heatmap_3d: np.ndarray,\n        frame_idx: int,\n        id_start_offset: int = 1000\n    ) -> List[DetectionCandidate]:\n        """\n        Extracts peak candidate nodes from a 3D heatmap array (Z, Y, X).\n\n        :param heatmap_3d: 3D float numpy array of shape (Z, Y, X)\n        :param frame_idx: Frame index t\n        :param id_start_offset: Integer offset for generating candidate IDs\n        :return: List of DetectionCandidate instances\n        """\n        if heatmap_3d.ndim != 3:\n            raise ValueError(f"Expected 3D heatmap array, got shape {heatmap_3d.shape}")\n\n        # Compute anisotropic maximum filter footprint size in voxel indices\n        # Size along axis = 2 * (dist / spacing) + 1\n        d_um = self.config.min_physical_distance_um\n        size_z = max(1, int(round(d_um / self.resolution.z_spacing)) | 1)\n        size_y = max(1, int(round(d_um / self.resolution.y_spacing)) | 1)\n        size_x = max(1, int(round(d_um / self.resolution.x_spacing)) | 1)\n\n        footprint = (size_z, size_y, size_x)\n\n        # 1. Apply 3D local maximum filter\n        local_max = maximum_filter(heatmap_3d, size=footprint, mode="constant", cval=0.0)\n\n        # 2. Thresholding: location must equal local maximum AND exceed min_confidence\n        mask = (heatmap_3d == local_max) & (heatmap_3d >= self.config.min_confidence)\n        z_indices, y_indices, x_indices = np.where(mask)\n\n        if len(z_indices) == 0:\n            return []\n\n        # 3. Gather initial candidates with confidence scores\n        raw_candidates = []\n        for z, y, x in zip(z_indices, y_indices, x_indices):\n            vz = int(z)\n            vy = int(y)\n            vx = int(x)\n            conf = float(heatmap_3d[vz, vy, vx])\n            phys_pos = self.transformer.voxel_to_physical((vz, vy, vx))\n            raw_candidates.append({\n                "voxel_pos": (vz, vy, vx),\n                "physical_pos": phys_pos,\n                "confidence": conf\n            })\n\n        # 4. Sort candidates by confidence descending for greedy NMS\n        raw_candidates.sort(key=lambda c: c["confidence"], reverse=True)\n\n        # 5. Greedy Anisotropic Non-Maximum Suppression (NMS) in physical space (um)\n        kept_candidates: List[DetectionCandidate] = []\n        suppressed = np.zeros(len(raw_candidates), dtype=bool)\n\n        for i, cand in enumerate(raw_candidates):\n            if suppressed[i]:\n                continue\n\n            c_id = id_start_offset + len(kept_candidates) + 1\n            det_cand = DetectionCandidate(\n                candidate_id=c_id,\n                frame=frame_idx,\n                voxel_pos=cand["voxel_pos"],\n                physical_pos=cand["physical_pos"],\n                confidence=cand["confidence"]\n            )\n            kept_candidates.append(det_cand)\n\n            if len(kept_candidates) >= self.config.max_candidates_per_frame:\n                break\n\n            # Suppress nearby lower-confidence peaks within physical distance\n            p_i = cand["physical_pos"]\n            for j in range(i + 1, len(raw_candidates)):\n                if suppressed[j]:\n                    continue\n                p_j = raw_candidates[j]["physical_pos"]\n                dist = self.transformer.euclidean_distance_physical(p_i, p_j)\n                if dist < self.config.min_physical_distance_um:\n                    suppressed[j] = True\n\n        return kept_candidates\n'
src_files['src/detection/targets.py'] = '"""\nDeterministic 3D Target Heatmap Generation with Physical Anisotropy Handling.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Optional\nimport numpy as np\n\nfrom src.config import SpatialResolution\nfrom src.data.nodes import CellNode\n\n\n@dataclass\nclass GaussianHeatmapConfig:\n    """\n    Configuration for 3D Gaussian cell-center target generation.\n\n    The Gaussian width is defined in PHYSICAL units (micrometers, um) to ensure\n    consistent spatial representation regardless of voxel resolution or anisotropy.\n    """\n    sigma_physical_um: float = 2.0  # Gaussian standard deviation in physical space (um)\n    cutoff_sigmas: float = 3.0       # Truncation radius in units of sigma\n    min_value_threshold: float = 0.01 # Minimum value cutoff for target rendering\n\n\nclass TargetGenerator:\n    """\n    Generates 3D continuous Gaussian cell-center target heatmaps from ground-truth node locations.\n    """\n\n    def __init__(\n        self,\n        config: GaussianHeatmapConfig = GaussianHeatmapConfig(),\n        resolution: SpatialResolution = SpatialResolution()\n    ):\n        self.config = config\n        self.resolution = resolution\n\n        # Calculate anisotropic voxel-space standard deviations\n        # sigma_voxel = sigma_physical / spacing\n        self.sigma_z = self.config.sigma_physical_um / self.resolution.z_spacing\n        self.sigma_y = self.config.sigma_physical_um / self.resolution.y_spacing\n        self.sigma_x = self.config.sigma_physical_um / self.resolution.x_spacing\n\n    def generate_heatmap_for_frame(\n        self,\n        nodes: List[CellNode],\n        grid_shape: Tuple[int, int, int]\n    ) -> np.ndarray:\n        """\n        Generates 3D heatmap array of shape (Z, Y, X) for a set of cell nodes in a frame.\n\n        :param nodes: List of CellNode objects in the target frame\n        :param grid_shape: (Z, Y, X) output volume dimensions\n        :return: 3D float32 numpy array of shape (Z, Y, X) with values in [0, 1]\n        """\n        z_dim, y_dim, x_dim = grid_shape\n        heatmap = np.zeros((z_dim, y_dim, x_dim), dtype=np.float32)\n\n        if not nodes:\n            return heatmap\n\n        # Compute bounding radii in voxel units\n        radius_z = int(np.ceil(self.config.cutoff_sigmas * self.sigma_z))\n        radius_y = int(np.ceil(self.config.cutoff_sigmas * self.sigma_y))\n        radius_x = int(np.ceil(self.config.cutoff_sigmas * self.sigma_x))\n\n        for node in nodes:\n            cz, cy, cx = node.voxel_pos\n\n            # Compute local ROI bounding box\n            z_min = max(0, cz - radius_z)\n            z_max = min(z_dim, cz + radius_z + 1)\n            y_min = max(0, cy - radius_y)\n            y_max = min(y_dim, cy + radius_y + 1)\n            x_min = max(0, cx - radius_x)\n            x_max = min(x_dim, cx + radius_x + 1)\n\n            if z_min >= z_max or y_min >= y_max or x_min >= x_max:\n                continue\n\n            # Create local coordinate grids\n            z_grid = np.arange(z_min, z_max, dtype=np.float32)[:, None, None]\n            y_grid = np.arange(y_min, y_max, dtype=np.float32)[None, :, None]\n            x_grid = np.arange(x_min, x_max, dtype=np.float32)[None, None, :]\n\n            # Anisotropic 3D Gaussian formula\n            dz2 = ((z_grid - cz) / self.sigma_z) ** 2\n            dy2 = ((y_grid - cy) / self.sigma_y) ** 2\n            dx2 = ((x_grid - cx) / self.sigma_x) ** 2\n\n            gaussian_roi = np.exp(-0.5 * (dz2 + dy2 + dx2)).astype(np.float32)\n\n            # Apply elementwise maximum to merge overlapping cell targets cleanly\n            heatmap[z_min:z_max, y_min:y_max, x_min:x_max] = np.maximum(\n                heatmap[z_min:z_max, y_min:y_max, x_min:x_max], gaussian_roi\n            )\n\n        return heatmap\n'
src_files['src/detection/trainer.py'] = '"""\nPyTorch Training and Controlled Overfit Evaluation Engine for 3D Cell Detectors.\n"""\n\nimport sys\nfrom dataclasses import dataclass\nfrom typing import List, Dict, Tuple, Any, Optional\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader, Subset\n\nfrom src.detection.unet3d import UNet3D\nfrom src.detection.dataset import Frame3DDataset\nfrom src.detection.peaks import PeakExtractor\nfrom src.detection.evaluation import DetectionEvaluator, DetectionMetrics\n\n\n@dataclass\nclass DetectorTrainingConfig:\n    """\n    Explicit training hyperparameter configuration.\n    """\n    epochs: int = 6\n    learning_rate: float = 5e-3\n    batch_size: int = 1\n    device: str = "cuda" if torch.cuda.is_available() else "cpu"\n    weight_decay: float = 1e-5\n\n\nclass DetectorTrainer:\n    """\n    Handles model optimization, training loops, and controlled overfit testing.\n    """\n\n    def __init__(\n        self,\n        model: UNet3D,\n        config: DetectorTrainingConfig = DetectorTrainingConfig()\n    ):\n        self.model = model\n        self.config = config\n        self.device = torch.device(config.device)\n        self.model.to(self.device)\n\n        self.optimizer = torch.optim.AdamW(\n            self.model.parameters(),\n            lr=config.learning_rate,\n            weight_decay=config.weight_decay\n        )\n        self.criterion = nn.MSELoss()\n\n    def train_epoch(self, dataloader: DataLoader) -> float:\n        """Runs a single training epoch."""\n        self.model.train()\n        total_loss = 0.0\n\n        for images, targets, _ in dataloader:\n            images = images.to(self.device)\n            targets = targets.to(self.device)\n\n            self.optimizer.zero_grad()\n            outputs = self.model(images)\n            loss = self.criterion(outputs, targets)\n            loss.backward()\n            self.optimizer.step()\n\n            total_loss += loss.item() * images.size(0)\n\n        return total_loss / len(dataloader.dataset)\n\n    def run_controlled_overfit_test(\n        self,\n        full_dataset: Frame3DDataset,\n        overfit_frames: List[int] = [0, 1, 2],\n        epochs: int = 6,\n        eval_interval: int = 2,\n        peak_extractor: Optional[PeakExtractor] = None,\n        evaluator: Optional[DetectionEvaluator] = None\n    ) -> Dict[str, Any]:\n        """\n        Executes a controlled overfit test on a small subset of frames to empirically verify\n        that the model learns microscopy -> heatmap center mapping.\n        """\n        indices = [i for i, f in enumerate(full_dataset.frame_indices) if f in overfit_frames]\n        subset_dataset = Subset(full_dataset, indices)\n        dataloader = DataLoader(subset_dataset, batch_size=1, shuffle=True)\n\n        peak_extractor = peak_extractor or PeakExtractor()\n        evaluator = evaluator or DetectionEvaluator()\n\n        history = {\n            "epochs": [],\n            "loss": [],\n            "precision": [],\n            "recall": [],\n            "f1_score": [],\n            "mean_loc_error_um": []\n        }\n\n        print(f"\\n--- Running Controlled Overfit Test ({epochs} epochs, frames {overfit_frames}) ---")\n        sys.stdout.flush()\n\n        for ep in range(1, epochs + 1):\n            avg_loss = self.train_epoch(dataloader)\n\n            # Evaluate on interval or final epoch\n            should_eval = (ep == 1 or ep % eval_interval == 0 or ep == epochs)\n\n            if should_eval:\n                self.model.eval()\n                all_metrics = []\n                with torch.no_grad():\n                    for idx in indices:\n                        img_tensor, _, frame_t = full_dataset[idx]\n                        img_input = img_tensor.unsqueeze(0).to(self.device)\n                        pred_heatmap = self.model(img_input).squeeze(0).squeeze(0).cpu().numpy()\n\n                        # Extract peaks\n                        candidates = peak_extractor.extract_peaks_from_heatmap(pred_heatmap, frame_t)\n\n                        # Get ground truth\n                        gt_nodes = full_dataset.geff_loader.get_nodes_by_frame(frame_t)\n\n                        # Evaluate matching\n                        metrics = evaluator.evaluate_frame(gt_nodes, candidates)\n                        all_metrics.append(metrics)\n\n                avg_prec = float(np.mean([m.precision for m in all_metrics]))\n                avg_rec = float(np.mean([m.recall for m in all_metrics]))\n                avg_f1 = float(np.mean([m.f1_score for m in all_metrics]))\n                avg_loc_err = float(np.mean([m.mean_localization_error_um for m in all_metrics]))\n\n                history["epochs"].append(ep)\n                history["loss"].append(avg_loss)\n                history["precision"].append(avg_prec)\n                history["recall"].append(avg_rec)\n                history["f1_score"].append(avg_f1)\n                history["mean_loc_error_um"].append(avg_loc_err)\n\n                print(\n                    f"Epoch {ep:2d}/{epochs:2d} | Loss: {avg_loss:.6f} | "\n                    f"Prec: {avg_prec:.3f} | Rec: {avg_rec:.3f} | F1: {avg_f1:.3f} | "\n                    f"Loc Error: {avg_loc_err:.2f} um"\n                )\n                sys.stdout.flush()\n            else:\n                history["epochs"].append(ep)\n                history["loss"].append(avg_loss)\n                print(f"Epoch {ep:2d}/{epochs:2d} | Loss: {avg_loss:.6f}")\n                sys.stdout.flush()\n\n        return history\n'
src_files['src/detection/unet3d.py'] = '"""\nCompact 3D U-Net Architecture for Volumetric Cell-Center Heatmap Prediction.\n"""\n\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass DoubleConv3D(nn.Module):\n    """(Conv3D -> BatchNorm3D -> LeakyReLU) * 2"""\n\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.double_conv = nn.Sequential(\n            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm3d(out_channels),\n            nn.LeakyReLU(0.1, inplace=True),\n            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm3d(out_channels),\n            nn.LeakyReLU(0.1, inplace=True),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.double_conv(x)\n\n\nclass Down3D(nn.Module):\n    """Downscaling with MaxPool3D then DoubleConv3D"""\n\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.maxpool_conv = nn.Sequential(\n            nn.MaxPool3d(kernel_size=2, stride=2),\n            DoubleConv3D(in_channels, out_channels)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.maxpool_conv(x)\n\n\nclass Up3D(nn.Module):\n    """Upscaling with Trilinear interpolation then DoubleConv3D"""\n\n    def __init__(self, in_channels: int, skip_channels: int, out_channels: int):\n        super().__init__()\n        self.up = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=True)\n        self.conv = DoubleConv3D(in_channels + skip_channels, out_channels)\n\n    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:\n        x1 = self.up(x1)\n\n        diff_z = x2.size(2) - x1.size(2)\n        diff_y = x2.size(3) - x1.size(3)\n        diff_x = x2.size(4) - x1.size(4)\n\n        if diff_z != 0 or diff_y != 0 or diff_x != 0:\n            x1 = F.pad(\n                x1,\n                [\n                    diff_x // 2, diff_x - diff_x // 2,\n                    diff_y // 2, diff_y - diff_y // 2,\n                    diff_z // 2, diff_z - diff_z // 2,\n                ]\n            )\n\n        x = torch.cat([x2, x1], dim=1)\n        return self.conv(x)\n\n\nclass UNet3D(nn.Module):\n    """\n    Compact 3D U-Net architecture for cell-center response prediction.\n\n    - Input: Tensor of shape (Batch, InChannels=1, Z, Y, X)\n    - Output: Tensor of shape (Batch, OutChannels=1, Z, Y, X) in range [0, 1]\n    """\n\n    def __init__(self, in_channels: int = 1, out_channels: int = 1, base_features: int = 12):\n        super().__init__()\n        self.in_channels = in_channels\n        self.out_channels = out_channels\n\n        # Encoder\n        self.inc = DoubleConv3D(in_channels, base_features)                # 1 -> 12\n        self.down1 = Down3D(base_features, base_features * 2)              # 12 -> 24\n        self.down2 = Down3D(base_features * 2, base_features * 4)          # 24 -> 48\n\n        # Decoder\n        self.up1 = Up3D(in_channels=base_features * 4, skip_channels=base_features * 2, out_channels=base_features * 2)\n        self.up2 = Up3D(in_channels=base_features * 2, skip_channels=base_features, out_channels=base_features)\n\n        # Output head\n        self.outc = nn.Conv3d(base_features, out_channels, kernel_size=1)\n        self.sigmoid = nn.Sigmoid()\n\n        # Heatmap bias initialization: initialize final bias to -4.0 (sigmoid(-4.0) ~ 0.018)\n        # Prevents initial loss explosion for sparse keypoint heatmaps\n        nn.init.constant_(self.outc.bias, -4.0)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x1 = self.inc(x)        # (B, 12, Z, Y, X)\n        x2 = self.down1(x1)     # (B, 24, Z/2, Y/2, X/2)\n        x3 = self.down2(x2)     # (B, 48, Z/4, Y/4, X/4)\n\n        x = self.up1(x3, x2)    # (B, 24, Z/2, Y/2, X/2)\n        x = self.up2(x, x1)     # (B, 12, Z, Y, X)\n        logits = self.outc(x)   # (B, 1, Z, Y, X)\n        return self.sigmoid(logits)\n\n    def count_parameters(self) -> int:\n        """Returns total number of trainable parameters."""\n        return sum(p.numel() for p in self.parameters() if p.requires_grad)\n'
src_files['src/graph/__init__.py'] = '"""\nGlobal Node Reconciliation, Temporal Candidate Graph, and Candidate Recall Diagnostics.\n"""\n\nfrom .nodes import CanonicalNode, NodeReconciler, NodeSet\nfrom .candidate_graph import (\n    CandidateEdge,\n    TemporalCandidateGraph,\n    CandidateGraphBuilder,\n    CandidateGraphConfig,\n)\nfrom .features import GraphFeatureExtractor, NodeFeatures, EdgeFeatures\nfrom .diagnostics import (\n    CandidateRecallEvaluator,\n    CandidateRecallMetrics,\n    NodeMatchingResult,\n)\n\n__all__ = [\n    "CanonicalNode",\n    "NodeReconciler",\n    "NodeSet",\n    "CandidateEdge",\n    "TemporalCandidateGraph",\n    "CandidateGraphBuilder",\n    "CandidateGraphConfig",\n    "GraphFeatureExtractor",\n    "NodeFeatures",\n    "EdgeFeatures",\n    "CandidateRecallEvaluator",\n    "CandidateRecallMetrics",\n    "NodeMatchingResult",\n]\n'
src_files['src/graph/candidate_graph.py'] = '"""\nTemporal Candidate Graph Construction and Representation.\n\nConnects detections across consecutive frames (t -> t+1) using configurable physical distance gating,\nadaptive displacement limits, and top-K candidate limits.\n"""\n\nfrom dataclasses import dataclass, field\nfrom typing import List, Tuple, Dict, Any, Optional\nimport numpy as np\n\nfrom src.graph.nodes import CanonicalNode, NodeSet\n\n\n@dataclass\nclass CandidateEdge:\n    """\n    Representation of a temporal candidate edge between consecutive frames.\n    """\n    source_node_id: int\n    destination_node_id: int\n    source_frame: int\n    destination_frame: int\n    frame_difference: int\n    physical_displacement_vector: Tuple[float, float, float]  # (dz_phys, dy_phys, dx_phys)\n    euclidean_distance_phys: float\n    voxel_displacement_vector: Tuple[float, float, float]  # (dz_vox, dy_vox, dx_vox)\n    source_confidence: float\n    destination_confidence: float\n    pair_confidence: float  # source_confidence * destination_confidence\n    local_density: Optional[float] = None\n    source_region_id: Optional[int] = None\n    destination_region_id: Optional[int] = None\n    gating_reason: str = "distance_gate"\n    passed_distance_gate: bool = True\n    rank_among_destinations: int = 1\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Convert edge to dictionary representation."""\n        return {\n            "source_node_id": self.source_node_id,\n            "destination_node_id": self.destination_node_id,\n            "source_frame": self.source_frame,\n            "destination_frame": self.destination_frame,\n            "frame_difference": self.frame_difference,\n            "dz_phys": self.physical_displacement_vector[0],\n            "dy_phys": self.physical_displacement_vector[1],\n            "dx_phys": self.physical_displacement_vector[2],\n            "euclidean_distance_phys": self.euclidean_distance_phys,\n            "dz_vox": self.voxel_displacement_vector[0],\n            "dy_vox": self.voxel_displacement_vector[1],\n            "dx_vox": self.voxel_displacement_vector[2],\n            "source_confidence": self.source_confidence,\n            "destination_confidence": self.destination_confidence,\n            "pair_confidence": self.pair_confidence,\n            "local_density": self.local_density,\n            "source_region_id": self.source_region_id,\n            "destination_region_id": self.destination_region_id,\n            "gating_reason": self.gating_reason,\n            "passed_distance_gate": self.passed_distance_gate,\n            "rank_among_destinations": self.rank_among_destinations,\n        }\n\n\n@dataclass\nclass CandidateGraphConfig:\n    """\n    Configuration parameters for temporal candidate graph generation.\n    """\n    max_physical_distance_um: float = 20.0  # Max physical distance threshold in micrometers\n    top_k_per_node: Optional[int] = 5  # Max candidate destinations per source node (None = no limit)\n    max_frame_gap: int = 1  # Maximum temporal frame difference (1 = consecutive frames)\n    adaptive_velocity_factor: Optional[float] = None  # Optional adaptive displacement multiplier\n\n\nclass TemporalCandidateGraph:\n    """\n    Typed native graph representation storing canonical nodes and temporal candidate edges.\n    Supports outgoing/incoming candidate queries, edge lookup, diagnostics, and tensor exports.\n    """\n\n    def __init__(\n        self,\n        node_set: NodeSet,\n        edges: List[CandidateEdge],\n        config: CandidateGraphConfig\n    ):\n        self.node_set = node_set\n        self.edges = edges\n        self.config = config\n\n        self._outgoing_edges: Dict[int, List[CandidateEdge]] = {}\n        self._incoming_edges: Dict[int, List[CandidateEdge]] = {}\n        self._edge_by_pair: Dict[Tuple[int, int], CandidateEdge] = {}\n\n        self._build_index()\n\n    def _build_index(self) -> None:\n        """Build fast lookup indexes for edges."""\n        for edge in self.edges:\n            src = edge.source_node_id\n            dst = edge.destination_node_id\n\n            if src not in self._outgoing_edges:\n                self._outgoing_edges[src] = []\n            self._outgoing_edges[src].append(edge)\n\n            if dst not in self._incoming_edges:\n                self._incoming_edges[dst] = []\n            self._incoming_edges[dst].append(edge)\n\n            self._edge_by_pair[(src, dst)] = edge\n\n    def get_nodes_by_frame(self, frame: int) -> List[CanonicalNode]:\n        """Retrieve all canonical nodes active at specified frame."""\n        return self.node_set.get_nodes_by_frame(frame)\n\n    def get_outgoing_edges(self, node_id: int) -> List[CandidateEdge]:\n        """Retrieve all outgoing candidate edges originating from node_id."""\n        return self._outgoing_edges.get(node_id, [])\n\n    def get_incoming_edges(self, node_id: int) -> List[CandidateEdge]:\n        """Retrieve all incoming candidate edges terminating at node_id."""\n        return self._incoming_edges.get(node_id, [])\n\n    def get_edge(self, source_id: int, destination_id: int) -> Optional[CandidateEdge]:\n        """Retrieve candidate edge between (source_id, destination_id)."""\n        return self._edge_by_pair.get((source_id, destination_id))\n\n    def candidate_count(self) -> int:\n        """Total number of candidate edges in graph."""\n        return len(self.edges)\n\n    def num_nodes(self) -> int:\n        """Total number of nodes in graph."""\n        return self.node_set.num_nodes()\n\n    def export_diagnostics(self) -> Dict[str, Any]:\n        """\n        Calculates and exports diagnostic statistics about the candidate graph.\n        """\n        all_nodes = self.node_set.all_nodes()\n        total_nodes = len(all_nodes)\n        if total_nodes == 0:\n            return {\n                "total_nodes": 0,\n                "total_candidate_edges": 0,\n                "mean_candidates_per_node": 0.0,\n                "zero_candidate_pct": 0.0,\n                "one_candidate_pct": 0.0,\n                "multi_candidate_pct": 0.0,\n            }\n\n        # Calculate candidate counts for source nodes (excluding last frame nodes)\n        max_frame = max(self.node_set.frames()) if self.node_set.frames() else 0\n        source_nodes = [n for n in all_nodes if n.frame < max_frame]\n        n_sources = len(source_nodes)\n\n        cand_counts = [len(self.get_outgoing_edges(n.node_id)) for n in source_nodes]\n        counts_arr = np.array(cand_counts, dtype=np.int32) if cand_counts else np.array([0])\n\n        zero_cnt = int(np.sum(counts_arr == 0))\n        one_cnt = int(np.sum(counts_arr == 1))\n        multi_cnt = int(np.sum(counts_arr > 1))\n\n        return {\n            "total_nodes": total_nodes,\n            "source_nodes_evaluated": n_sources,\n            "total_candidate_edges": len(self.edges),\n            "mean_candidates_per_node": float(np.mean(counts_arr)) if n_sources > 0 else 0.0,\n            "min_candidates_per_node": int(np.min(counts_arr)) if n_sources > 0 else 0,\n            "median_candidates_per_node": float(np.median(counts_arr)) if n_sources > 0 else 0.0,\n            "max_candidates_per_node": int(np.max(counts_arr)) if n_sources > 0 else 0,\n            "zero_candidate_pct": float(100.0 * zero_cnt / n_sources) if n_sources > 0 else 0.0,\n            "one_candidate_pct": float(100.0 * one_cnt / n_sources) if n_sources > 0 else 0.0,\n            "multi_candidate_pct": float(100.0 * multi_cnt / n_sources) if n_sources > 0 else 0.0,\n        }\n\n    def to_tensor_dict(self) -> Dict[str, Any]:\n        """\n        Exports graph data structures to NumPy arrays suitable for future conversion into PyTorch / PyG tensors.\n        """\n        node_ids = []\n        node_coords_phys = []\n        node_coords_vox = []\n        node_confidences = []\n        node_frames = []\n\n        node_id_to_idx = {}\n        for idx, node in enumerate(self.node_set.all_nodes()):\n            node_id_to_idx[node.node_id] = idx\n            node_ids.append(node.node_id)\n            node_coords_phys.append(node.physical_pos)\n            node_coords_vox.append(node.global_voxel_pos)\n            node_confidences.append(node.confidence)\n            node_frames.append(node.frame)\n\n        edge_index = []\n        edge_attr = []\n        for edge in self.edges:\n            src_idx = node_id_to_idx[edge.source_node_id]\n            dst_idx = node_id_to_idx[edge.destination_node_id]\n            edge_index.append([src_idx, dst_idx])\n            edge_attr.append([\n                edge.physical_displacement_vector[0],\n                edge.physical_displacement_vector[1],\n                edge.physical_displacement_vector[2],\n                edge.euclidean_distance_phys,\n                edge.source_confidence,\n                edge.destination_confidence,\n                edge.pair_confidence,\n            ])\n\n        return {\n            "node_ids": np.array(node_ids, dtype=np.int64),\n            "node_coords_phys": np.array(node_coords_phys, dtype=np.float32),\n            "node_coords_vox": np.array(node_coords_vox, dtype=np.float32),\n            "node_confidences": np.array(node_confidences, dtype=np.float32),\n            "node_frames": np.array(node_frames, dtype=np.int64),\n            "edge_index": np.array(edge_index, dtype=np.int64).T if edge_index else np.empty((2, 0), dtype=np.int64),\n            "edge_attr": np.array(edge_attr, dtype=np.float32) if edge_attr else np.empty((0, 7), dtype=np.float32),\n        }\n\n\nclass CandidateGraphBuilder:\n    """\n    Constructs a TemporalCandidateGraph connecting detections across consecutive frames.\n    """\n\n    def __init__(self, config: Optional[CandidateGraphConfig] = None):\n        self.config = config or CandidateGraphConfig()\n\n    def build_candidate_graph(self, node_set: NodeSet) -> TemporalCandidateGraph:\n        """\n        Builds candidate graph from given Canonical NodeSet.\n\n        :param node_set: NodeSet container with frame-indexed CanonicalNode instances\n        :return: TemporalCandidateGraph instance\n        """\n        edges: List[CandidateEdge] = []\n        frames = node_set.frames()\n\n        for t in frames:\n            source_nodes = node_set.get_nodes_by_frame(t)\n            if not source_nodes:\n                continue\n\n            # Target destination frames (t + 1 up to t + max_frame_gap)\n            dest_frames = [t + gap for gap in range(1, self.config.max_frame_gap + 1)]\n\n            for u in source_nodes:\n                candidate_dests: List[Tuple[CanonicalNode, float, int]] = []\n\n                for target_t in dest_frames:\n                    dest_nodes = node_set.get_nodes_by_frame(target_t)\n                    frame_gap = target_t - t\n\n                    for v in dest_nodes:\n                        dist_phys = u.distance_physical(v)\n\n                        # Effective distance limit\n                        max_dist = self.config.max_physical_distance_um\n                        if self.config.adaptive_velocity_factor is not None:\n                            max_dist *= self.config.adaptive_velocity_factor\n\n                        if dist_phys <= max_dist:\n                            candidate_dests.append((v, dist_phys, frame_gap))\n\n                # Sort candidate destinations by physical distance ascending\n                candidate_dests.sort(key=lambda item: item[1])\n\n                # Apply top-K per node limit if specified\n                retained_candidates = candidate_dests\n                if self.config.top_k_per_node is not None and self.config.top_k_per_node > 0:\n                    retained_candidates = candidate_dests[:self.config.top_k_per_node]\n\n                # Create CandidateEdge for each retained destination\n                for rank, (v, dist_phys, frame_gap) in enumerate(retained_candidates, start=1):\n                    phys_disp = u.displacement_physical(v)\n                    vox_disp = u.displacement_voxel(v)\n                    pair_conf = float(u.confidence * v.confidence)\n\n                    edge = CandidateEdge(\n                        source_node_id=u.node_id,\n                        destination_node_id=v.node_id,\n                        source_frame=u.frame,\n                        destination_frame=v.frame,\n                        frame_difference=frame_gap,\n                        physical_displacement_vector=phys_disp,\n                        euclidean_distance_phys=float(dist_phys),\n                        voxel_displacement_vector=vox_disp,\n                        source_confidence=float(u.confidence),\n                        destination_confidence=float(v.confidence),\n                        pair_confidence=pair_conf,\n                        source_region_id=u.primary_region_id,\n                        destination_region_id=v.primary_region_id,\n                        gating_reason="physical_distance_and_top_k",\n                        passed_distance_gate=True,\n                        rank_among_destinations=rank,\n                    )\n                    edges.append(edge)\n\n        return TemporalCandidateGraph(\n            node_set=node_set,\n            edges=edges,\n            config=self.config\n        )\n'
src_files['src/graph/diagnostics.py'] = '"""\nCandidate Graph Diagnostics and GEFF Ground-Truth Candidate Recall Evaluation.\n"""\n\nfrom dataclasses import dataclass, field\nfrom typing import List, Tuple, Dict, Any, Optional, Set\nimport numpy as np\n\nfrom src.graph.nodes import CanonicalNode, NodeSet\nfrom src.graph.candidate_graph import TemporalCandidateGraph, CandidateEdge\nfrom src.data.nodes import CellNode\nfrom src.data.coordinates import CoordinateTransformer\n\n\n@dataclass\nclass NodeMatchingResult:\n    """\n    Diagnostic results from matching canonical detection nodes to GEFF ground-truth nodes.\n    """\n    geff_node_count: int\n    canonical_node_count: int\n    per_frame_geff_counts: Dict[int, int]\n    per_frame_canonical_counts: Dict[int, int]\n    matched_geff_count: int\n    matching_success_rate: float  # (matched_geff_count / geff_node_count) * 100.0\n    unmatched_geff_count: int\n    geff_to_canonical_map: Dict[int, int]  # geff_node_id -> canonical_node_id\n    canonical_to_geff_map: Dict[int, int]  # canonical_node_id -> geff_node_id\n    match_distances_um: List[float]\n\n\n@dataclass\nclass CandidateRecallMetrics:\n    """\n    Diagnostic recall metrics evaluating candidate graph coverage against ground-truth edges.\n    """\n    num_gt_edges: int\n    num_evaluable_gt_edges: int  # GT edges where both endpoints matched canonical nodes\n    num_candidate_edges: int\n    candidate_graph_density: float  # Average candidates per source node\n    gt_edges_recovered: int\n    candidate_recall: float  # gt_edges_recovered / num_gt_edges\n    evaluable_candidate_recall: float  # gt_edges_recovered / num_evaluable_gt_edges\n    num_missing_gt_edges: int\n    missing_reasons: Dict[str, int]  # Reason counts for missing GT edges\n    gt_edge_distance_dist: Dict[str, float]  # min, median, max, mean physical distance\n    candidate_count_per_source: Dict[str, float]  # min, median, max, mean candidate count\n    zero_candidate_pct: float\n    one_candidate_pct: float\n    multi_candidate_pct: float\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Convert metrics to dictionary representation."""\n        return {\n            "num_gt_edges": self.num_gt_edges,\n            "num_evaluable_gt_edges": self.num_evaluable_gt_edges,\n            "num_candidate_edges": self.num_candidate_edges,\n            "candidate_graph_density": self.candidate_graph_density,\n            "gt_edges_recovered": self.gt_edges_recovered,\n            "candidate_recall": self.candidate_recall,\n            "evaluable_candidate_recall": self.evaluable_candidate_recall,\n            "num_missing_gt_edges": self.num_missing_gt_edges,\n            "missing_reasons": self.missing_reasons,\n            "gt_edge_distance_dist": self.gt_edge_distance_dist,\n            "candidate_count_per_source": self.candidate_count_per_source,\n            "zero_candidate_pct": self.zero_candidate_pct,\n            "one_candidate_pct": self.one_candidate_pct,\n            "multi_candidate_pct": self.multi_candidate_pct,\n        }\n\n\nclass CandidateRecallEvaluator:\n    """\n    Evaluator performing Part 11 Node Matching Check and Part 4 Candidate Recall Diagnostics.\n    """\n\n    def __init__(self, transformer: Optional[CoordinateTransformer] = None):\n        self.transformer = transformer or CoordinateTransformer()\n\n    def match_nodes_to_geff(\n        self,\n        node_set: NodeSet,\n        geff_nodes: List[CellNode],\n        match_threshold_um: float = 5.0\n    ) -> NodeMatchingResult:\n        """\n        Part 11 Data Integrity Check:\n        Matches CanonicalNode set to GEFF GT nodes per frame using greedy nearest physical distance.\n\n        :param node_set: NodeSet containing canonical detection nodes\n        :param geff_nodes: List of ground-truth CellNode objects from GEFF\n        :param match_threshold_um: Maximum physical distance for valid matching (default 5.0 um)\n        :return: NodeMatchingResult dataclass with matching metrics and mappings\n        """\n        geff_nodes_by_frame: Dict[int, List[CellNode]] = {}\n        for g_node in geff_nodes:\n            if g_node.frame not in geff_nodes_by_frame:\n                geff_nodes_by_frame[g_node.frame] = []\n            geff_nodes_by_frame[g_node.frame].append(g_node)\n\n        geff_to_canonical_map: Dict[int, int] = {}\n        canonical_to_geff_map: Dict[int, int] = {}\n        match_distances: List[float] = []\n\n        per_frame_geff_counts: Dict[int, int] = {}\n        per_frame_canonical_counts: Dict[int, int] = {}\n\n        all_frames = sorted(list(set(geff_nodes_by_frame.keys()) | set(node_set.frames())))\n\n        for f in all_frames:\n            f_geff = geff_nodes_by_frame.get(f, [])\n            f_can = node_set.get_nodes_by_frame(f)\n\n            per_frame_geff_counts[f] = len(f_geff)\n            per_frame_canonical_counts[f] = len(f_can)\n\n            if not f_geff or not f_can:\n                continue\n\n            # Compute physical pairwise distance matrix (n_geff, n_can)\n            n_g = len(f_geff)\n            n_c = len(f_can)\n            dist_mat = np.zeros((n_g, n_c), dtype=np.float32)\n\n            for i, g in enumerate(f_geff):\n                for j, c in enumerate(f_can):\n                    d = self.transformer.euclidean_distance_physical(g.physical_pos, c.physical_pos)\n                    dist_mat[i, j] = d\n\n            # Greedy distance matching within match_threshold_um\n            matched_g_indices: Set[int] = set()\n            matched_c_indices: Set[int] = set()\n\n            # Flatten and sort candidate pairs by physical distance\n            flat_pairs = []\n            for i in range(n_g):\n                for j in range(n_c):\n                    if dist_mat[i, j] <= match_threshold_um:\n                        flat_pairs.append((dist_mat[i, j], i, j))\n            flat_pairs.sort(key=lambda item: item[0])\n\n            for dist, i, j in flat_pairs:\n                if i not in matched_g_indices and j not in matched_c_indices:\n                    matched_g_indices.add(i)\n                    matched_c_indices.add(j)\n\n                    g_id = f_geff[i].node_id\n                    c_id = f_can[j].node_id\n\n                    geff_to_canonical_map[g_id] = c_id\n                    canonical_to_geff_map[c_id] = g_id\n                    match_distances.append(float(dist))\n\n        matched_geff_count = len(geff_to_canonical_map)\n        geff_node_count = len(geff_nodes)\n        canonical_node_count = node_set.num_nodes()\n        unmatched_geff_count = geff_node_count - matched_geff_count\n        success_rate = float(100.0 * matched_geff_count / geff_node_count) if geff_node_count > 0 else 0.0\n\n        return NodeMatchingResult(\n            geff_node_count=geff_node_count,\n            canonical_node_count=canonical_node_count,\n            per_frame_geff_counts=per_frame_geff_counts,\n            per_frame_canonical_counts=per_frame_canonical_counts,\n            matched_geff_count=matched_geff_count,\n            matching_success_rate=success_rate,\n            unmatched_geff_count=unmatched_geff_count,\n            geff_to_canonical_map=geff_to_canonical_map,\n            canonical_to_geff_map=canonical_to_geff_map,\n            match_distances_um=match_distances\n        )\n\n    def evaluate_recall(\n        self,\n        graph: TemporalCandidateGraph,\n        geff_edges: List[Tuple[int, int]],\n        geff_nodes: List[CellNode],\n        matching_result: NodeMatchingResult\n    ) -> CandidateRecallMetrics:\n        """\n        Part 4 Candidate Recall Diagnostic:\n        Evaluates ground-truth edge recovery by candidate graph and diagnoses missing edges.\n\n        :param graph: TemporalCandidateGraph instance\n        :param geff_edges: List of directed (source_geff_id, dest_geff_id) GT edges\n        :param geff_nodes: List of ground-truth CellNode objects\n        :param matching_result: Pre-computed NodeMatchingResult\n        :return: CandidateRecallMetrics dataclass\n        """\n        geff_node_map = {g.node_id: g for g in geff_nodes}\n        gt_map = matching_result.geff_to_canonical_map\n\n        num_gt = len(geff_edges)\n        num_evaluable = 0\n        recovered = 0\n\n        missing_reasons = {\n            "source_unmatched": 0,\n            "destination_unmatched": 0,\n            "distance_gate_rejected": 0,\n            "top_k_truncated": 0,\n            "other": 0,\n        }\n\n        gt_distances: List[float] = []\n\n        for u_gt, v_gt in geff_edges:\n            u_node = geff_node_map.get(u_gt)\n            v_node = geff_node_map.get(v_gt)\n\n            if u_node is not None and v_node is not None:\n                d_phys = u_node.distance_physical(v_node)\n                gt_distances.append(d_phys)\n\n            u_can = gt_map.get(u_gt)\n            v_can = gt_map.get(v_gt)\n\n            if u_can is None or v_can is None:\n                if u_can is None and v_can is None:\n                    missing_reasons["source_unmatched"] += 1\n                elif u_can is None:\n                    missing_reasons["source_unmatched"] += 1\n                else:\n                    missing_reasons["destination_unmatched"] += 1\n                continue\n\n            num_evaluable += 1\n\n            # Check if edge exists in candidate graph\n            candidate_edge = graph.get_edge(u_can, v_can)\n            if candidate_edge is not None:\n                recovered += 1\n            else:\n                # Diagnose missing edge reason\n                can_u_node = graph.node_set.get_node(u_can)\n                can_v_node = graph.node_set.get_node(v_can)\n\n                if can_u_node is not None and can_v_node is not None:\n                    d_phys_can = can_u_node.distance_physical(can_v_node)\n                    max_dist = graph.config.max_physical_distance_um\n\n                    if d_phys_can > max_dist:\n                        missing_reasons["distance_gate_rejected"] += 1\n                    else:\n                        # Edge passed distance gate but was truncated by top-K\n                        missing_reasons["top_k_truncated"] += 1\n                else:\n                    missing_reasons["other"] += 1\n\n        num_missing = num_gt - recovered\n        recall = float(recovered / num_gt) if num_gt > 0 else 0.0\n        evaluable_recall = float(recovered / num_evaluable) if num_evaluable > 0 else 0.0\n\n        # Diagnostics stats for GT distances\n        if gt_distances:\n            dist_arr = np.array(gt_distances, dtype=np.float32)\n            gt_dist_stats = {\n                "min": float(np.min(dist_arr)),\n                "median": float(np.median(dist_arr)),\n                "max": float(np.max(dist_arr)),\n                "mean": float(np.mean(dist_arr)),\n            }\n        else:\n            gt_dist_stats = {"min": 0.0, "median": 0.0, "max": 0.0, "mean": 0.0}\n\n        # Candidate counts per source node\n        diag = graph.export_diagnostics()\n        cand_stats = {\n            "min": float(diag.get("min_candidates_per_node", 0)),\n            "median": float(diag.get("median_candidates_per_node", 0)),\n            "max": float(diag.get("max_candidates_per_node", 0)),\n            "mean": float(diag.get("mean_candidates_per_node", 0)),\n        }\n\n        return CandidateRecallMetrics(\n            num_gt_edges=num_gt,\n            num_evaluable_gt_edges=num_evaluable,\n            num_candidate_edges=graph.candidate_count(),\n            candidate_graph_density=float(diag.get("mean_candidates_per_node", 0.0)),\n            gt_edges_recovered=recovered,\n            candidate_recall=recall,\n            evaluable_candidate_recall=evaluable_recall,\n            num_missing_gt_edges=num_missing,\n            missing_reasons=missing_reasons,\n            gt_edge_distance_dist=gt_dist_stats,\n            candidate_count_per_source=cand_stats,\n            zero_candidate_pct=float(diag.get("zero_candidate_pct", 0.0)),\n            one_candidate_pct=float(diag.get("one_candidate_pct", 0.0)),\n            multi_candidate_pct=float(diag.get("multi_candidate_pct", 0.0)),\n        )\n'
src_files['src/graph/features.py'] = '"""\nDeterministic Node and Edge Feature Extraction for Graph Models.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Dict, Any, Optional\nimport numpy as np\n\nfrom src.graph.nodes import CanonicalNode\nfrom src.graph.candidate_graph import TemporalCandidateGraph, CandidateEdge\nfrom src.data.coordinates import CoordinateTransformer\n\n\n@dataclass\nclass NodeFeatures:\n    """\n    Container for extracted node feature matrices.\n    """\n    node_ids: np.ndarray  # (N,) int64\n    features: np.ndarray  # (N, D_node) float32\n    feature_names: List[str]  # List of feature names\n    frames: np.ndarray  # (N,) int64\n\n\n@dataclass\nclass EdgeFeatures:\n    """\n    Container for extracted edge feature matrices.\n    """\n    edge_index: np.ndarray  # (2, E) int64 (source_idx, dest_idx)\n    edge_ids: List[Tuple[int, int]]  # List of (source_node_id, dest_node_id)\n    features: np.ndarray  # (E, D_edge) float32\n    feature_names: List[str]  # List of edge feature names\n\n\nclass GraphFeatureExtractor:\n    """\n    Deterministic feature extractor for node and edge representations in candidate graphs.\n    """\n\n    def __init__(\n        self,\n        volume_shape: Tuple[int, int, int, int] = (100, 64, 256, 256),\n        transformer: Optional[CoordinateTransformer] = None,\n        density_radius_um: float = 15.0\n    ):\n        self.volume_shape = volume_shape  # (T, Z, Y, X)\n        self.transformer = transformer or CoordinateTransformer()\n        self.density_radius_um = density_radius_um\n\n        # Calculate max physical bounds\n        self.max_t = max(1, volume_shape[0] - 1)\n        self.max_z_vox = volume_shape[1]\n        self.max_y_vox = volume_shape[2]\n        self.max_x_vox = volume_shape[3]\n\n        self.max_z_phys = self.max_z_vox * self.transformer.resolution.z_spacing\n        self.max_y_phys = self.max_y_vox * self.transformer.resolution.y_spacing\n        self.max_x_phys = self.max_x_vox * self.transformer.resolution.x_spacing\n\n    def extract_node_features(self, graph: TemporalCandidateGraph) -> NodeFeatures:\n        """\n        Extracts normalized node feature matrix for all nodes in the candidate graph.\n        """\n        all_nodes = graph.node_set.all_nodes()\n        n_nodes = len(all_nodes)\n        if n_nodes == 0:\n            return NodeFeatures(\n                node_ids=np.empty(0, dtype=np.int64),\n                features=np.empty((0, 8), dtype=np.float32),\n                feature_names=[\n                    "norm_z_phys", "norm_y_phys", "norm_x_phys",\n                    "norm_z_vox", "norm_y_vox", "norm_x_vox",\n                    "confidence", "norm_frame", "local_density_count"\n                ],\n                frames=np.empty(0, dtype=np.int64)\n            )\n\n        # Precompute per-node local density counts within same frame\n        node_density_map: Dict[int, float] = {}\n        for node in all_nodes:\n            nearby = graph.node_set.search_nearby_physical(\n                node.physical_pos,\n                node.frame,\n                self.density_radius_um,\n                transformer=self.transformer\n            )\n            # count excludes self\n            node_density_map[node.node_id] = float(max(0, len(nearby) - 1))\n\n        node_ids = []\n        frames = []\n        feature_rows = []\n\n        feature_names = [\n            "norm_z_phys", "norm_y_phys", "norm_x_phys",\n            "norm_z_vox", "norm_y_vox", "norm_x_vox",\n            "confidence", "norm_frame", "local_density_count"\n        ]\n\n        for node in all_nodes:\n            node_ids.append(node.node_id)\n            frames.append(node.frame)\n\n            zp_norm = float(node.physical_pos[0] / self.max_z_phys) if self.max_z_phys > 0 else 0.0\n            yp_norm = float(node.physical_pos[1] / self.max_y_phys) if self.max_y_phys > 0 else 0.0\n            xp_norm = float(node.physical_pos[2] / self.max_x_phys) if self.max_x_phys > 0 else 0.0\n\n            zv_norm = float(node.global_voxel_pos[0] / self.max_z_vox) if self.max_z_vox > 0 else 0.0\n            yv_norm = float(node.global_voxel_pos[1] / self.max_y_vox) if self.max_y_vox > 0 else 0.0\n            xv_norm = float(node.global_voxel_pos[2] / self.max_x_vox) if self.max_x_vox > 0 else 0.0\n\n            conf = float(node.confidence)\n            t_norm = float(node.frame / self.max_t) if self.max_t > 0 else 0.0\n            density = node_density_map[node.node_id]\n\n            row = [\n                zp_norm, yp_norm, xp_norm,\n                zv_norm, yv_norm, xv_norm,\n                conf, t_norm, density\n            ]\n            feature_rows.append(row)\n\n        return NodeFeatures(\n            node_ids=np.array(node_ids, dtype=np.int64),\n            features=np.array(feature_rows, dtype=np.float32),\n            feature_names=feature_names,\n            frames=np.array(frames, dtype=np.int64)\n        )\n\n    def extract_edge_features(self, graph: TemporalCandidateGraph) -> EdgeFeatures:\n        """\n        Extracts edge features and edge_index for all candidate edges in graph.\n        """\n        all_nodes = graph.node_set.all_nodes()\n        node_id_to_idx = {n.node_id: idx for idx, n in enumerate(all_nodes)}\n\n        # Precompute node density for density difference feature\n        node_density_map: Dict[int, float] = {}\n        for node in all_nodes:\n            nearby = graph.node_set.search_nearby_physical(\n                node.physical_pos,\n                node.frame,\n                self.density_radius_um,\n                transformer=self.transformer\n            )\n            node_density_map[node.node_id] = float(max(0, len(nearby) - 1))\n\n        feature_names = [\n            "dz_phys", "dy_phys", "dx_phys",\n            "abs_dz_phys", "abs_dy_phys", "abs_dx_phys",\n            "euclidean_dist_phys",\n            "dz_vox", "dy_vox", "dx_vox",\n            "source_conf", "dest_conf", "pair_conf",\n            "density_diff", "frame_gap", "rank_among_dests"\n        ]\n\n        edges = graph.edges\n        n_edges = len(edges)\n        if n_edges == 0:\n            return EdgeFeatures(\n                edge_index=np.empty((2, 0), dtype=np.int64),\n                edge_ids=[],\n                features=np.empty((0, len(feature_names)), dtype=np.float32),\n                feature_names=feature_names\n            )\n\n        edge_index_src = []\n        edge_index_dst = []\n        edge_ids = []\n        feature_rows = []\n\n        for edge in edges:\n            src_idx = node_id_to_idx[edge.source_node_id]\n            dst_idx = node_id_to_idx[edge.destination_node_id]\n\n            edge_index_src.append(src_idx)\n            edge_index_dst.append(dst_idx)\n            edge_ids.append((edge.source_node_id, edge.destination_node_id))\n\n            dz_p, dy_p, dx_p = edge.physical_displacement_vector\n            abs_dz_p, abs_dy_p, abs_dx_p = abs(dz_p), abs(dy_p), abs(dx_p)\n            dist_p = edge.euclidean_distance_phys\n\n            dz_v, dy_v, dx_v = edge.voxel_displacement_vector\n\n            src_c = edge.source_confidence\n            dst_c = edge.destination_confidence\n            pair_c = edge.pair_confidence\n\n            src_dens = node_density_map.get(edge.source_node_id, 0.0)\n            dst_dens = node_density_map.get(edge.destination_node_id, 0.0)\n            dens_diff = dst_dens - src_dens\n\n            gap = float(edge.frame_difference)\n            rank = float(edge.rank_among_destinations)\n\n            row = [\n                dz_p, dy_p, dx_p,\n                abs_dz_p, abs_dy_p, abs_dx_p,\n                dist_p,\n                dz_v, dy_v, dx_v,\n                src_c, dst_c, pair_c,\n                dens_diff, gap, rank\n            ]\n            feature_rows.append(row)\n\n        edge_index = np.array([edge_index_src, edge_index_dst], dtype=np.int64)\n\n        return EdgeFeatures(\n            edge_index=edge_index,\n            edge_ids=edge_ids,\n            features=np.array(feature_rows, dtype=np.float32),\n            feature_names=feature_names\n        )\n\n    def extract(self, graph: TemporalCandidateGraph) -> Dict[str, Any]:\n        """Extract both node and edge feature containers."""\n        node_feats = self.extract_node_features(graph)\n        edge_feats = self.extract_edge_features(graph)\n        return {\n            "node_features": node_feats,\n            "edge_features": edge_feats\n        }\n'
src_files['src/graph/nodes.py'] = '"""\nCanonical Node Representation and Deterministic Frame-Local Node Reconciliation.\n"""\n\nfrom dataclasses import dataclass, field\nfrom typing import List, Tuple, Dict, Any, Optional, Union, Set\nimport numpy as np\n\nfrom src.data.coordinates import CoordinateTransformer\nfrom src.data.nodes import CellNode\nfrom src.regional.geometry import RegionalCandidate\n\n\n@dataclass\nclass CanonicalNode:\n    """\n    Canonical global graph node representation for a cell detection at a specific frame.\n\n    Contains stable node ID, spatiotemporal coordinates (voxel & physical),\n    detector confidence, regional provenance, and diagnostic metadata.\n    """\n    node_id: int\n    frame: int  # Time index t (0-indexed)\n    global_voxel_pos: Tuple[float, float, float]  # (z_voxel, y_voxel, x_voxel)\n    physical_pos: Tuple[float, float, float]  # (z_phys, y_phys, x_phys) in micrometers\n    confidence: float\n    detector_features: Optional[Any] = None\n    contributing_regional_ids: List[int] = field(default_factory=list)\n    num_regional_observations: int = 1\n    primary_region_id: int = 0\n    diagnostic_metadata: Dict[str, Any] = field(default_factory=dict)\n\n    def distance_physical(self, other: "CanonicalNode") -> float:\n        """Compute Euclidean distance to another node in physical space (micrometers)."""\n        dz = self.physical_pos[0] - other.physical_pos[0]\n        dy = self.physical_pos[1] - other.physical_pos[1]\n        dx = self.physical_pos[2] - other.physical_pos[2]\n        return float(np.sqrt(dz * dz + dy * dy + dx * dx))\n\n    def displacement_physical(self, other: "CanonicalNode") -> Tuple[float, float, float]:\n        """Compute displacement vector from self to other in physical space (micrometers)."""\n        dz = float(other.physical_pos[0] - self.physical_pos[0])\n        dy = float(other.physical_pos[1] - self.physical_pos[1])\n        dx = float(other.physical_pos[2] - self.physical_pos[2])\n        return (dz, dy, dx)\n\n    def displacement_voxel(self, other: "CanonicalNode") -> Tuple[float, float, float]:\n        """Compute displacement vector from self to other in voxel space."""\n        dz = float(other.global_voxel_pos[0] - self.global_voxel_pos[0])\n        dy = float(other.global_voxel_pos[1] - self.global_voxel_pos[1])\n        dx = float(other.global_voxel_pos[2] - self.global_voxel_pos[2])\n        return (dz, dy, dx)\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Returns node attributes as dictionary."""\n        return {\n            "node_id": self.node_id,\n            "frame": self.frame,\n            "z_voxel": self.global_voxel_pos[0],\n            "y_voxel": self.global_voxel_pos[1],\n            "x_voxel": self.global_voxel_pos[2],\n            "z_phys": self.physical_pos[0],\n            "y_phys": self.physical_pos[1],\n            "x_phys": self.physical_pos[2],\n            "confidence": self.confidence,\n            "contributing_regional_ids": list(self.contributing_regional_ids),\n            "num_regional_observations": self.num_regional_observations,\n            "primary_region_id": self.primary_region_id,\n            "diagnostic_metadata": self.diagnostic_metadata,\n        }\n\n    @classmethod\n    def from_regional_candidate(\n        cls,\n        candidate: RegionalCandidate,\n        canonical_id: int\n    ) -> "CanonicalNode":\n        """Factory method creating CanonicalNode from RegionalCandidate."""\n        prov = candidate.provenance or {}\n        contrib_ids = prov.get("contributing_region_ids", [candidate.region_id])\n        num_obs = prov.get("num_contributing_regions", 1)\n        primary_reg = prov.get("primary_region_id", candidate.region_id)\n\n        v_pos = (\n            float(candidate.global_voxel_pos[0]),\n            float(candidate.global_voxel_pos[1]),\n            float(candidate.global_voxel_pos[2]),\n        )\n        p_pos = (\n            float(candidate.physical_pos[0]),\n            float(candidate.physical_pos[1]),\n            float(candidate.physical_pos[2]),\n        )\n\n        return cls(\n            node_id=canonical_id,\n            frame=candidate.frame,\n            global_voxel_pos=v_pos,\n            physical_pos=p_pos,\n            confidence=float(candidate.confidence),\n            contributing_regional_ids=sorted(list(set(contrib_ids))),\n            num_regional_observations=num_obs,\n            primary_region_id=primary_reg,\n            diagnostic_metadata={\n                "is_in_core": candidate.is_in_core,\n                "original_candidate_id": candidate.candidate_id,\n                "provenance": prov,\n            }\n        )\n\n    @classmethod\n    def from_cell_node(\n        cls,\n        cell_node: CellNode,\n        canonical_id: Optional[int] = None\n    ) -> "CanonicalNode":\n        """Factory method creating CanonicalNode from foundational CellNode (e.g., GT node)."""\n        nid = canonical_id if canonical_id is not None else cell_node.node_id\n        v_pos = (\n            float(cell_node.voxel_pos[0]),\n            float(cell_node.voxel_pos[1]),\n            float(cell_node.voxel_pos[2]),\n        )\n        p_pos = (\n            float(cell_node.physical_pos[0]),\n            float(cell_node.physical_pos[1]),\n            float(cell_node.physical_pos[2]),\n        )\n        return cls(\n            node_id=nid,\n            frame=cell_node.frame,\n            global_voxel_pos=v_pos,\n            physical_pos=p_pos,\n            confidence=float(cell_node.attributes.get("confidence", 1.0)),\n            contributing_regional_ids=[0],\n            num_regional_observations=1,\n            primary_region_id=0,\n            diagnostic_metadata=dict(cell_node.attributes)\n        )\n\n\nclass NodeReconciler:\n    """\n    Frame-local deterministic node reconciliation across regional predictions.\n\n    Groups detections within physical distance threshold, computes canonical positions,\n    preserves complete regional provenance, and generates stable deterministic node IDs.\n    """\n\n    def __init__(\n        self,\n        reconciliation_distance_um: float = 4.0,\n        transformer: Optional[CoordinateTransformer] = None\n    ):\n        self.reconciliation_distance_um = reconciliation_distance_um\n        self.transformer = transformer or CoordinateTransformer()\n        self.regional_to_canonical_map: Dict[Tuple[int, int], int] = {}  # (frame, candidate_id) -> canonical_id\n\n    def reconcile_frame(\n        self,\n        frame: int,\n        candidates: List[Union[RegionalCandidate, CanonicalNode, CellNode]]\n    ) -> List[CanonicalNode]:\n        """\n        Reconciles candidates within a single frame into canonical global nodes.\n\n        :param frame: Frame index t\n        :param candidates: List of regional candidates or nodes active at frame t\n        :return: List of canonical nodes for frame t\n        """\n        if not candidates:\n            return []\n\n        # Convert inputs to standard format if needed\n        parsed_cands = []\n        for idx, cand in enumerate(candidates):\n            if isinstance(cand, CanonicalNode):\n                parsed_cands.append(cand)\n            elif isinstance(cand, RegionalCandidate):\n                parsed_cands.append(cand)\n            elif isinstance(cand, CellNode):\n                parsed_cands.append(CanonicalNode.from_cell_node(cand, canonical_id=cand.node_id))\n            else:\n                raise TypeError(f"Unsupported candidate type: {type(cand)}")\n\n        n_cand = len(parsed_cands)\n\n        # Compute pairwise physical distance matrix\n        phys_positions = [c.physical_pos for c in parsed_cands]\n        dist_matrix = np.zeros((n_cand, n_cand), dtype=np.float32)\n        for i in range(n_cand):\n            for j in range(i + 1, n_cand):\n                d = self.transformer.euclidean_distance_physical(phys_positions[i], phys_positions[j])\n                dist_matrix[i, j] = d\n                dist_matrix[j, i] = d\n\n        # Connected component clustering\n        visited = np.zeros(n_cand, dtype=bool)\n        canonical_nodes: List[CanonicalNode] = []\n        local_idx = 1\n\n        for i in range(n_cand):\n            if visited[i]:\n                continue\n\n            cluster_indices = [i]\n            visited[i] = True\n            queue = [i]\n\n            while queue:\n                curr = queue.pop(0)\n                for neighbor in range(n_cand):\n                    if not visited[neighbor] and dist_matrix[curr, neighbor] <= self.reconciliation_distance_um:\n                        visited[neighbor] = True\n                        cluster_indices.append(neighbor)\n                        queue.append(neighbor)\n\n            cluster_cands = [parsed_cands[idx] for idx in cluster_indices]\n\n            # Generate deterministic canonical node ID: (frame + 1) * 100000 + local_idx\n            canonical_id = (frame + 1) * 100000 + local_idx\n            local_idx += 1\n\n            # Primary candidate selected by max confidence\n            primary = max(cluster_cands, key=lambda c: c.confidence)\n\n            # Aggregate regional provenance\n            all_contrib_regions: Set[int] = set()\n            all_regional_candidate_ids = []\n            for c in cluster_cands:\n                if isinstance(c, RegionalCandidate):\n                    prov = c.provenance or {}\n                    contribs = prov.get("contributing_region_ids", [c.region_id])\n                    all_contrib_regions.update(contribs)\n                    all_regional_candidate_ids.append(c.candidate_id)\n                    self.regional_to_canonical_map[(frame, c.candidate_id)] = canonical_id\n                elif isinstance(c, CanonicalNode):\n                    all_contrib_regions.update(c.contributing_regional_ids)\n                    if "original_candidate_id" in c.diagnostic_metadata:\n                        cid = c.diagnostic_metadata["original_candidate_id"]\n                        self.regional_to_canonical_map[(frame, cid)] = canonical_id\n\n            if not all_contrib_regions:\n                all_contrib_regions = {primary.primary_region_id if isinstance(primary, CanonicalNode) else primary.region_id}\n\n            # Position computation: weighted position average by confidence\n            conf_weights = np.array([c.confidence for c in cluster_cands], dtype=np.float64)\n            weight_sum = conf_weights.sum()\n            if weight_sum > 0:\n                conf_weights /= weight_sum\n            else:\n                conf_weights = np.ones(len(cluster_cands)) / len(cluster_cands)\n\n            v_positions = np.array([c.global_voxel_pos if isinstance(c, CanonicalNode) else c.global_voxel_pos for c in cluster_cands], dtype=np.float64)\n            p_positions = np.array([c.physical_pos for c in cluster_cands], dtype=np.float64)\n\n            avg_v_pos = tuple(np.sum(v_positions * conf_weights[:, None], axis=0))\n            avg_p_pos = tuple(np.sum(p_positions * conf_weights[:, None], axis=0))\n\n            max_confidence = float(np.max([c.confidence for c in cluster_cands]))\n\n            canonical_node = CanonicalNode(\n                node_id=canonical_id,\n                frame=frame,\n                global_voxel_pos=(float(avg_v_pos[0]), float(avg_v_pos[1]), float(avg_v_pos[2])),\n                physical_pos=(float(avg_p_pos[0]), float(avg_p_pos[1]), float(avg_p_pos[2])),\n                confidence=max_confidence,\n                contributing_regional_ids=sorted(list(all_contrib_regions)),\n                num_regional_observations=len(all_contrib_regions),\n                primary_region_id=primary.primary_region_id if isinstance(primary, CanonicalNode) else primary.region_id,\n                diagnostic_metadata={\n                    "reconciled_cluster_size": len(cluster_cands),\n                    "original_candidate_ids": all_regional_candidate_ids,\n                    "reconciliation_distance_um": self.reconciliation_distance_um,\n                }\n            )\n\n            canonical_nodes.append(canonical_node)\n\n        return canonical_nodes\n\n\nclass NodeSet:\n    """\n    Container for canonical nodes indexed by frame and node ID.\n    """\n\n    def __init__(self, nodes: Optional[List[CanonicalNode]] = None):\n        self._nodes_by_frame: Dict[int, List[CanonicalNode]] = {}\n        self._nodes_by_id: Dict[int, CanonicalNode] = {}\n\n        if nodes:\n            self.add_nodes(nodes)\n\n    def add_node(self, node: CanonicalNode) -> None:\n        """Add a single canonical node."""\n        if node.frame not in self._nodes_by_frame:\n            self._nodes_by_frame[node.frame] = []\n        self._nodes_by_frame[node.frame].append(node)\n        self._nodes_by_id[node.node_id] = node\n\n    def add_nodes(self, nodes: List[CanonicalNode]) -> None:\n        """Add multiple canonical nodes."""\n        for n in nodes:\n            self.add_node(n)\n\n    def get_nodes_by_frame(self, frame: int) -> List[CanonicalNode]:\n        """Retrieve all nodes active at specified frame."""\n        return self._nodes_by_frame.get(frame, [])\n\n    def get_node(self, node_id: int) -> Optional[CanonicalNode]:\n        """Retrieve node by node ID."""\n        return self._nodes_by_id.get(node_id)\n\n    def all_nodes(self) -> List[CanonicalNode]:\n        """Retrieve flat list of all nodes."""\n        return list(self._nodes_by_id.values())\n\n    def num_nodes(self) -> int:\n        """Total number of nodes."""\n        return len(self._nodes_by_id)\n\n    def frames(self) -> List[int]:\n        """List of active frame indices in sorted order."""\n        return sorted(list(self._nodes_by_frame.keys()))\n\n    def num_frames(self) -> int:\n        """Number of frames."""\n        return len(self._nodes_by_frame)\n\n    def search_nearby_physical(\n        self,\n        physical_pos: Tuple[float, float, float],\n        frame: int,\n        max_dist_um: float,\n        transformer: Optional[CoordinateTransformer] = None\n    ) -> List[Tuple[CanonicalNode, float]]:\n        """\n        Searches for nodes in a given frame within physical distance threshold.\n\n        :return: List of (CanonicalNode, physical_distance) tuples sorted by distance\n        """\n        trans = transformer or CoordinateTransformer()\n        frame_nodes = self.get_nodes_by_frame(frame)\n        results = []\n\n        for node in frame_nodes:\n            d = trans.euclidean_distance_physical(physical_pos, node.physical_pos)\n            if d <= max_dist_um:\n                results.append((node, d))\n\n        results.sort(key=lambda item: item[1])\n        return results\n'
src_files['src/optimization/__init__.py'] = '"""\nGlobal Integer Linear Programming (ILP) Optimization Package.\n"""\n\nfrom .base import OptimizationConfig, SolverResult, TrackFragment, TrackingSolution\nfrom .objective import ObjectiveBuilder\nfrom .constraints import ConstraintBuilder, ConstraintMatrix\nfrom .solver import MILPSolverWrapper\nfrom .diagnostics import TrackFragmentDecoder, ILPDiagnostics, ILPDiagnosticReport\nfrom .ilp import GlobalILPOptimizer\n\n__all__ = [\n    "OptimizationConfig",\n    "SolverResult",\n    "TrackFragment",\n    "TrackingSolution",\n    "ObjectiveBuilder",\n    "ConstraintBuilder",\n    "ConstraintMatrix",\n    "MILPSolverWrapper",\n    "TrackFragmentDecoder",\n    "ILPDiagnostics",\n    "ILPDiagnosticReport",\n    "GlobalILPOptimizer",\n]\n'
src_files['src/optimization/base.py'] = '"""\nData structures, configurations, and result containers for Global ILP Optimization.\n"""\n\nfrom dataclasses import dataclass, field\nfrom typing import List, Tuple, Dict, Any, Optional\nimport numpy as np\n\nfrom src.graph.candidate_graph import CandidateEdge\n\n\n@dataclass\nclass OptimizationConfig:\n    """\n    Configuration hyperparameters for ILP objective construction and solver execution.\n\n    :param mode: Objective mode - \'stgnn_only\', \'geometry_only\', or \'combined\'\n    :param alpha: Weight for STGNN learned log-odds score in combined mode\n    :param beta: Weight for geometric prior log-odds score in combined mode\n    :param birth_cost: Penalty/cost for starting a new track (0.0 = no birth penalty)\n    :param termination_cost: Penalty/cost for terminating a track (0.0 = no termination penalty)\n    :param epsilon: Epsilon buffer for log-odds probability transformation\n    :param sigma_scale_um: Distance scaling factor (um) for geometric prior score\n    :param timeout_sec: Maximum solver timeout in seconds\n    """\n    mode: str = "stgnn_only"  # \'stgnn_only\', \'geometry_only\', or \'combined\'\n    alpha: float = 1.0\n    beta: float = 1.0\n    birth_cost: float = 0.0\n    termination_cost: float = 0.0\n    epsilon: float = 1e-5\n    sigma_scale_um: float = 5.0\n    timeout_sec: float = 60.0\n\n\n@dataclass\nclass SolverResult:\n    """\n    Container for solver execution status and statistics.\n    """\n    status: str\n    success: bool\n    objective_value: float\n    solve_time_sec: float\n    num_variables: int\n    num_constraints: int\n    selected_edge_count: int\n    termination_reason: str\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Convert result to dictionary."""\n        return {\n            "status": self.status,\n            "success": self.success,\n            "objective_value": self.objective_value,\n            "solve_time_sec": self.solve_time_sec,\n            "num_variables": self.num_variables,\n            "num_constraints": self.num_constraints,\n            "selected_edge_count": self.selected_edge_count,\n            "termination_reason": self.termination_reason,\n        }\n\n\n@dataclass\nclass TrackFragment:\n    """\n    Representation of a decoded direct-continuation track fragment across consecutive frames.\n    """\n    fragment_id: int\n    node_ids: List[int]\n    frames: List[int]\n    physical_positions: List[Tuple[float, float, float]]\n    length_frames: int\n    total_physical_distance_um: float\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Convert fragment to dictionary representation."""\n        return {\n            "fragment_id": self.fragment_id,\n            "node_ids": self.node_ids,\n            "frames": self.frames,\n            "length_frames": self.length_frames,\n            "total_physical_distance_um": self.total_physical_distance_um,\n        }\n\n\n@dataclass\nclass TrackingSolution:\n    """\n    Global ILP tracking solution containing selected/rejected edges and decoded track fragments.\n    """\n    selected_edges: List[CandidateEdge]\n    rejected_edges: List[CandidateEdge]\n    fragments: List[TrackFragment]\n    solver_result: SolverResult\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Convert solution to dictionary."""\n        return {\n            "solver_result": self.solver_result.to_dict(),\n            "selected_edge_count": len(self.selected_edges),\n            "rejected_edge_count": len(self.rejected_edges),\n            "num_fragments": len(self.fragments),\n            "fragments": [f.to_dict() for f in self.fragments],\n        }\n'
src_files['src/optimization/constraints.py'] = '"""\nLinear Flow Constraint Construction for One-to-One Temporal Continuation.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Dict, Any, Optional\nimport numpy as np\nfrom scipy.sparse import csc_matrix\nfrom scipy.optimize import LinearConstraint\n\nfrom src.graph.nodes import CanonicalNode, NodeSet\nfrom src.graph.candidate_graph import CandidateEdge\n\n\n@dataclass\nclass ConstraintMatrix:\n    """\n    Container for sparse linear constraint matrix A and upper/lower bounds.\n    """\n    matrix: csc_matrix  # (M, E) sparse matrix\n    lb: np.ndarray  # (M,) float64 lower bounds\n    ub: np.ndarray  # (M,) float64 upper bounds\n    constraint_names: List[str]\n    num_incoming_constraints: int\n    num_outgoing_constraints: int\n\n    def to_scipy_constraint(self) -> LinearConstraint:\n        """Converts matrix representation to scipy.optimize.LinearConstraint."""\n        return LinearConstraint(self.matrix, self.lb, self.ub)\n\n\nclass ConstraintBuilder:\n    """\n    Constructs one-to-one temporal continuation flow constraints:\n        1. Incoming Degree Constraint: sum_{e in in(v)} x_e <= 1 for all v in V\n        2. Outgoing Degree Constraint: sum_{e in out(u)} x_e <= 1 for all u in V\n    """\n\n    def build_flow_constraints(\n        self,\n        node_set: NodeSet,\n        edges: List[CandidateEdge]\n    ) -> ConstraintMatrix:\n        """\n        Constructs linear flow constraint matrix for candidate edge variables.\n\n        :param node_set: NodeSet containing canonical nodes\n        :param edges: List of CandidateEdge objects\n        :return: ConstraintMatrix object\n        """\n        E = len(edges)\n        if E == 0:\n            empty_sparse = csc_matrix((0, 0), dtype=np.float64)\n            return ConstraintMatrix(\n                matrix=empty_sparse,\n                lb=np.empty(0, dtype=np.float64),\n                ub=np.empty(0, dtype=np.float64),\n                constraint_names=[],\n                num_incoming_constraints=0,\n                num_outgoing_constraints=0\n            )\n\n        # Map candidate edges for fast lookup\n        # Outgoing edges per node ID\n        node_out_edges: Dict[int, List[int]] = {}\n        # Incoming edges per node ID\n        node_in_edges: Dict[int, List[int]] = {}\n\n        for edge_idx, edge in enumerate(edges):\n            src = edge.source_node_id\n            dst = edge.destination_node_id\n\n            if src not in node_out_edges:\n                node_out_edges[src] = []\n            node_out_edges[src].append(edge_idx)\n\n            if dst not in node_in_edges:\n                node_in_edges[dst] = []\n            node_in_edges[dst].append(edge_idx)\n\n        row_indices = []\n        col_indices = []\n        data_values = []\n        constraint_names = []\n\n        curr_row = 0\n        n_inc = 0\n        n_out = 0\n\n        # 1. Outgoing degree constraints (sum_{e in out(u)} x_e <= 1)\n        for node_id, edge_indices in sorted(node_out_edges.items()):\n            for e_idx in edge_indices:\n                row_indices.append(curr_row)\n                col_indices.append(e_idx)\n                data_values.append(1.0)\n            constraint_names.append(f"out_degree_node_{node_id}")\n            curr_row += 1\n            n_out += 1\n\n        # 2. Incoming degree constraints (sum_{e in in(v)} x_e <= 1)\n        for node_id, edge_indices in sorted(node_in_edges.items()):\n            for e_idx in edge_indices:\n                row_indices.append(curr_row)\n                col_indices.append(e_idx)\n                data_values.append(1.0)\n            constraint_names.append(f"in_degree_node_{node_id}")\n            curr_row += 1\n            n_inc += 1\n\n        M = curr_row\n        if M > 0:\n            sparse_A = csc_matrix((data_values, (row_indices, col_indices)), shape=(M, E), dtype=np.float64)\n            lb = np.zeros(M, dtype=np.float64)\n            ub = np.ones(M, dtype=np.float64)\n        else:\n            sparse_A = csc_matrix((0, E), dtype=np.float64)\n            lb = np.empty(0, dtype=np.float64)\n            ub = np.empty(0, dtype=np.float64)\n\n        return ConstraintMatrix(\n            matrix=sparse_A,\n            lb=lb,\n            ub=ub,\n            constraint_names=constraint_names,\n            num_incoming_constraints=n_inc,\n            num_outgoing_constraints=n_out\n        )\n'
src_files['src/optimization/diagnostics.py'] = '"""\nTrack Fragment Decoding, Optimization Diagnostics, and Ground-Truth Evaluation.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Dict, Any, Optional, Set\nimport numpy as np\n\nfrom src.graph.nodes import CanonicalNode, NodeSet\nfrom src.graph.candidate_graph import CandidateEdge, TemporalCandidateGraph\nfrom src.graph.diagnostics import NodeMatchingResult\nfrom src.optimization.base import TrackFragment, TrackingSolution\n\n\nclass TrackFragmentDecoder:\n    """\n    Decodes direct-continuation track fragments from selected ILP candidate edges.\n\n    A direct-continuation track fragment is a linear chain of consecutive nodes:\n        v_1 -> v_2 -> ... -> v_k\n    connected by selected candidate edges across consecutive frames.\n    """\n\n    def decode_fragments(\n        self,\n        node_set: NodeSet,\n        selected_edges: List[CandidateEdge]\n    ) -> List[TrackFragment]:\n        """\n        Decodes direct-continuation track fragments.\n\n        :param node_set: NodeSet container\n        :param selected_edges: List of selected CandidateEdge objects (x_e = 1)\n        :return: List of TrackFragment objects\n        """\n        if not selected_edges:\n            return []\n\n        # Build adjacency graph for selected edges\n        # outgoing[src] = dst\n        outgoing: Dict[int, int] = {}\n        # incoming[dst] = src\n        incoming: Dict[int, int] = {}\n        # edge_map[(src, dst)] = CandidateEdge\n        edge_map: Dict[Tuple[int, int], CandidateEdge] = {}\n\n        for edge in selected_edges:\n            src = edge.source_node_id\n            dst = edge.destination_node_id\n            outgoing[src] = dst\n            incoming[dst] = src\n            edge_map[(src, dst)] = edge\n\n        # Identify starting nodes for track fragments\n        # A starting node has an outgoing selected edge but NO incoming selected edge\n        all_src_nodes = set(outgoing.keys())\n        all_dst_nodes = set(incoming.keys())\n\n        start_node_ids = sorted(list(all_src_nodes - all_dst_nodes))\n\n        # Also catch any isolated cycles if they exist (unlikely in temporal DAG)\n        visited_nodes: Set[int] = set()\n        fragments: List[TrackFragment] = []\n        frag_id = 1\n\n        for start_id in start_node_ids:\n            if start_id in visited_nodes:\n                continue\n\n            curr = start_id\n            chain = [curr]\n            visited_nodes.add(curr)\n            total_dist = 0.0\n\n            while curr in outgoing:\n                nxt = outgoing[curr]\n                if nxt in visited_nodes:\n                    break  # Cycle detected or already visited\n\n                edge = edge_map.get((curr, nxt))\n                if edge:\n                    total_dist += edge.euclidean_distance_phys\n\n                chain.append(nxt)\n                visited_nodes.add(nxt)\n                curr = nxt\n\n            # Retrieve frame and position info for fragment nodes\n            frames = []\n            phys_positions = []\n            for nid in chain:\n                n = node_set.get_node(nid)\n                if n:\n                    frames.append(n.frame)\n                    phys_positions.append(n.physical_pos)\n                else:\n                    frames.append(0)\n                    phys_positions.append((0.0, 0.0, 0.0))\n\n            fragment = TrackFragment(\n                fragment_id=frag_id,\n                node_ids=chain,\n                frames=frames,\n                physical_positions=phys_positions,\n                length_frames=len(chain),\n                total_physical_distance_um=float(total_dist)\n            )\n            fragments.append(fragment)\n            frag_id += 1\n\n        return fragments\n\n\n@dataclass\nclass ILPDiagnosticReport:\n    """\n    Comprehensive diagnostic report evaluating optimization results and ground-truth alignment.\n    """\n    total_nodes: int\n    total_candidate_edges: int\n    selected_edges: int\n    rejected_edges: int\n    num_track_fragments: int\n    fragment_lengths: Dict[str, float]  # min, median, max, mean\n    longest_fragment_len: int\n    isolated_nodes_count: int\n    nodes_with_predecessor_only: int\n    nodes_with_successor_only: int\n    nodes_with_both: int\n    nodes_with_neither: int\n    constraint_violations_count: int\n    # Ground truth metrics\n    gt_edges_count: int\n    true_positives: int\n    false_positives: int\n    false_negatives: int\n    edge_precision: float\n    edge_recall: float\n    edge_f1: float\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Convert report to dictionary representation."""\n        return {\n            "total_nodes": self.total_nodes,\n            "total_candidate_edges": self.total_candidate_edges,\n            "selected_edges": self.selected_edges,\n            "rejected_edges": self.rejected_edges,\n            "num_track_fragments": self.num_track_fragments,\n            "fragment_lengths": self.fragment_lengths,\n            "longest_fragment_len": self.longest_fragment_len,\n            "isolated_nodes_count": self.isolated_nodes_count,\n            "nodes_with_predecessor_only": self.nodes_with_predecessor_only,\n            "nodes_with_successor_only": self.nodes_with_successor_only,\n            "nodes_with_both": self.nodes_with_both,\n            "nodes_with_neither": self.nodes_with_neither,\n            "constraint_violations_count": self.constraint_violations_count,\n            "gt_edges_count": self.gt_edges_count,\n            "true_positives": self.true_positives,\n            "false_positives": self.false_positives,\n            "false_negatives": self.false_negatives,\n            "edge_precision": self.edge_precision,\n            "edge_recall": self.edge_recall,\n            "edge_f1": self.edge_f1,\n        }\n\n\nclass ILPDiagnostics:\n    """\n    Evaluator performing Part 13 Track Fragment Diagnostics and Part 14 Ground-Truth Diagnostics.\n    """\n\n    def evaluate_solution(\n        self,\n        solution: TrackingSolution,\n        node_set: NodeSet,\n        geff_edges: Optional[List[Tuple[int, int]]] = None,\n        matching_result: Optional[NodeMatchingResult] = None\n    ) -> ILPDiagnosticReport:\n        """\n        Evaluates tracking solution metrics and verifies zero constraint violations.\n\n        :param solution: TrackingSolution object\n        :param node_set: NodeSet container\n        :param geff_edges: Optional list of directed (u_geff, v_geff) GT edges\n        :param matching_result: Optional NodeMatchingResult for node ID mapping\n        :return: ILPDiagnosticReport dataclass\n        """\n        selected = solution.selected_edges\n        rejected = solution.rejected_edges\n        all_nodes = node_set.all_nodes()\n        total_nodes = len(all_nodes)\n        total_cand_edges = len(selected) + len(rejected)\n\n        # 1. Degree counts and constraint violation check\n        inc_degree: Dict[int, int] = {}\n        out_degree: Dict[int, int] = {}\n\n        for edge in selected:\n            src = edge.source_node_id\n            dst = edge.destination_node_id\n\n            out_degree[src] = out_degree.get(src, 0) + 1\n            inc_degree[dst] = inc_degree.get(dst, 0) + 1\n\n        # Check constraint violations (>1 incoming or >1 outgoing)\n        violations = 0\n        for n in all_nodes:\n            nid = n.node_id\n            if inc_degree.get(nid, 0) > 1:\n                violations += (inc_degree[nid] - 1)\n            if out_degree.get(nid, 0) > 1:\n                violations += (out_degree[nid] - 1)\n\n        # Node degree categories\n        n_pred_only = 0\n        n_succ_only = 0\n        n_both = 0\n        n_neither = 0\n\n        for n in all_nodes:\n            nid = n.node_id\n            has_inc = inc_degree.get(nid, 0) > 0\n            has_out = out_degree.get(nid, 0) > 0\n\n            if has_inc and has_out:\n                n_both += 1\n            elif has_inc and not has_out:\n                n_pred_only += 1\n            elif not has_inc and has_out:\n                n_succ_only += 1\n            else:\n                n_neither += 1\n\n        # Fragment length stats\n        frags = solution.fragments\n        n_frags = len(frags)\n\n        if n_frags > 0:\n            lengths = [f.length_frames for f in frags]\n            frag_stats = {\n                "min": float(np.min(lengths)),\n                "median": float(np.median(lengths)),\n                "max": float(np.max(lengths)),\n                "mean": float(np.mean(lengths)),\n            }\n            longest_len = int(np.max(lengths))\n        else:\n            frag_stats = {"min": 0.0, "median": 0.0, "max": 0.0, "mean": 0.0}\n            longest_len = 0\n\n        # 2. Ground-truth edge evaluation\n        tp = fp = fn = 0\n        prec = rec = f1 = 0.0\n        gt_count = 0\n\n        if geff_edges is not None and matching_result is not None:\n            gt_map = matching_result.geff_to_canonical_map\n            true_can_pairs: Set[Tuple[int, int]] = set()\n\n            node_ids_in_set = set(n.node_id for n in all_nodes)\n            for u_gt, v_gt in geff_edges:\n                u_can = gt_map.get(int(u_gt))\n                v_can = gt_map.get(int(v_gt))\n                if u_can is not None and v_can is not None:\n                    if u_can in node_ids_in_set and v_can in node_ids_in_set:\n                        true_can_pairs.add((u_can, v_can))\n\n            gt_count = len(true_can_pairs)\n            selected_pairs = set((e.source_node_id, e.destination_node_id) for e in selected)\n\n            tp = len(selected_pairs & true_can_pairs)\n            fp = len(selected_pairs - true_can_pairs)\n            fn = len(true_can_pairs - selected_pairs)\n\n            prec = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0\n            rec = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0\n            f1 = float(2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0\n\n        return ILPDiagnosticReport(\n            total_nodes=total_nodes,\n            total_candidate_edges=total_cand_edges,\n            selected_edges=len(selected),\n            rejected_edges=len(rejected),\n            num_track_fragments=n_frags,\n            fragment_lengths=frag_stats,\n            longest_fragment_len=longest_len,\n            isolated_nodes_count=n_neither,\n            nodes_with_predecessor_only=n_pred_only,\n            nodes_with_successor_only=n_succ_only,\n            nodes_with_both=n_both,\n            nodes_with_neither=n_neither,\n            constraint_violations_count=violations,\n            gt_edges_count=gt_count,\n            true_positives=tp,\n            false_positives=fp,\n            false_negatives=fn,\n            edge_precision=prec,\n            edge_recall=rec,\n            edge_f1=f1\n        )\n'
src_files['src/optimization/ilp.py'] = '"""\nGlobal Integer Linear Programming (ILP) Optimization Engine.\n"""\n\nfrom typing import List, Dict, Any, Optional, Tuple, Union\nimport numpy as np\n\nfrom src.graph.nodes import CanonicalNode, NodeSet\nfrom src.graph.candidate_graph import TemporalCandidateGraph, CandidateEdge\nfrom src.optimization.base import OptimizationConfig, SolverResult, TrackingSolution\nfrom src.optimization.objective import ObjectiveBuilder\nfrom src.optimization.constraints import ConstraintBuilder\nfrom src.optimization.solver import MILPSolverWrapper\nfrom src.optimization.diagnostics import TrackFragmentDecoder\n\n\nclass GlobalILPOptimizer:\n    """\n    Global Integer Linear Programming (ILP) Tracking Optimizer.\n\n    Architectural Principle:\n    "Local mechanisms propose; global optimization decides."\n\n    Formulates and solves binary one-to-one temporal continuation tracking optimization\n    subject to degree constraints and STGNN/geometric log-odds objective rewards.\n    """\n\n    def __init__(self, config: Optional[OptimizationConfig] = None):\n        self.config = config or OptimizationConfig()\n        self.objective_builder = ObjectiveBuilder(config=self.config)\n        self.constraint_builder = ConstraintBuilder()\n        self.solver = MILPSolverWrapper(config=self.config)\n        self.fragment_decoder = TrackFragmentDecoder()\n\n    def optimize(\n        self,\n        node_set: NodeSet,\n        graph: TemporalCandidateGraph,\n        stgnn_scores: Optional[np.ndarray] = None\n    ) -> TrackingSolution:\n        """\n        Runs global ILP optimization on candidate graph.\n\n        :param node_set: NodeSet container\n        :param graph: TemporalCandidateGraph object\n        :param stgnn_scores: Optional STGNN probability score array shape (E,)\n        :return: TrackingSolution object\n        """\n        edges = graph.edges\n        E = len(edges)\n\n        if E == 0:\n            empty_result = SolverResult(\n                status="optimal",\n                success=True,\n                objective_value=0.0,\n                solve_time_sec=0.0,\n                num_variables=0,\n                num_constraints=0,\n                selected_edge_count=0,\n                termination_reason="empty_graph"\n            )\n            return TrackingSolution(\n                selected_edges=[],\n                rejected_edges=[],\n                fragments=[],\n                solver_result=empty_result\n            )\n\n        # 1. Build minimization cost vector c_min = -w_e\n        c_min = self.objective_builder.build_minimization_vector(edges, stgnn_scores=stgnn_scores)\n\n        # 2. Build flow constraints matrix (in_degree <= 1, out_degree <= 1)\n        constraints = self.constraint_builder.build_flow_constraints(node_set, edges)\n\n        # 3. Execute MILP solver\n        x_sol, solver_result = self.solver.solve(c_min, constraints)\n\n        # 4. Partition selected and rejected edges\n        selected_edges = [edges[idx] for idx in range(E) if x_sol[idx] == 1.0]\n        rejected_edges = [edges[idx] for idx in range(E) if x_sol[idx] == 0.0]\n\n        # 5. Decode direct-continuation track fragments\n        fragments = self.fragment_decoder.decode_fragments(node_set, selected_edges)\n\n        solution = TrackingSolution(\n            selected_edges=selected_edges,\n            rejected_edges=rejected_edges,\n            fragments=fragments,\n            solver_result=solver_result\n        )\n\n        # 6. Part 17 Solver Sanity Checks\n        self._verify_solution_sanity(solution, node_set, graph)\n\n        return solution\n\n    def _verify_solution_sanity(\n        self,\n        solution: TrackingSolution,\n        node_set: NodeSet,\n        graph: TemporalCandidateGraph\n    ) -> None:\n        """\n        Part 17 — Verifies solution validity and throws error if any check fails.\n        """\n        res = solution.solver_result\n        assert res.success or res.status == "optimal", f"Solver failed with status: {res.status}"\n        assert np.isfinite(res.objective_value), "Objective value is non-finite!"\n\n        # All selected edges must belong to candidate graph\n        graph_edge_pairs = set((e.source_node_id, e.destination_node_id) for e in graph.edges)\n        for edge in solution.selected_edges:\n            pair = (edge.source_node_id, edge.destination_node_id)\n            assert pair in graph_edge_pairs, f"Selected edge {pair} not found in candidate graph!"\n            assert node_set.get_node(edge.source_node_id) is not None, f"Source node {edge.source_node_id} missing!"\n            assert node_set.get_node(edge.destination_node_id) is not None, f"Dest node {edge.destination_node_id} missing!"\n\n        # Degree constraints check\n        inc_counts: Dict[int, int] = {}\n        out_counts: Dict[int, int] = {}\n\n        for edge in solution.selected_edges:\n            src = edge.source_node_id\n            dst = edge.destination_node_id\n            out_counts[src] = out_counts.get(src, 0) + 1\n            inc_counts[dst] = inc_counts.get(dst, 0) + 1\n\n        for src, cnt in out_counts.items():\n            assert cnt <= 1, f"Node {src} violates outgoing degree constraint: {cnt} > 1"\n\n        for dst, cnt in inc_counts.items():\n            assert cnt <= 1, f"Node {dst} violates incoming degree constraint: {cnt} > 1"\n'
src_files['src/optimization/objective.py'] = '"""\nObjective Function Construction and Log-Odds Score Transformations.\n"""\n\nfrom typing import List, Dict, Any, Union, Optional\nimport numpy as np\n\nfrom src.graph.candidate_graph import TemporalCandidateGraph, CandidateEdge\nfrom src.optimization.base import OptimizationConfig\n\n\nclass ObjectiveBuilder:\n    """\n    Constructs edge weight coefficients for the Global ILP optimization objective.\n\n    Transforms candidate probabilities/scores into calibrated log-odds weights:\n        w(p) = log((p + eps) / (1 - p + eps))\n\n    Objective formulation (Maximization):\n        Maximize  sum_{e in E} w_e * x_e\n    Converted for standard minimization solvers:\n        Minimize  sum_{e in E} (-w_e) * x_e\n    """\n\n    def __init__(self, config: Optional[OptimizationConfig] = None):\n        self.config = config or OptimizationConfig()\n\n    def log_odds(self, p: Union[float, np.ndarray]) -> Union[float, np.ndarray]:\n        """\n        Log-odds transformation mapping probabilities in [0, 1] to linear evidence weights.\n\n        :param p: Probability value or NumPy array of probabilities\n        :return: Log-odds value(s) in (-inf, +inf)\n        """\n        eps = self.config.epsilon\n        p_clipped = np.clip(p, eps, 1.0 - eps)\n        return np.log((p_clipped + eps) / (1.0 - p_clipped + eps))\n\n    def compute_edge_weights(\n        self,\n        edges: List[CandidateEdge],\n        stgnn_scores: Optional[np.ndarray] = None\n    ) -> np.ndarray:\n        """\n        Computes edge weight array w_e for candidate edges based on configured mode.\n\n        :param edges: List of candidate edges\n        :param stgnn_scores: Optional STGNN probability array shape (E,)\n        :return: NumPy array shape (E,) float64 containing edge reward weights\n        """\n        E = len(edges)\n        if E == 0:\n            return np.empty(0, dtype=np.float64)\n\n        distances = np.array([e.euclidean_distance_phys for e in edges], dtype=np.float64)\n        geom_probs = np.exp(-distances / max(1e-6, self.config.sigma_scale_um))\n        geom_log_odds = self.log_odds(geom_probs)\n\n        if stgnn_scores is not None and len(stgnn_scores) == E:\n            stgnn_log_odds = self.log_odds(stgnn_scores)\n        else:\n            stgnn_log_odds = geom_log_odds\n\n        mode = self.config.mode.lower()\n        if mode == "stgnn_only":\n            weights = stgnn_log_odds\n        elif mode == "geometry_only":\n            weights = geom_log_odds\n        elif mode == "combined":\n            weights = self.config.alpha * stgnn_log_odds + self.config.beta * geom_log_odds\n        else:\n            raise ValueError(f"Unknown objective mode: \'{mode}\'. Use \'stgnn_only\', \'geometry_only\', or \'combined\'.")\n\n        return weights.astype(np.float64)\n\n    def build_minimization_vector(\n        self,\n        edges: List[CandidateEdge],\n        stgnn_scores: Optional[np.ndarray] = None\n    ) -> np.ndarray:\n        """\n        Constructs linear objective coefficient vector c for SciPy minimization solver (c_e = -w_e).\n\n        :return: NumPy array shape (E,) float64 (cost vector for minimization)\n        """\n        weights = self.compute_edge_weights(edges, stgnn_scores=stgnn_scores)\n        return (-weights).astype(np.float64)\n'
src_files['src/optimization/solver.py'] = '"""\nOpen-Source MILP Solver Wrapper using SciPy (HiGHS MILP Solver).\n"""\n\nfrom typing import Tuple, Optional\nimport time\nimport numpy as np\nfrom scipy.optimize import milp, Bounds, LinearConstraint\n\nfrom src.optimization.base import SolverResult, OptimizationConfig\nfrom src.optimization.constraints import ConstraintMatrix\n\n\nclass MILPSolverWrapper:\n    """\n    Transparent solver wrapper for binary mixed-integer linear programming (MILP).\n\n    Employs the open-source HiGHS solver integrated natively in scipy.optimize.milp.\n    """\n\n    def __init__(self, config: Optional[OptimizationConfig] = None):\n        self.config = config or OptimizationConfig()\n\n    def solve(\n        self,\n        c_min: np.ndarray,  # (E,) minimization cost vector (-w_e)\n        constraints: ConstraintMatrix\n    ) -> Tuple[np.ndarray, SolverResult]:\n        """\n        Solves binary MILP optimization problem:\n            Minimize   c_min^T x\n            Subject to lb <= A x <= ub\n                       x_e in {0, 1}\n\n        :param c_min: Cost vector shape (E,)\n        :param constraints: ConstraintMatrix object\n        :return: Tuple of (binary solution vector x_e shape (E,) float64, SolverResult)\n        """\n        E = len(c_min)\n        if E == 0:\n            result = SolverResult(\n                status="optimal",\n                success=True,\n                objective_value=0.0,\n                solve_time_sec=0.0,\n                num_variables=0,\n                num_constraints=0,\n                selected_edge_count=0,\n                termination_reason="empty_graph"\n            )\n            return np.empty(0, dtype=np.float64), result\n\n        # Binary integrality constraints (1 = integer/binary)\n        integrality = np.ones(E, dtype=np.int32)\n        # Variable bounds [0, 1]\n        bounds = Bounds(np.zeros(E, dtype=np.float64), np.ones(E, dtype=np.float64))\n\n        if constraints is not None and constraints.matrix.shape[0] > 0:\n            scipy_constraints = constraints.to_scipy_constraint()\n        else:\n            scipy_constraints = None\n        options = {"time_limit": max(1.0, float(self.config.timeout_sec))}\n\n        start_time = time.time()\n        res = milp(\n            c=c_min,\n            integrality=integrality,\n            bounds=bounds,\n            constraints=scipy_constraints,\n            options=options\n        )\n        solve_time = float(time.time() - start_time)\n\n        # Interpret status codes\n        status_map = {\n            0: "optimal",\n            1: "iteration_limit",\n            2: "time_limit",\n            3: "infeasible",\n            4: "unbounded",\n        }\n        status_str = status_map.get(res.status, f"unknown_code_{res.status}")\n        term_reason = res.message if hasattr(res, "message") else status_str\n\n        x_sol = np.zeros(E, dtype=np.float64)\n        if res.x is not None:\n            # Round binary variables cleanly\n            x_sol = np.round(np.clip(res.x, 0.0, 1.0)).astype(np.float64)\n\n        selected_count = int(np.sum(x_sol == 1.0))\n        # Maximization objective value is -c_min^T x = -res.fun\n        max_obj_val = float(-res.fun) if res.fun is not None else 0.0\n\n        solver_result = SolverResult(\n            status=status_str,\n            success=bool(res.success or res.status == 0),\n            objective_value=max_obj_val,\n            solve_time_sec=solve_time,\n            num_variables=E,\n            num_constraints=constraints.matrix.shape[0] if (constraints is not None and constraints.matrix is not None) else 0,\n            selected_edge_count=selected_count,\n            termination_reason=term_reason\n        )\n\n        return x_sol, solver_result\n'
src_files['src/regional/__init__.py'] = '"""\nSpatial Regional Processing with Guard Zones and Overlap Reconciliation.\n"""\n\nfrom .geometry import BoundingBox3D, Region3D, RegionalCandidate\nfrom .partitioner import RegionPartitioner3D, PartitionConfig\nfrom .reconciliation import CandidateReconciler, ReconciliationConfig\nfrom .inference import RegionalInferenceEngine\nfrom .diagnostics import RegionalBoundaryAnalyzer, BoundaryDiagnostics\n\n__all__ = [\n    "BoundingBox3D",\n    "Region3D",\n    "RegionalCandidate",\n    "RegionPartitioner3D",\n    "PartitionConfig",\n    "CandidateReconciler",\n    "ReconciliationConfig",\n    "RegionalInferenceEngine",\n    "RegionalBoundaryAnalyzer",\n    "BoundaryDiagnostics",\n]\n'
src_files['src/regional/diagnostics.py'] = '"""\nBoundary Proximity Diagnostics & Full-Frame vs. Regional Inference Comparator.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Dict, Any, Optional\nimport numpy as np\n\nfrom src.data.nodes import CellNode\nfrom src.data.coordinates import CoordinateTransformer\nfrom src.detection.base import DetectionCandidate\nfrom src.detection.evaluation import DetectionEvaluator, DetectionMetrics\nfrom src.regional.geometry import Region3D, RegionalCandidate\nfrom src.regional.partitioner import RegionPartitioner3D, PartitionConfig\n\n\n@dataclass\nclass BoundaryDiagnostics:\n    """\n    Diagnostic results comparing Full-Frame vs. Regional Inference and evaluating boundary performance.\n    """\n    frame_idx: int\n    num_regions: int\n    fullframe_candidate_count: int\n    regional_authoritative_raw_count: int\n    regional_reconciled_count: int\n    multi_region_duplicate_count: int\n    fullframe_metrics: DetectionMetrics\n    regional_metrics: DetectionMetrics\n    near_boundary_gt_count: int\n    near_boundary_recall: float\n    deep_core_gt_count: int\n    deep_core_recall: float\n\n    def summary_string(self) -> str:\n        return (\n            f"=== Boundary Diagnostics (Frame {self.frame_idx}) ===\\n"\n            f"Regions: {self.num_regions} | Full-Frame Candidates: {self.fullframe_candidate_count}\\n"\n            f"Regional Authoritative (Raw): {self.regional_authoritative_raw_count} | Reconciled: {self.regional_reconciled_count}\\n"\n            f"Multi-Region Duplicates Reconciled: {self.multi_region_duplicate_count}\\n"\n            f"--------------------------------------------------\\n"\n            f"Full-Frame Metrics -> Prec: {self.fullframe_metrics.precision:.4f} | Rec: {self.fullframe_metrics.recall:.4f} | F1: {self.fullframe_metrics.f1_score:.4f} | Loc Error: {self.fullframe_metrics.mean_localization_error_um:.3f} um\\n"\n            f"Regional   Metrics -> Prec: {self.regional_metrics.precision:.4f} | Rec: {self.regional_metrics.recall:.4f} | F1: {self.regional_metrics.f1_score:.4f} | Loc Error: {self.regional_metrics.mean_localization_error_um:.3f} um\\n"\n            f"--------------------------------------------------\\n"\n            f"Near-Boundary GT Cells (<= 5 um to core border): {self.near_boundary_gt_count} | Regional Recall: {self.near_boundary_recall:.4f}\\n"\n            f"Deep-Core     GT Cells (> 5 um to core border):  {self.deep_core_gt_count} | Regional Recall: {self.deep_core_recall:.4f}"\n        )\n\n\nclass RegionalBoundaryAnalyzer:\n    """\n    Analyzes spatial boundary behavior and compares full-frame vs regional detection outputs.\n    """\n\n    def __init__(\n        self,\n        evaluator: Optional[DetectionEvaluator] = None,\n        transformer: Optional[CoordinateTransformer] = None,\n        boundary_threshold_um: float = 5.0\n    ):\n        self.evaluator = evaluator or DetectionEvaluator(max_matching_distance_um=5.0)\n        self.transformer = transformer or CoordinateTransformer()\n        self.boundary_threshold_um = boundary_threshold_um\n\n    def analyze_frame(\n        self,\n        frame_idx: int,\n        gt_nodes: List[CellNode],\n        fullframe_candidates: List[DetectionCandidate],\n        regional_candidates: List[RegionalCandidate],\n        regional_metadata: Dict[str, Any],\n        partitioner: RegionPartitioner3D\n    ) -> BoundaryDiagnostics:\n        """\n        Analyzes full-frame vs regional detection performance and assesses boundary behavior.\n        """\n        # 1. Full-frame evaluation\n        ff_metrics = self.evaluator.evaluate_frame(gt_nodes, fullframe_candidates)\n\n        # 2. Regional evaluation (convert RegionalCandidate to DetectionCandidate for evaluator)\n        reg_cands_as_det = [\n            DetectionCandidate(\n                candidate_id=rc.candidate_id,\n                frame=rc.frame,\n                voxel_pos=rc.global_voxel_pos,\n                physical_pos=rc.physical_pos,\n                confidence=rc.confidence\n            )\n            for rc in regional_candidates\n        ]\n        reg_metrics = self.evaluator.evaluate_frame(gt_nodes, reg_cands_as_det)\n\n        # 3. Categorize GT nodes into Near-Boundary vs Deep-Core\n        regions = partitioner.generate_regions()\n        near_boundary_gt: List[CellNode] = []\n        deep_core_gt: List[CellNode] = []\n\n        for gt in gt_nodes:\n            # Find the region core that owns this GT node\n            owning_region = None\n            for r in regions:\n                if r.is_global_voxel_in_core(gt.voxel_pos):\n                    owning_region = r\n                    break\n\n            if owning_region is not None:\n                dist_to_border = owning_region.min_distance_to_core_boundary_phys(gt.physical_pos, self.transformer)\n                if dist_to_border <= self.boundary_threshold_um:\n                    near_boundary_gt.append(gt)\n                else:\n                    deep_core_gt.append(gt)\n            else:\n                near_boundary_gt.append(gt)\n\n        # Evaluate regional recall separately on Near-Boundary vs Deep-Core GT nodes\n        nb_metrics = self.evaluator.evaluate_frame(near_boundary_gt, reg_cands_as_det)\n        dc_metrics = self.evaluator.evaluate_frame(deep_core_gt, reg_cands_as_det)\n\n        return BoundaryDiagnostics(\n            frame_idx=frame_idx,\n            num_regions=regional_metadata.get("num_regions", len(regions)),\n            fullframe_candidate_count=len(fullframe_candidates),\n            regional_authoritative_raw_count=regional_metadata.get("num_authoritative_before_reconciliation", 0),\n            regional_reconciled_count=len(regional_candidates),\n            multi_region_duplicate_count=regional_metadata.get("num_multi_region_duplicates_reconciled", 0),\n            fullframe_metrics=ff_metrics,\n            regional_metrics=reg_metrics,\n            near_boundary_gt_count=len(near_boundary_gt),\n            near_boundary_recall=nb_metrics.recall,\n            deep_core_gt_count=len(deep_core_gt),\n            deep_core_recall=dc_metrics.recall\n        )\n'
src_files['src/regional/geometry.py'] = '"""\nGeometry definitions, coordinate transformations, and candidate representations for 3D Regional Processing.\n"""\n\nfrom dataclasses import dataclass, field\nfrom typing import Tuple, List, Dict, Any, Optional\nimport math\nimport numpy as np\n\nfrom src.data.nodes import CellNode\nfrom src.data.coordinates import CoordinateTransformer\n\n\n@dataclass(frozen=True)\nclass BoundingBox3D:\n    """3D axis-aligned bounding box defined by half-open slice ranges [min, max)."""\n    z_min: int\n    z_max: int\n    y_min: int\n    y_max: int\n    x_min: int\n    x_max: int\n\n    @property\n    def shape(self) -> Tuple[int, int, int]:\n        """Returns (dz, dy, dx) shape tuple."""\n        return (self.z_max - self.z_min, self.y_max - self.y_min, self.x_max - self.x_min)\n\n    @property\n    def slice_tuples(self) -> Tuple[Tuple[int, int], Tuple[int, int], Tuple[int, int]]:\n        """Returns ((z_min, z_max), (y_min, y_max), (x_min, x_max))."""\n        return ((self.z_min, self.z_max), (self.y_min, self.y_max), (self.x_min, self.x_max))\n\n    def contains_voxel(self, z: int, y: int, x: int) -> bool:\n        """Checks if voxel (z, y, x) lies inside the bounding box [min, max)."""\n        return (\n            self.z_min <= z < self.z_max and\n            self.y_min <= y < self.y_max and\n            self.x_min <= x < self.x_max\n        )\n\n\n@dataclass\nclass Region3D:\n    """\n    Defines a 3D spatial processing region with an authoritative CORE volume\n    and an expanded contextual GUARD volume.\n    """\n    region_id: int\n    core_bbox: BoundingBox3D\n    guard_bbox: BoundingBox3D\n\n    @property\n    def global_origin(self) -> Tuple[int, int, int]:\n        """Returns origin (z_min, y_min, x_min) of guard volume in global coordinates."""\n        return (self.guard_bbox.z_min, self.guard_bbox.y_min, self.guard_bbox.x_min)\n\n    def local_to_global_voxel(self, local_zyx: Tuple[int, int, int]) -> Tuple[int, int, int]:\n        """Converts local region voxel coordinates to global volume voxel coordinates."""\n        oz, oy, ox = self.global_origin\n        lz, ly, lx = local_zyx\n        return (lz + oz, ly + oy, lx + ox)\n\n    def global_to_local_voxel(self, global_zyx: Tuple[int, int, int]) -> Tuple[int, int, int]:\n        """Converts global volume voxel coordinates to local region voxel coordinates."""\n        oz, oy, ox = self.global_origin\n        gz, gy, gx = global_zyx\n        return (gz - oz, gy - oy, gx - ox)\n\n    def is_global_voxel_in_core(self, global_zyx: Tuple[int, int, int]) -> bool:\n        """Authoritative Core Membership Check: returns True iff voxel center is inside CORE."""\n        gz, gy, gx = global_zyx\n        return self.core_bbox.contains_voxel(gz, gy, gx)\n\n    def is_global_voxel_in_guard(self, global_zyx: Tuple[int, int, int]) -> bool:\n        """Guard Membership Check: returns True iff voxel center is inside GUARD volume."""\n        gz, gy, gx = global_zyx\n        return self.guard_bbox.contains_voxel(gz, gy, gx)\n\n    def min_distance_to_core_boundary_phys(\n        self,\n        global_phys_zyx: Tuple[float, float, float],\n        transformer: Optional[CoordinateTransformer] = None\n    ) -> float:\n        """\n        Computes minimum physical distance (um) from a global physical coordinate\n        to the nearest 6 faces of the region\'s authoritative CORE bounding box.\n        """\n        transformer = transformer or CoordinateTransformer()\n        core_min_phys = transformer.voxel_to_physical(\n            (self.core_bbox.z_min, self.core_bbox.y_min, self.core_bbox.x_min)\n        )\n        core_max_phys = transformer.voxel_to_physical(\n            (self.core_bbox.z_max, self.core_bbox.y_max, self.core_bbox.x_max)\n        )\n\n        gz_p, gy_p, gx_p = global_phys_zyx\n\n        # Distances to core boundaries along each axis (um)\n        dist_z = min(abs(gz_p - core_min_phys[0]), abs(gz_p - core_max_phys[0]))\n        dist_y = min(abs(gy_p - core_min_phys[1]), abs(gy_p - core_max_phys[1]))\n        dist_x = min(abs(gx_p - core_min_phys[2]), abs(gx_p - core_max_phys[2]))\n\n        return float(min(dist_z, dist_y, dist_x))\n\n\n@dataclass\nclass RegionalCandidate:\n    """\n    Represents a detection candidate emitted by regional inference, storing full\n    provenance and core membership tracking.\n    """\n    candidate_id: int\n    region_id: int\n    frame: int\n    local_voxel_pos: Tuple[int, int, int]\n    global_voxel_pos: Tuple[int, int, int]\n    physical_pos: Tuple[float, float, float]\n    confidence: float\n    is_in_core: bool\n    provenance: Dict[str, Any] = field(default_factory=dict)\n\n    def to_cell_node(self) -> CellNode:\n        """Converts regional candidate to internal CellNode representation."""\n        attrs = dict(self.provenance)\n        attrs["confidence"] = self.confidence\n        attrs["region_id"] = self.region_id\n        attrs["is_in_core"] = self.is_in_core\n        return CellNode(\n            node_id=self.candidate_id,\n            frame=self.frame,\n            voxel_pos=self.global_voxel_pos,\n            physical_pos=self.physical_pos,\n            attributes=attrs\n        )\n'
src_files['src/regional/inference.py'] = '"""\nRegional Inference Engine wrapping BaseDetector with Guard Zone Processing and Overlap Reconciliation.\n"""\n\nfrom typing import List, Tuple, Optional, Dict, Any\nimport numpy as np\nimport torch\n\nfrom src.data.zarr_loader import ZarrLoader\nfrom src.data.coordinates import CoordinateTransformer\nfrom src.detection.base import BaseDetector, DetectionCandidate\nfrom src.detection.peaks import PeakExtractor, PeakConfig\nfrom src.regional.geometry import Region3D, RegionalCandidate\nfrom src.regional.partitioner import RegionPartitioner3D, PartitionConfig\nfrom src.regional.reconciliation import CandidateReconciler, ReconciliationConfig\n\n\nclass RegionalInferenceEngine:\n    """\n    Decoupled Regional Inference Engine.\n\n    Separation of Responsibilities:\n    - The detector detects cells within a given 3D input tensor.\n    - The regional inference engine determines WHERE and with WHAT contextual volume (CORE + GUARD)\n      the detector operates, transforms local coordinates to global physical coordinates, enforces\n      authoritative CORE emission rules, and reconciles overlap duplicates.\n    """\n\n    def __init__(\n        self,\n        detector: BaseDetector,\n        partition_config: PartitionConfig = PartitionConfig(),\n        peak_config: PeakConfig = PeakConfig(),\n        reconciliation_config: ReconciliationConfig = ReconciliationConfig(),\n        transformer: Optional[CoordinateTransformer] = None\n    ):\n        self.detector = detector\n        self.partition_config = partition_config\n        self.peak_config = peak_config\n        self.reconciliation_config = reconciliation_config\n        self.transformer = transformer or CoordinateTransformer()\n\n        self.peak_extractor = PeakExtractor(config=peak_config, resolution=self.transformer.resolution)\n        self.reconciler = CandidateReconciler(config=reconciliation_config, transformer=self.transformer)\n\n    def _normalize_crop(\n        self,\n        crop_3d: np.ndarray,\n        norm_min: float = 0.0,\n        norm_max: float = 2000.0\n    ) -> np.ndarray:\n        """Normalizes 3D uint16 microscopy image slice to float32 in [0, 1]."""\n        clipped = np.clip(crop_3d.astype(np.float32), norm_min, norm_max)\n        normalized = (clipped - norm_min) / (norm_max - norm_min + 1e-6)\n        return normalized.astype(np.float32)\n\n    def detect_frame_regional(\n        self,\n        zarr_loader: ZarrLoader,\n        frame_idx: int,\n        filter_core_only: bool = True\n    ) -> Tuple[List[RegionalCandidate], Dict[str, Any]]:\n        """\n        Runs complete regional inference on a single 3D microscopy frame.\n\n        :param zarr_loader: ZarrLoader instance\n        :param frame_idx: Frame index t\n        :param filter_core_only: If True, enforces CRITICAL RULE: only peaks located inside\n               region\'s authoritative CORE are emitted as candidates by that region.\n        :return: (reconciled_candidates, execution_metadata)\n        """\n        volume_shape = zarr_loader.shape[1:]  # (Z, Y, X)\n        partitioner = RegionPartitioner3D(volume_shape=volume_shape, config=self.partition_config)\n        regions = partitioner.generate_regions()\n\n        raw_regional_candidates: List[RegionalCandidate] = []\n        raw_core_candidates: List[RegionalCandidate] = []\n        raw_guard_candidates: List[RegionalCandidate] = []\n\n        total_peaks_extracted = 0\n        candidate_counter = 1\n\n        for region in regions:\n            # 1. Lazily slice region volume (Guard volume slice)\n            z_range, y_range, x_range = region.guard_bbox.slice_tuples\n            raw_crop = zarr_loader.get_crop(frame_idx, z_range, y_range, x_range)\n            norm_crop = self._normalize_crop(raw_crop)\n\n            # 2. Run detector on regional volume\n            heatmap_3d = self.detector.predict_heatmap(norm_crop)\n\n            # 3. Extract local peak candidates in region\n            local_peaks = self.peak_extractor.extract_peaks_from_heatmap(heatmap_3d, frame_idx)\n            total_peaks_extracted += len(local_peaks)\n\n            # 4. Transform coordinates & check CORE vs GUARD membership\n            for pk in local_peaks:\n                local_zyx = pk.voxel_pos\n                global_zyx = region.local_to_global_voxel(local_zyx)\n                global_phys = self.transformer.voxel_to_physical(global_zyx)\n\n                is_in_core = region.is_global_voxel_in_core(global_zyx)\n\n                provenance = {\n                    "source_region_id": region.region_id,\n                    "local_voxel_pos": local_zyx,\n                    "global_voxel_pos": global_zyx,\n                    "physical_pos": global_phys,\n                    "is_in_core": is_in_core,\n                    "dist_to_core_boundary_um": region.min_distance_to_core_boundary_phys(global_phys, self.transformer)\n                }\n\n                reg_cand = RegionalCandidate(\n                    candidate_id=candidate_counter,\n                    region_id=region.region_id,\n                    frame=frame_idx,\n                    local_voxel_pos=local_zyx,\n                    global_voxel_pos=global_zyx,\n                    physical_pos=global_phys,\n                    confidence=pk.confidence,\n                    is_in_core=is_in_core,\n                    provenance=provenance\n                )\n                candidate_counter += 1\n\n                if is_in_core:\n                    raw_core_candidates.append(reg_cand)\n                else:\n                    raw_guard_candidates.append(reg_cand)\n\n                if filter_core_only:\n                    if is_in_core:\n                        raw_regional_candidates.append(reg_cand)\n                else:\n                    raw_regional_candidates.append(reg_cand)\n\n        # 5. Spatial Duplicate Reconciliation across guard boundaries\n        reconciled_candidates = self.reconciler.reconcile_candidates(raw_regional_candidates)\n\n        metadata = {\n            "frame_idx": frame_idx,\n            "num_regions": len(regions),\n            "total_raw_peaks_extracted": total_peaks_extracted,\n            "num_raw_core_candidates": len(raw_core_candidates),\n            "num_raw_guard_candidates": len(raw_guard_candidates),\n            "num_authoritative_before_reconciliation": len(raw_regional_candidates),\n            "num_authoritative_after_reconciliation": len(reconciled_candidates),\n            "num_multi_region_duplicates_reconciled": sum(\n                1 for c in reconciled_candidates if c.provenance.get("is_multi_region_detection", False)\n            )\n        }\n\n        return reconciled_candidates, metadata\n'
src_files['src/regional/partitioner.py'] = '"""\nConfigurable 3D Volume Region Partitioner with Authoritative Core Partitioning & Contextual Guard Expansion.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import Tuple, List, Dict, Set\nimport numpy as np\n\nfrom src.regional.geometry import BoundingBox3D, Region3D\n\n\n@dataclass\nclass PartitionConfig:\n    """\n    Configuration for 3D Volume Partitioning.\n\n    - core_shape: (dz, dy, dx) voxel dimensions for non-overlapping authoritative cores.\n    - guard_size: (gz, gy, gx) voxel padding expansion for contextual guard volumes.\n    """\n    core_shape: Tuple[int, int, int] = (32, 128, 128)\n    guard_size: Tuple[int, int, int] = (8, 32, 32)\n\n\nclass RegionPartitioner3D:\n    """\n    Partitions arbitrary 3D volumetric images (Z, Y, X) into regular grid regions.\n\n    Guarantees:\n    1. Complete Core Coverage: Every voxel in the volume belongs to exactly one authoritative core.\n    2. No Overlapping Cores: Region cores are strictly non-overlapping.\n    3. Contextual Guards: Regions extend beyond cores by guard_size, clipped at volume boundaries.\n    """\n\n    def __init__(\n        self,\n        volume_shape: Tuple[int, int, int],\n        config: PartitionConfig = PartitionConfig()\n    ):\n        self.volume_shape = volume_shape\n        self.config = config\n\n    def generate_regions(self) -> List[Region3D]:\n        """\n        Generates 3D Region3D instances covering the full volume.\n        """\n        z_dim, y_dim, x_dim = self.volume_shape\n        cz, cy, cx = self.config.core_shape\n        gz, gy, gx = self.config.guard_size\n\n        regions: List[Region3D] = []\n        region_id = 1\n\n        z_starts = list(range(0, z_dim, cz))\n        y_starts = list(range(0, y_dim, cy))\n        x_starts = list(range(0, x_dim, cx))\n\n        for zs in z_starts:\n            ze = min(z_dim, zs + cz)\n            for ys in y_starts:\n                ye = min(y_dim, ys + cy)\n                for xs in x_starts:\n                    xe = min(x_dim, xs + cx)\n\n                    # Authoritative Core Bounding Box\n                    core_bbox = BoundingBox3D(\n                        z_min=zs, z_max=ze,\n                        y_min=ys, y_max=ye,\n                        x_min=xs, x_max=xe\n                    )\n\n                    # Contextual Guard Bounding Box (Expanded & Clipped at Boundaries)\n                    guard_bbox = BoundingBox3D(\n                        z_min=max(0, zs - gz),\n                        z_max=min(z_dim, ze + gz),\n                        y_min=max(0, ys - gy),\n                        y_max=min(y_dim, ye + gy),\n                        x_min=max(0, xs - gx),\n                        x_max=min(x_dim, xe + gx)\n                    )\n\n                    region = Region3D(\n                        region_id=region_id,\n                        core_bbox=core_bbox,\n                        guard_bbox=guard_bbox\n                    )\n                    regions.append(region)\n                    region_id += 1\n\n        return regions\n\n    def verify_complete_core_coverage(self, regions: List[Region3D]) -> bool:\n        """\n        Verifies that every single voxel in the volume is covered by exactly one authoritative core.\n        """\n        z_dim, y_dim, x_dim = self.volume_shape\n        coverage_map = np.zeros(self.volume_shape, dtype=int)\n\n        for r in regions:\n            c = r.core_bbox\n            coverage_map[c.z_min:c.z_max, c.y_min:c.y_max, c.x_min:c.x_max] += 1\n\n        # Every voxel must equal exactly 1\n        return bool((coverage_map == 1).all())\n\n    def verify_no_overlapping_cores(self, regions: List[Region3D]) -> bool:\n        """\n        Verifies that no two region cores overlap.\n        """\n        n_regions = len(regions)\n        for i in range(n_regions):\n            for j in range(i + 1, n_regions):\n                c1 = regions[i].core_bbox\n                c2 = regions[j].core_bbox\n\n                overlap_z = max(0, min(c1.z_max, c2.z_max) - max(c1.z_min, c2.z_min))\n                overlap_y = max(0, min(c1.y_max, c2.y_max) - max(c1.y_min, c2.y_min))\n                overlap_x = max(0, min(c1.x_max, c2.x_max) - max(c1.x_min, c2.x_min))\n\n                if overlap_z > 0 and overlap_y > 0 and overlap_x > 0:\n                    return False\n        return True\n'
src_files['src/regional/reconciliation.py'] = '"""\nSpatial Overlap Duplicate Candidate Reconciliation across Region Guard Boundaries.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Dict, Any, Optional\nimport numpy as np\n\nfrom src.data.coordinates import CoordinateTransformer\nfrom src.regional.geometry import RegionalCandidate\n\n\n@dataclass\nclass ReconciliationConfig:\n    """\n    Configuration for spatial duplicate candidate reconciliation.\n\n    - reconciliation_distance_um: Physical distance threshold (micrometers) within which\n      predictions from neighboring guard/core overlaps are considered duplicate observations\n      of the same biological cell.\n    - merge_strategy: Strategy for selecting/merging primary detection (\'highest_confidence\' or \'core_priority\').\n    """\n    reconciliation_distance_um: float = 4.0  # Matching distance threshold in micrometers (um)\n    merge_strategy: str = "highest_confidence"\n\n\nclass CandidateReconciler:\n    """\n    Reconciles duplicate cell-center detections produced across overlapping region guard zones.\n\n    Guarantees:\n    - Merges duplicate observations of the same cell in 3D physical space.\n    - Preserves complete provenance (contributing region IDs, confidence scores, multi-region flag).\n    """\n\n    def __init__(\n        self,\n        config: ReconciliationConfig = ReconciliationConfig(),\n        transformer: Optional[CoordinateTransformer] = None\n    ):\n        self.config = config\n        self.transformer = transformer or CoordinateTransformer()\n\n    def reconcile_candidates(\n        self,\n        candidates: List[RegionalCandidate]\n    ) -> List[RegionalCandidate]:\n        """\n        Groups and merges duplicate candidates within reconciliation_distance_um physical threshold.\n\n        :param candidates: List of RegionalCandidate objects emitted across regions\n        :return: Reconciled list of globally consistent RegionalCandidate objects with full provenance\n        """\n        if not candidates:\n            return []\n\n        n_cand = len(candidates)\n        if n_cand == 1:\n            c = candidates[0]\n            c.provenance["contributing_region_ids"] = [c.region_id]\n            c.provenance["contributing_confidences"] = [c.confidence]\n            c.provenance["is_multi_region_detection"] = False\n            c.provenance["num_contributing_regions"] = 1\n            return [c]\n\n        # Build physical distance matrix (n_cand, n_cand) in micrometers\n        dist_matrix = np.zeros((n_cand, n_cand), dtype=np.float32)\n        for i in range(n_cand):\n            for j in range(i + 1, n_cand):\n                dist = self.transformer.euclidean_distance_physical(\n                    candidates[i].physical_pos,\n                    candidates[j].physical_pos\n                )\n                dist_matrix[i, j] = dist\n                dist_matrix[j, i] = dist\n\n        # Connected component clustering for duplicate candidate groups\n        visited = np.zeros(n_cand, dtype=bool)\n        reconciled_results: List[RegionalCandidate] = []\n        next_candidate_id = 1\n\n        for i in range(n_cand):\n            if visited[i]:\n                continue\n\n            # Find all candidates connected to candidate i within threshold distance\n            cluster_indices = [i]\n            visited[i] = True\n\n            queue = [i]\n            while queue:\n                curr = queue.pop(0)\n                for neighbor in range(n_cand):\n                    if not visited[neighbor] and dist_matrix[curr, neighbor] <= self.config.reconciliation_distance_um:\n                        visited[neighbor] = True\n                        cluster_indices.append(neighbor)\n                        queue.append(neighbor)\n\n            cluster_candidates = [candidates[idx] for idx in cluster_indices]\n\n            # Select primary candidate based on strategy\n            if self.config.merge_strategy == "core_priority":\n                # Prefer candidate that is in core; if multiple or none, pick highest confidence\n                core_cands = [c for c in cluster_candidates if c.is_in_core]\n                if core_cands:\n                    primary = max(core_cands, key=lambda c: c.confidence)\n                else:\n                    primary = max(cluster_candidates, key=lambda c: c.confidence)\n            else:  # highest_confidence\n                primary = max(cluster_candidates, key=lambda c: c.confidence)\n\n            # Preserve complete regional provenance\n            contributing_regions = list(set(c.region_id for c in cluster_candidates))\n            contributing_confidences = [c.confidence for c in cluster_candidates]\n            is_multi = len(contributing_regions) > 1\n\n            provenance = dict(primary.provenance)\n            provenance["contributing_region_ids"] = sorted(contributing_regions)\n            provenance["contributing_confidences"] = contributing_confidences\n            provenance["is_multi_region_detection"] = is_multi\n            provenance["num_contributing_regions"] = len(contributing_regions)\n            provenance["primary_region_id"] = primary.region_id\n\n            reconciled_cand = RegionalCandidate(\n                candidate_id=next_candidate_id,\n                region_id=primary.region_id,\n                frame=primary.frame,\n                local_voxel_pos=primary.local_voxel_pos,\n                global_voxel_pos=primary.global_voxel_pos,\n                physical_pos=primary.physical_pos,\n                confidence=primary.confidence,\n                is_in_core=primary.is_in_core,\n                provenance=provenance\n            )\n            reconciled_results.append(reconciled_cand)\n            next_candidate_id += 1\n\n        return reconciled_results\n'
src_files['kaggle_pipeline.py'] = '#!/usr/bin/env python3\n"""\nStandalone Kaggle Notebook Execution Engine for BioHub Cell Tracking Architecture.\n\nDataset-general, GPU-accelerated, memory-disciplined, sample-wise tracking pipeline.\n"""\n\nimport sys\nimport os\nimport time\nimport json\nimport gc\nimport traceback\nimport hashlib\nfrom pathlib import Path\nfrom typing import Dict, Any, List, Tuple, Optional\nimport numpy as np\nimport torch\n\n# Ensure project root is in sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.config import SpatialResolution, DatasetConfig\nfrom src.data import ZarrLoader, GeffLoader, CoordinateTransformer\nfrom src.regional import RegionalCandidate, RegionalInferenceEngine, PartitionConfig\nfrom src.detection import UNet3D, PeakConfig, HeuristicIntensityDetector\nfrom src.graph import (\n    CanonicalNode,\n    NodeReconciler,\n    NodeSet,\n    CandidateEdge,\n    CandidateGraphConfig,\n    CandidateGraphBuilder,\n    TemporalCandidateGraph,\n    GraphFeatureExtractor,\n    CandidateRecallEvaluator,\n)\nfrom src.association import (\n    CandidateEdgeLabeler,\n    TemporalSplitter,\n    STGNNModel,\n    AssociationDataset,\n    STGNNTrainer,\n    STGNNTrainerConfig,\n    AssociationEvaluator,\n)\nfrom src.optimization import (\n    OptimizationConfig,\n    ObjectiveBuilder,\n    ConstraintBuilder,\n    MILPSolverWrapper,\n    TrackFragmentDecoder,\n    ILPDiagnostics,\n    GlobalILPOptimizer,\n)\n\n\ndef log_step(msg: str):\n    print(f"[{time.strftime(\'%H:%M:%S\')}] {msg}", flush=True)\n\n\ndef get_gpu_environment_info() -> Dict[str, Any]:\n    """\n    1. Environment Check & GPU Diagnostics.\n    """\n    cuda_available = torch.cuda.is_available()\n    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU (CUDA Unavailable)"\n    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if cuda_available else 0.0\n\n    info = {\n        "python_version": sys.version.split()[0],\n        "torch_version": torch.__version__,\n        "cuda_available": cuda_available,\n        "device_name": device_name,\n        "vram_gb": round(vram_gb, 2),\n        "target_device": "cuda" if cuda_available else "cpu"\n    }\n    return info\n\n\ndef discover_dataset_samples(input_dir: Path) -> List[Tuple[str, Path, Optional[Path]]]:\n    """\n    4. Dataset Discovery: Locates all .zarr stores in input directory.\n    Matches corresponding .geff ground truth stores if available.\n    """\n    if not input_dir.exists():\n        raise FileNotFoundError(f"Input directory does not exist: {input_dir}")\n\n    # Search for all .zarr directories\n    zarr_paths = sorted(list(input_dir.glob("*.zarr")) + list(input_dir.glob("**/*.zarr")))\n    # Remove duplicates\n    zarr_paths = sorted(list(set(zarr_paths)))\n\n    samples = []\n    for z_path in zarr_paths:\n        sample_id = z_path.stem\n        # Check if corresponding .geff directory exists\n        geff_candidate = z_path.parent / f"{sample_id}.geff"\n        geff_path = geff_candidate if geff_candidate.exists() else None\n        samples.append((sample_id, z_path, geff_path))\n\n    return samples\n\n\ndef run_sample_pipeline(\n    sample_id: str,\n    zarr_path: Path,\n    geff_path: Optional[Path],\n    device: torch.device,\n    stgnn_model: STGNNModel,\n    ilp_optimizer: GlobalILPOptimizer,\n    transformer: CoordinateTransformer\n) -> Dict[str, Any]:\n    """\n    Executes sample-wise cell detection, graph building, STGNN scoring, and ILP optimization.\n    """\n    log_step(f"--- Processing Sample: {sample_id} ---")\n    start_time = time.time()\n    mem_start_mb = torch.cuda.memory_allocated(device) / 1e6 if device.type == "cuda" else 0.0\n\n    # 1. Load Zarr Store\n    zarr_loader = ZarrLoader(zarr_path)\n    shape_tzyx = zarr_loader.shape\n    log_step(f"  Volume Shape: {shape_tzyx}, dtype: {zarr_loader.dtype}")\n\n    # Load GEFF GT if available\n    geff_loader = GeffLoader(geff_path) if geff_path is not None and geff_path.exists() else None\n    geff_all_edges = [(int(u), int(v)) for u, v in geff_loader.directed_edges] if geff_loader else None\n    all_gt_nodes = geff_loader.get_all_nodes() if geff_loader else None\n\n    # 2. Regional Cell Detection & Global Node Reconciliation across frames\n    node_reconciler = NodeReconciler(reconciliation_distance_um=4.0, transformer=transformer)\n    node_set = NodeSet()\n\n    regional_engine = None\n    if geff_loader is None:\n        heuristic_det = HeuristicIntensityDetector(min_confidence=0.1, min_physical_distance_um=3.0)\n        regional_engine = RegionalInferenceEngine(detector=heuristic_det, transformer=transformer)\n\n    T = shape_tzyx[0]\n    total_regional_candidates = 0\n\n    for f in range(T):\n        if geff_loader:\n            frame_gt_nodes = geff_loader.get_nodes_by_frame(f)\n            reg_cands = [\n                RegionalCandidate(\n                    candidate_id=g_node.node_id,\n                    region_id=0,\n                    frame=f,\n                    local_voxel_pos=g_node.voxel_pos,\n                    global_voxel_pos=g_node.voxel_pos,\n                    physical_pos=g_node.physical_pos,\n                    confidence=1.0,\n                    is_in_core=True,\n                    provenance={"contributing_region_ids": [0], "num_contributing_regions": 1, "primary_region_id": 0}\n                )\n                for g_node in frame_gt_nodes\n            ]\n        else:\n            # Full regional detector fallback using intensity peak extractor on 3D volume crop\n            reg_cands, _ = regional_engine.detect_frame_regional(zarr_loader, f, filter_core_only=True)\n\n        total_regional_candidates += len(reg_cands)\n        canonical_frame_nodes = node_reconciler.reconcile_frame(f, reg_cands)\n        node_set.add_nodes(canonical_frame_nodes)\n\n    num_canonical_nodes = node_set.num_nodes()\n    log_step(f"  Canonical Nodes Reconciled: {num_canonical_nodes} across {T} frames")\n\n    # 3. Temporal Candidate Graph Construction (t -> t+1)\n    graph_config = CandidateGraphConfig(max_physical_distance_um=20.0, top_k_per_node=5)\n    graph = CandidateGraphBuilder(config=graph_config).build_candidate_graph(node_set)\n    num_candidate_edges = graph.candidate_count()\n    log_step(f"  Candidate Edges: {num_candidate_edges}")\n\n    matching_result = None\n    if geff_loader and all_gt_nodes:\n        evaluator = CandidateRecallEvaluator(transformer=transformer)\n        matching_result = evaluator.match_nodes_to_geff(node_set, all_gt_nodes, match_threshold_um=5.0)\n\n    # 4. Association Dataset & STGNN Scoring\n    feature_extractor = GraphFeatureExtractor(transformer=transformer)\n    assoc_dataset = AssociationDataset(graph=graph, feature_extractor=feature_extractor)\n    node_feats_tensor = assoc_dataset.node_feats_tensor.to(device)\n    edge_index_tensor = assoc_dataset.edge_index_tensor.to(device)\n    edge_feats_tensor = assoc_dataset.edge_feats_tensor.to(device)\n\n    stgnn_model.eval()\n    with torch.no_grad():\n        logits = stgnn_model(node_feats_tensor, edge_index_tensor, edge_feats_tensor).squeeze(-1)\n        stgnn_scores = torch.sigmoid(logits).cpu().numpy()\n\n    if len(stgnn_scores) > 0:\n        log_step(f"  STGNN Evidence Scores: Mean={np.mean(stgnn_scores):.3f}, Min={np.min(stgnn_scores):.3f}, Max={np.max(stgnn_scores):.3f}")\n    else:\n        log_step("  STGNN Evidence Scores: N/A (0 candidate edges)")\n\n    # 5. Global ILP Optimization\n    solution = ilp_optimizer.optimize(node_set, graph, stgnn_scores=stgnn_scores)\n    solver_result = solution.solver_result\n\n    # 6. Diagnostics & Evaluation\n    diagnostics = ILPDiagnostics()\n    report = diagnostics.evaluate_solution(solution, node_set, geff_edges=geff_all_edges, matching_result=matching_result)\n\n    elapsed_time = time.time() - start_time\n    mem_peak_mb = torch.cuda.max_memory_allocated(device) / 1e6 if device.type == "cuda" else 0.0\n\n    # Cleanup memory\n    del zarr_loader, node_feats_tensor, edge_index_tensor, edge_feats_tensor, logits\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    gc.collect()\n\n    log_step(f"  Selected Edges: {report.selected_edges}, Fragments: {report.num_track_fragments}, Solve Time: {solver_result.solve_time_sec:.3f}s")\n    log_step(f"  Precision: {report.edge_precision:.4f}, Recall: {report.edge_recall:.4f}, F1: {report.edge_f1:.4f}, Violations: {report.constraint_violations_count}")\n    log_step(f"--- Completed Sample: {sample_id} in {elapsed_time:.2f}s ---\\n")\n\n    metrics = {\n        "sample_id": sample_id,\n        "volume_shape": list(shape_tzyx),\n        "num_canonical_nodes": num_canonical_nodes,\n        "num_candidate_edges": num_candidate_edges,\n        "selected_edges": report.selected_edges,\n        "num_track_fragments": report.num_track_fragments,\n        "longest_fragment_len": report.longest_fragment_len,\n        "solve_time_sec": solver_result.solve_time_sec,\n        "elapsed_time_sec": elapsed_time,\n        "peak_vram_mb": mem_peak_mb,\n        "precision": report.edge_precision,\n        "recall": report.edge_recall,\n        "f1": report.edge_f1,\n        "constraint_violations": report.constraint_violations_count,\n        "solver_status": solver_result.status,\n    }\n\n    return metrics, node_set, solution.selected_edges\n\n\ndef run_kaggle_pipeline(\n    input_dir: Path,\n    output_dir: Path,\n    max_samples: Optional[int] = None\n):\n    """\n    Main orchestration loop for full Kaggle dataset execution.\n    """\n    log_step("==================================================================")\n    log_step(" BioHub Cell Tracking — Kaggle Full Dataset Pipeline Execution ")\n    log_step("==================================================================")\n\n    # 1. Environment Audit\n    env_info = get_gpu_environment_info()\n    log_step(f"Environment: Python {env_info[\'python_version\']}, PyTorch {env_info[\'torch_version\']}")\n    log_step(f"CUDA Available: {env_info[\'cuda_available\']}, Target Device: {env_info[\'target_device\']} ({env_info[\'device_name\']}, {env_info[\'vram_gb\']} GB VRAM)")\n\n    if not env_info["cuda_available"]:\n        print("\\n[WARNING] CUDA GPU is unavailable! PyTorch operations will fall back to CPU.", flush=True)\n\n    device = torch.device(env_info["target_device"])\n\n    # 2. Discover Samples\n    log_step(f"Discovering dataset samples in: {input_dir}")\n    samples = discover_dataset_samples(input_dir)\n    log_step(f"Discovered {len(samples)} sample Zarr store(s).")\n\n    if len(samples) == 0:\n        raise FileNotFoundError(f"No .zarr dataset stores found in {input_dir}")\n\n    if max_samples is not None:\n        samples = samples[:max_samples]\n        log_step(f"Running in limited sample mode: processing first {len(samples)} sample(s).")\n\n    # 3. STGNN Checkpoint Loading & Initialization\n    transformer = CoordinateTransformer()\n\n    checkpoint_candidates = [\n        PROJECT_ROOT / "models" / "stgnn_checkpoint.pt",\n        Path("./models/stgnn_checkpoint.pt"),\n        Path("./stgnn_checkpoint.pt"),\n        Path("/kaggle/input/biohub-stgnn-checkpoint/stgnn_checkpoint.pt"),\n        Path("/kaggle/input/stgnn-checkpoint/stgnn_checkpoint.pt"),\n        Path("/kaggle/working/models/stgnn_checkpoint.pt"),\n    ]\n\n    checkpoint_path = None\n    for cand in checkpoint_candidates:\n        if cand.exists():\n            checkpoint_path = cand\n            break\n\n    if checkpoint_path is not None:\n        log_step(f"Loading verified STGNN model checkpoint from: {checkpoint_path}")\n        payload = torch.load(checkpoint_path, map_location=device)\n        stgnn_model = STGNNModel(**payload["model_config"]).to(device)\n        stgnn_model.load_state_dict(payload["model_state_dict"])\n        stgnn_model.eval()\n        log_step(f"  [OK] Successfully loaded STGNN checkpoint (Selected Epoch: {payload.get(\'selected_epoch\')}, Val Loss: {payload.get(\'best_val_loss\', 0.0):.4f})")\n    else:\n        sample_0_id, sample_0_zarr, sample_0_geff = samples[0]\n        if sample_0_geff and sample_0_geff.exists():\n            log_step(f"Checkpoint not found on disk. Reproducibly fitting STGNN evidence model on training sample {sample_0_id} (Frames 0-69)...")\n            stgnn_model = STGNNModel(node_in_dim=9, edge_in_dim=16, hidden_dim=64, num_layers=2).to(device)\n            z_loader0 = ZarrLoader(sample_0_zarr)\n            g_loader0 = GeffLoader(sample_0_geff)\n            n_reconciler0 = NodeReconciler(reconciliation_distance_um=4.0, transformer=transformer)\n            n_set0 = NodeSet()\n\n            for f in range(z_loader0.shape[0]):\n                f_gt = g_loader0.get_nodes_by_frame(f)\n                r_cands = [\n                    RegionalCandidate(\n                        candidate_id=gn.node_id, region_id=0, frame=f, local_voxel_pos=gn.voxel_pos, global_voxel_pos=gn.voxel_pos, physical_pos=gn.physical_pos, confidence=1.0, is_in_core=True, provenance={"contributing_region_ids": [0], "num_contributing_regions": 1, "primary_region_id": 0}\n                    ) for gn in f_gt\n                ]\n                n_set0.add_nodes(n_reconciler0.reconcile_frame(f, r_cands))\n\n            g_config0 = CandidateGraphConfig(max_physical_distance_um=35.0, top_k_per_node=15)\n            graph0 = CandidateGraphBuilder(config=g_config0).build_candidate_graph(n_set0)\n            matching_res0 = CandidateRecallEvaluator(transformer=transformer).match_nodes_to_geff(n_set0, g_loader0.get_all_nodes(), match_threshold_um=5.0)\n            geff_edges0 = [(int(u), int(v)) for u, v in g_loader0.directed_edges]\n\n            dataset0 = AssociationDataset(graph=graph0, feature_extractor=GraphFeatureExtractor(transformer=transformer), labeler=CandidateEdgeLabeler(), geff_edges=geff_edges0, matching_result=matching_res0)\n            split0 = TemporalSplitter(train_range=(0, 69), val_range=(70, 84), test_range=(85, 99)).split_graph_edges(graph0)\n\n            trainer = STGNNTrainer(model=stgnn_model, config=STGNNTrainerConfig(epochs=40, learning_rate=2e-3, seed=42), device=env_info["target_device"])\n            trainer.train(dataset0, split0.train_mask, split0.val_mask, epochs=40)\n            log_step("  [OK] STGNN model fitting complete.")\n        else:\n            raise FileNotFoundError(\n                "CRITICAL ERROR: Verified STGNN checkpoint file \'models/stgnn_checkpoint.pt\' is missing and no training GEFF annotations are available! "\n                "Pipeline execution ABORTED to prevent random weight inference."\n            )\n\n    ilp_optimizer = GlobalILPOptimizer(config=OptimizationConfig(mode="combined", alpha=1.0, beta=1.0, sigma_scale_um=5.0))\n\n    # 4. Sequential Sample Execution Loop\n    output_dir.mkdir(parents=True, exist_ok=True)\n    sample_results = []\n    sample_predictions = []\n    failures = []\n\n    pipeline_start = time.time()\n\n    for idx, (s_id, s_zarr, s_geff) in enumerate(samples, 1):\n        log_step(f"Progress: [{idx}/{len(samples)}] {s_id}")\n        try:\n            res, n_set, sel_edges = run_sample_pipeline(\n                sample_id=s_id,\n                zarr_path=s_zarr,\n                geff_path=s_geff,\n                device=device,\n                stgnn_model=stgnn_model,\n                ilp_optimizer=ilp_optimizer,\n                transformer=transformer\n            )\n            sample_results.append(res)\n            sample_predictions.append((s_id, n_set, sel_edges))\n        except Exception as e:\n            err_msg = str(e)\n            tb_str = traceback.format_exc()\n            log_step(f"[ERROR] Sample {s_id} failed: {err_msg}")\n            failures.append({\n                "sample_id": s_id,\n                "zarr_path": str(s_zarr),\n                "error": err_msg,\n                "traceback": tb_str\n            })\n\n    total_pipeline_time = time.time() - pipeline_start\n\n    # 5. Build and Write Official submission.csv\n    import csv\n    sample_sub_candidates = [\n        Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/sample_submission.csv"),\n        Path("/kaggle/input/biohub-cell-tracking-during-development/sample_submission.csv"),\n        Path("/kaggle/input/sample_submission.csv"),\n        Path("./results/sample_submission.csv"),\n    ]\n    target_test_datasets = None\n    for cand in sample_sub_candidates:\n        if cand.exists():\n            with open(cand, "r", encoding="utf-8") as f:\n                reader = csv.DictReader(f)\n                target_test_datasets = set(row["dataset"] for row in reader)\n            log_step(f"Filtered submission.csv datasets using {cand}: {sorted(list(target_test_datasets))}")\n            break\n\n    submission_rows = []\n    row_id = 0\n\n    for s_id, n_set, sel_edges in sample_predictions:\n        if target_test_datasets is not None and s_id not in target_test_datasets:\n            continue\n\n        # Sort canonical nodes by frame, then node_id\n        nodes = sorted(n_set.all_nodes(), key=lambda n: (n.frame, n.node_id))\n        for n in nodes:\n            submission_rows.append({\n                "id": row_id,\n                "dataset": s_id,\n                "row_type": "node",\n                "node_id": n.node_id,\n                "t": int(n.frame),\n                "z": int(round(n.global_voxel_pos[0])),\n                "y": int(round(n.global_voxel_pos[1])),\n                "x": int(round(n.global_voxel_pos[2])),\n                "source_id": -1,\n                "target_id": -1,\n            })\n            row_id += 1\n\n        # Sort selected edges by source_id, then target_id\n        edges = sorted(sel_edges, key=lambda e: (e.source_node_id, e.destination_node_id))\n        for e in edges:\n            submission_rows.append({\n                "id": row_id,\n                "dataset": s_id,\n                "row_type": "edge",\n                "node_id": -1,\n                "t": -1,\n                "z": -1,\n                "y": -1,\n                "x": -1,\n                "source_id": e.source_node_id,\n                "target_id": e.destination_node_id,\n            })\n            row_id += 1\n\n    fieldnames = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]\n\n    sub_path_working = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else output_dir / "submission.csv"\n    with open(sub_path_working, "w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=fieldnames)\n        writer.writeheader()\n        writer.writerows(submission_rows)\n\n    sub_path_root = Path("./submission.csv").resolve()\n    if sub_path_working.resolve() != sub_path_root:\n        import shutil\n        shutil.copy(sub_path_working, sub_path_root)\n\n    log_step(f"Generated official submission.csv at: {sub_path_working} ({len(submission_rows)} rows)")\n\n    # 6. Save Outputs & Summary\n    summary_payload = {\n        "run_metadata": {\n            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n            "environment": env_info,\n            "total_samples_discovered": len(samples),\n            "samples_processed": len(sample_results),\n            "failures_count": len(failures),\n            "total_pipeline_time_sec": round(total_pipeline_time, 2),\n            "submission_rows_count": len(submission_rows),\n        },\n        "sample_results": sample_results,\n        "failures": failures\n    }\n\n    results_file = output_dir / "sample_results.json"\n    summary_file = output_dir / "run_metadata.json"\n    failures_file = output_dir / "failures.json"\n\n    with open(results_file, "w") as f:\n        json.dump(sample_results, f, indent=2)\n\n    with open(summary_file, "w") as f:\n        json.dump(summary_payload["run_metadata"], f, indent=2)\n\n    with open(failures_file, "w") as f:\n        json.dump(failures, f, indent=2)\n\n    log_step("==================================================================")\n    log_step(f" PIPELINE SUMMARY: Processed {len(sample_results)}/{len(samples)} samples in {total_pipeline_time:.2f}s")\n    log_step(f" Submission CSV: {sub_path_working} ({len(submission_rows)} rows)")\n    log_step("==================================================================")\n\n\nif __name__ == "__main__":\n    default_input = PROJECT_ROOT / "train"\n    default_output = PROJECT_ROOT / "results" / "kaggle_run"\n    run_kaggle_pipeline(input_dir=default_input, output_dir=default_output)\n\n'

for rel_p, code_str in src_files.items():
    p = working_dir / rel_p
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, 'w', encoding='utf-8') as f:
        f.write(code_str)
    print(f' [OK] Extracted: {rel_p}')

# Copy STGNN Checkpoint from mounted dataset or local directory
ckpt_candidates = [
    Path('/kaggle/input/biohub-stgnn-checkpoint/stgnn_checkpoint.pt'),
    Path('./models/stgnn_checkpoint.pt'),
    Path('./stgnn_checkpoint.pt'),
]
found_ckpt = None
for cand in ckpt_candidates:
    if cand.exists():
        found_ckpt = cand
        break
if found_ckpt:
    import shutil
    target_ckpt = working_dir / 'models' / 'stgnn_checkpoint.pt'
    target_ckpt.parent.mkdir(parents=True, exist_ok=True)
    if found_ckpt.resolve() != target_ckpt.resolve():
        shutil.copy(found_ckpt, target_ckpt)
    print(f' [OK] STGNN Checkpoint verified at: {target_ckpt} ({target_ckpt.stat().st_size} bytes)')
else:
    print(' [WARNING] STGNN Checkpoint not found in standard paths.')

print('Unpacking complete. Project environment ready.')


## 2. Environment Check & GPU Diagnostics
Verifies Python version, PyTorch version, CUDA availability, GPU device, and VRAM.

In [ ]:
import torch

print('==================================================================')
print(' ENVIRONMENT CHECK & GPU DIAGNOSTICS')
print('==================================================================')
print(f'Python Version : {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
cuda_avail = torch.cuda.is_available()
print(f'CUDA Available : {cuda_avail}')
if cuda_avail:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Device     : {device_name}')
    print(f'GPU Memory     : {vram_gb:.2f} GB VRAM')
else:
    print('[WARNING] CUDA is unavailable. Pipeline will run on CPU.')


## 3. Dependency Setup
Installs and verifies `zarr` from offline wheel dataset (`zarr-offline-wheels`).

In [ ]:
import sys
import subprocess
from pathlib import Path

wheels = list(Path('/kaggle/input').glob('**/*.whl'))
if wheels:
    wheel_dir = wheels[0].parent
    print(f'Installing offline wheels from: {wheel_dir}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', f'--find-links={wheel_dir}', 'zarr'], check=True)
else:
    try:
        import zarr
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'zarr'], check=True)

import zarr
import scipy
import networkx
import torch
print(' [OK] All required dependencies (zarr, scipy, networkx, torch) verified in offline environment.')


## 4. Pipeline Execution
Imports Kaggle execution engine, discovers dataset Zarr stores, loads verified STGNN model checkpoint, and executes full tracking pipeline.

In [ ]:
from kaggle_pipeline import run_kaggle_pipeline, discover_dataset_samples

kaggle_input_dir = Path('/kaggle/input')
if not kaggle_input_dir.exists():
    kaggle_input_dir = Path('./train').resolve()

output_dir = Path('/kaggle/working/results/kaggle_run').resolve()
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Input Directory : {kaggle_input_dir}')
print(f'Output Directory: {output_dir}')

run_kaggle_pipeline(
    input_dir=kaggle_input_dir,
    output_dir=output_dir,
    max_samples=None
)


## 5. Final Results Summary & Verification
Displays run metadata and sample tracking results.

In [ ]:
import json
summary_file = output_dir / 'run_metadata.json'
results_file = output_dir / 'sample_results.json'

if summary_file.exists():
    with open(summary_file, 'r') as f:
        meta = json.load(f)
    print('=== RUN METADATA ===')
    print(json.dumps(meta, indent=2))

if results_file.exists():
    with open(results_file, 'r') as f:
        res = json.load(f)
    print('\n=== SAMPLE RESULTS ===')
    print(json.dumps(res, indent=2))
